# ABC2027 CNN-TCN V9: chuẩn hóa frame và kiểm tra nhánh khử nền

V9 kiểm tra hai yếu tố: **chuẩn hóa biên độ từng frame** và **giữ/bỏ residual sau EMA**.
Model học mọi Chute01-Chute04 và hoạt động không ngã, với cửa sổ 8 giây.
Notebook độc lập, đã nhúng toàn bộ mã cần thiết; không cần notebook hay checkpoint V8.

Upload notebook lên Colab, chọn GPU, sửa DATA_ROOT ở cell cấu hình rồi chạy lần lượt.
Mỗi fold lấy official train để chia fit / inner / outer theo nhóm nguồn:
fit học trọng số và thống kê preprocessing; inner chọn checkpoint/ngưỡng;
outer được chấm sau khi khóa các lựa chọn. Đây là 3 fold theo nhóm, chưa phải full LOSO.
Public là official validation `streams/val`, đã được xem ở các phiên bản trước, chưa phải test độc lập.


In [ ]:
import importlib.util
import subprocess
import sys
missing = [package for module,package in {
    "numpy":"numpy", "pandas":"pandas", "h5py":"h5py", "scipy":"scipy",
    "torch":"torch", "matplotlib":"matplotlib", "jinja2":"jinja2"
}.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 1. Nạp pipeline

CIR (3 radar, T, 368 bins) ở 50 Hz -> residual và delta, 6 kênh.
Residual = |CIR - nền EMA|; delta = |CIR hiện tại - CIR trước đó|.
B/D chia từng frame của mỗi kênh cho max(max theo range, mức sàn), **trước pooling**.
Mức sàn là phân vị 10% của cực đại frame trên fit, lấy mẫu thời gian đều,
không dùng nhãn để chọn frame. Đây là sàn biên độ thực nghiệm, chưa phải noise floor vật lý.
Cần kiểm tra lại nếu chuẩn hóa tăng báo động giả; chuẩn hóa không làm tăng SNR.

Sau đó gộp 368 -> 92 bins và 50 -> 10 Hz; z-score từng kênh chỉ fit trên fit rồi clip [-8,8].
C/D đưa 3 kênh residual về 0, mean=0, scale=1; giữ nguyên kích thước mạng.
CNN 24/48 -> TCN 64/64/96 -> ba đầu xác suất action/start/end theo thời gian.
Chưa cộng toàn bộ range bins, chưa bật Chebyshev, chưa cắt 50 frame.
EMA/delta khởi tạo lại ở đầu từng view cùng split, duy trì state qua các chunk đọc tệp.
Khi suy luận public, xử lý toàn luồng, không dùng biên event hay composition để cắt input.

Chuẩn hóa trên bản đồ chuyển động là biến thể lấy cảm hứng từ paper;
khác công thức chuẩn hóa CIR có dấu rồi cộng không gian của tác giả.
Cell dưới chỉ nạp định nghĩa, chưa train.


In [ ]:
import sys, types
from pathlib import Path
import json
V4_SOURCE='"""ABC2027 V4: isolated source views, event sampling and mild radar gain."""\nfrom __future__ import annotations\n\n# %% Configuration\nimport copy\nimport hashlib\nimport json\nimport math\nimport platform\nimport random\nimport re\nimport sys\nimport time\nimport zipfile\nfrom collections import defaultdict\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport h5py\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.signal import lfilter\nfrom torch.utils.data import DataLoader, Dataset\n\n\n@dataclass\nclass Config:\n    seed: int = 2027\n    batch_size: int = 32\n    max_epochs: int = 20\n    patience: int = 5\n    lr: float = 3e-4\n    weight_decay: float = 2e-4\n    window_frames: int = 80\n    target_hz: int = 10\n    train_hop_frames: int = 20\n    infer_hop_frames: int = 2\n    windows_per_event: int = 3\n    negative_ratio: int = 3\n    event_sampler: bool = True\n    jitter_frames: int = 10\n    gain_augment: bool = True\n    gain_probability: float = 0.5\n    gain_low: float = 0.9\n    gain_high: float = 1.1\n    feature_noise_std: float = 0.0\n    train_eval_max_windows: int = 2048\n    ema_tau_s: float = 3.0\n    smoke: bool = False\n\n    def validate(self):\n        if self.target_hz != 10 or self.window_frames != 80:\n            raise ValueError("This baseline uses 10 Hz and 8-second windows.")\n        if min(self.batch_size, self.max_epochs, self.patience, self.train_hop_frames,\n               self.infer_hop_frames, self.windows_per_event, self.negative_ratio,\n               self.train_eval_max_windows) < 1:\n            raise ValueError("Counts, hops, batch size and epochs must be positive.")\n        if not (0 < self.gain_low <= self.gain_high and 0 <= self.gain_probability <= 1):\n            raise ValueError("Invalid radar gain augmentation settings.")\n        if self.feature_noise_std < 0 or self.jitter_frames < 0 or self.ema_tau_s <= 0:\n            raise ValueError("Noise and jitter must be nonnegative.")\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.set_num_threads(min(4, torch.get_num_threads()))\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, default=float, allow_nan=False), encoding="utf-8")\n\n\ndef truth(value):\n    return str(value).strip().lower() in {"1", "true", "yes"}\n\n\ndef overlap(a, b, c, d):\n    return max(0.0, min(b, d)-max(a, c))\n\n\ndef source_group(path, kind):\n    pattern = r"_(\\d+)\\.json$" if kind == "adl" else r"Test_P\\d+_(\\d+)"\n    match = re.search(pattern, str(path))\n    if match is None:\n        raise ValueError(f"Cannot parse source group: {path}")\n    return f"{kind}:{match.group(1)}"\n\n\ndef boolean_runs(mask):\n    edges = np.flatnonzero(np.diff(np.r_[False, mask, False].astype(np.int8)))\n    return [(int(a), int(b)) for a, b in zip(edges[::2], edges[1::2])]\n\n\n# %% Metadata and isolated views\ndef exclusive_ranges(parts, n_frames, holdout):\n    fit, val = np.zeros(n_frames, bool), np.zeros(n_frames, bool)\n    for part in parts:\n        a = max(0, int(round(float(part["t_start"])*50)))\n        b = min(n_frames, int(round(float(part["t_end"])*50)))\n        (val if part["group"] in holdout else fit)[a:b] = True\n    return {"fit": boolean_runs(fit & ~val), "holdout": boolean_runs(val & ~fit)}\n\n\ndef local_events(events, start, end):\n    rows = []\n    for event in events:\n        a, b = float(event["t_start"]), float(event["t_end"])\n        if overlap(start, end, a, b) <= 0:\n            continue\n        row = dict(event)\n        row["truncated"] = a < start or b > end\n        row["ignore"] = truth(event["ignore"]) or row["truncated"]\n        row["t_start"], row["t_end"] = max(a, start)-start, min(b, end)-start\n        rows.append(row)\n    return rows\n\n\ndef read_metadata(root):\n    events = pd.read_csv(root/"streams_events.csv")\n    parts = pd.read_csv(root/"streams_composition.csv")\n    parts["group"] = [source_group(p, k) for p, k in zip(parts.path, parts.kind)]\n    return (events, parts,\n            {sid: g.to_dict("records") for sid, g in events.groupby("stream_id")},\n            {sid: g.to_dict("records") for sid, g in parts.groupby("stream_id")})\n\n\n# %% Signal preprocessing\ndef extract_maps(handle, start, end, tau_s=3.0, chunk_frames=5000):\n    """Reset EMA/difference history per view; retain all 92 pooled range bins."""\n    if tuple(handle["cir"].shape[::2]) != (3, 368):\n        raise ValueError("Expected CIR shape (3, T, 368).")\n    alpha = math.exp(-1/(50*tau_s))\n    previous = state = pending = None\n    features_out, times_out, processed = [], [], 0\n    for offset in range(start, end, chunk_frames):\n        x = handle["cir"][:, offset:min(offset+chunk_frames, end)].astype(np.float64)\n        n = x.shape[1]\n        if previous is None:\n            previous = x[:, 0].copy()\n            state = (alpha*previous).reshape(-1, 1)\n        flat = x.transpose(0, 2, 1).reshape(-1, n)\n        background, state = lfilter([1-alpha], [1, -alpha], flat, axis=1, zi=state)\n        residual = np.abs((flat-background).reshape(3, 368, n).transpose(0, 2, 1))\n        delta = np.abs(np.diff(x, axis=1, prepend=previous[:, None]))\n        previous = x[:, -1].copy()\n        z = np.concatenate([residual, delta], axis=0).reshape(6, n, 92, 4).mean(axis=3)\n        if pending is not None:\n            z = np.concatenate([pending, z], axis=1)\n        blocks = z.shape[1]//5\n        usable = blocks*5\n        if blocks:\n            features_out.append(z[:, :usable].reshape(6, blocks, 5, 92).mean(axis=2).astype(np.float32))\n            times_out.append((processed+np.arange(blocks)*5+2)/50)\n            processed += usable\n        pending = z[:, usable:]\n    if pending is not None and pending.shape[1]:\n        features_out.append(pending.mean(axis=1, keepdims=True).astype(np.float32))\n        times_out.append(np.array([(processed+(pending.shape[1]-1)/2)/50]))\n    return np.concatenate(features_out, axis=1), np.concatenate(times_out).astype(np.float32)\n\n\ndef build_views(root, cache_root, holdout, cfg, public=False):\n    _, parts, event_map, part_map = read_metadata(root)\n    train_groups = set(parts[parts.stream_id.str.startswith("train_")].group)\n    if not set(holdout) <= train_groups:\n        raise ValueError(f"Unknown training holdout groups: {set(holdout)-train_groups}")\n    cache_root.mkdir(parents=True, exist_ok=True)\n    folder = root/"streams"/("val" if public else "train")\n    files = sorted(folder.glob("stream_*.h5"))\n    if cfg.smoke:\n        files = files[:1 if public else 2]\n    views, raw, manifest, skipped = {}, {}, [], []\n    for file in files:\n        sid = f"{folder.name}_{file.stem.split(\'_\')[1]}"\n        with h5py.File(file, "r") as handle:\n            n = handle["cir"].shape[1]\n            ranges = {"public": [(0, n)]} if public else exclusive_ranges(part_map.get(sid, []), n, set(holdout))\n            for side, spans in ranges.items():\n                if cfg.smoke and not public:\n                    pos, neg = [], []\n                    for a, b in spans:\n                        if b-a < 50:\n                            continue\n                        bucket = pos if any(overlap(a/50, b/50, e["t_start"], e["t_end"]) > 0 for e in event_map.get(sid, [])) else neg\n                        bucket.append((a, b))\n                    spans = pos[:3]+neg[:3]\n                for number, (a, b) in enumerate(spans):\n                    if b-a < 50:\n                        skipped.append({"source_stream": sid, "side": side, "start": a/50, "end": b/50, "reason": "shorter_than_1s"})\n                        continue\n                    if cfg.smoke:\n                        b = min(b, a+3000)\n                    view_id = sid if public else f"{sid}__{side}_{number:04d}"\n                    identity = {"version": "v4-isolated-maps-float32-v1", "file": str(file.resolve()),\n                                "size": file.stat().st_size, "mtime": file.stat().st_mtime_ns,\n                                "start": int(a), "end": int(b), "tau": cfg.ema_tau_s}\n                    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]\n                    path = cache_root/f"{digest}.npz"\n                    if path.exists():\n                        with np.load(path) as saved:\n                            x, times = saved["features"], saved["times"]\n                    else:\n                        x, times = extract_maps(handle, a, b, cfg.ema_tau_s)\n                        np.savez_compressed(path, features=x, times=times)\n                    if not np.isfinite(x).all():\n                        raise ValueError(f"Nonfinite features: {view_id}")\n                    present = {p["group"] for p in part_map.get(sid, []) if overlap(a/50, b/50, p["t_start"], p["t_end"]) > 1e-6}\n                    if side == "fit" and present & set(holdout):\n                        raise AssertionError("Holdout group reached fit preprocessing")\n                    if side == "holdout" and not present <= set(holdout):\n                        raise AssertionError("Fit group reached holdout preprocessing")\n                    ev = local_events(event_map.get(sid, []), a/50, b/50)\n                    duration = (b-a)/50\n                    views[view_id] = {"side": side, "duration": duration, "events": ev,\n                                      "groups": sorted(present), "source_stream": sid}\n                    raw[view_id] = (x, times)\n                    manifest.append({"view_id": view_id, "source_stream": sid, "side": side,\n                                     "source_start_s": a/50, "source_end_s": b/50, "duration_s": duration,\n                                     "groups": ";".join(sorted(present)), "events": len(ev),\n                                     "truncated_events": sum(e["truncated"] for e in ev)})\n        print(f"Features {sid}: {sum(r[\'source_stream\'] == sid for r in manifest)} isolated views", flush=True)\n    if not raw:\n        raise ValueError(f"No usable data in {folder}")\n    return views, raw, manifest, skipped\n\n\ndef fit_normalizer(raw, views):\n    total, total2, count = np.zeros(6), np.zeros(6), 0\n    for key, (x, _) in raw.items():\n        if views[key]["side"] != "fit":\n            continue\n        x = x.astype(np.float64)\n        total += x.sum(axis=(1, 2))\n        total2 += np.square(x).sum(axis=(1, 2))\n        count += x.shape[1]*x.shape[2]\n    if not count:\n        raise ValueError("Empty fit data for normalizer")\n    mean = total/count\n    scale = np.sqrt(np.maximum(total2/count-mean**2, 1e-16))\n    return mean.astype(np.float32), scale.astype(np.float32)\n\n\n# %% Labels, event sampling and augmentation\ndef targets_for(times, events):\n    action, starts, ends = [np.zeros(len(times), np.float32) for _ in range(3)]\n    weights = np.ones(len(times), np.float32)\n    for event in events:\n        a, b = event["t_start"], event["t_end"]\n        distance = np.maximum(np.maximum(a-times, times-b), 0)\n        action = np.maximum(action, np.clip(1-distance/0.3, 0, 1))\n        starts = np.maximum(starts, np.exp(-0.5*((times-a)/0.2)**2))\n        ends = np.maximum(ends, np.exp(-0.5*((times-b)/0.2)**2))\n        nearby = (times >= a-2) & (times <= b+2)\n        if truth(event["ignore"]):\n            weights[nearby] = 0\n        elif str(event["quality"]).lower() == "low" or truth(event["needs_review"]):\n            weights[nearby] = np.minimum(weights[nearby], 0.35)\n    return np.stack([action, starts, ends]), weights\n\n\ndef window_starts(n, win, hop):\n    rows = list(range(0, max(1, n-win+1), hop))\n    if rows[-1] != max(0, n-win):\n        rows.append(max(0, n-win))\n    return rows\n\n\ndef candidate_windows(raw, views, cfg):\n    positives, negatives, fit_eval, val_eval = defaultdict(list), [], [], []\n    for key, view in views.items():\n        times = raw[key][1]\n        for start in window_starts(len(times), cfg.window_frames, cfg.train_hop_frames):\n            item = (key, start)\n            if view["side"] == "holdout":\n                val_eval.append(item)\n                continue\n            if view["side"] != "fit":\n                continue\n            fit_eval.append(item)\n            a, b = float(times[start]), float(times[min(len(times), start+cfg.window_frames)-1]+0.1)\n            hits = [(overlap(a, b, e["t_start"], e["t_end"]), e["event_id"]) for e in view["events"] if not truth(e["ignore"])]\n            hits = [hit for hit in hits if hit[0] > 0]\n            if hits:\n                owner = max(hits)[1]\n                positives[owner].append(item)\n            else:\n                negatives.append(item)\n    if not positives or not negatives or not val_eval:\n        raise ValueError("Need positive fit events, negative fit windows and holdout windows")\n    rng = np.random.default_rng(cfg.seed)\n    if len(fit_eval) > cfg.train_eval_max_windows:\n        fit_eval = [fit_eval[i] for i in sorted(rng.choice(len(fit_eval), cfg.train_eval_max_windows, replace=False))]\n    return dict(positives), negatives, fit_eval, val_eval\n\n\ndef sample_epoch(positives, negatives, raw, views, cfg, epoch):\n    rng = np.random.default_rng(cfg.seed+epoch)\n    index, manifest, seen = [], [], set()\n    for event_id, candidates in sorted(positives.items()):\n        count = min(cfg.windows_per_event, len(candidates)) if cfg.event_sampler else len(candidates)\n        for i in rng.choice(len(candidates), count, replace=False):\n            key, start = candidates[i]\n            if cfg.event_sampler and cfg.jitter_frames:\n                proposal = int(np.clip(start+rng.integers(-cfg.jitter_frames, cfg.jitter_frames+1),\n                                       0, max(0, raw[key][0].shape[1]-cfg.window_frames)))\n                times = raw[key][1]\n                event = next(e for e in views[key]["events"] if e["event_id"] == event_id)\n                if overlap(float(times[proposal]), float(times[min(len(times), proposal+cfg.window_frames)-1]+0.1),\n                           event["t_start"], event["t_end"]) > 0:\n                    start = proposal\n            if (key, start) in seen:\n                start = candidates[i][1]\n            if (key, start) in seen:\n                continue\n            seen.add((key, start))\n            index.append((key, start))\n            manifest.append({"epoch": epoch, "view_id": key, "start_frame": start, "owner_event": event_id, "kind": "event"})\n    count = min(len(negatives), cfg.negative_ratio*len(index))\n    kept = 0\n    for i in rng.permutation(len(negatives)):\n        if kept >= count:\n            break\n        key, start = negatives[i]\n        if (key, start) in seen:\n            continue\n        seen.add((key, start))\n        kept += 1\n        index.append((key, start))\n        manifest.append({"epoch": epoch, "view_id": key, "start_frame": start, "owner_event": "", "kind": "background"})\n    rng.shuffle(index)\n    return index, manifest\n\n\ndef normalize_window(x, mean, scale, cfg, rng=None):\n    x = x.astype(np.float32, copy=True)\n    if rng is not None and cfg.gain_augment and rng.random() < cfg.gain_probability:\n        gains = rng.uniform(cfg.gain_low, cfg.gain_high, 3).astype(np.float32)\n        x *= np.r_[gains, gains][:, None, None]\n    x = (x-mean[:, None, None])/scale[:, None, None]\n    if rng is not None and cfg.feature_noise_std:\n        x += rng.normal(0, cfg.feature_noise_std, x.shape).astype(np.float32)\n    return np.clip(x, -8, 8).astype(np.float32)\n\n\ndef raw_window(raw, key, start, win):\n    x = raw[key][0][:, start:start+win]\n    return np.pad(x, ((0, 0), (0, win-x.shape[1]), (0, 0)), mode="edge")\n\n\nclass WindowDataset(Dataset):\n    def __init__(self, index, raw, views, mean, scale, cfg, augment=False, epoch=0):\n        self.index, self.raw = index, raw\n        self.mean, self.scale, self.cfg = mean, scale, cfg\n        self.rng = np.random.default_rng(cfg.seed+100000+epoch) if augment else None\n        keys = {key for key, _ in index}\n        self.targets = {key: targets_for(raw[key][1], views[key]["events"]) for key in keys}\n\n    def __len__(self):\n        return len(self.index)\n\n    def __getitem__(self, item):\n        key, start = self.index[item]\n        win = self.cfg.window_frames\n        x = normalize_window(raw_window(self.raw, key, start, win), self.mean, self.scale, self.cfg, self.rng)\n        targets, weights = self.targets[key]\n        n = min(win, targets.shape[1]-start)\n        target = np.pad(targets[:, start:start+n], ((0, 0), (0, win-n)))\n        weight = np.pad(weights[start:start+n], (0, win-n))\n        return torch.from_numpy(x), torch.from_numpy(target), torch.from_numpy(weight)\n\n\n# %% CNN and temporal convolution network\nclass TemporalBlock(nn.Module):\n    def __init__(self, cin, cout, dilation):\n        super().__init__()\n        padding = 2*dilation\n        self.net = nn.Sequential(\n            nn.Conv1d(cin, cout, 5, padding=padding, dilation=dilation),\n            nn.GroupNorm(8, cout), nn.GELU(), nn.Dropout(0.20),\n            nn.Conv1d(cout, cout, 5, padding=padding, dilation=dilation),\n            nn.GroupNorm(8, cout), nn.GELU(), nn.Dropout(0.20))\n        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()\n\n    def forward(self, x):\n        return self.net(x)+self.skip(x)\n\n\nclass CNNTCNV4(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.cnn = nn.Sequential(\n            nn.Conv2d(6, 24, (1, 7), padding=(0, 3)), nn.GroupNorm(6, 24), nn.GELU(), nn.Dropout2d(0.10),\n            nn.Conv2d(24, 48, (1, 5), padding=(0, 2)), nn.GroupNorm(6, 48), nn.GELU(),\n            nn.AdaptiveAvgPool2d((None, 1)))\n        self.tcn = nn.Sequential(TemporalBlock(48, 64, 1), TemporalBlock(64, 64, 2), TemporalBlock(64, 96, 4))\n        self.dropout = nn.Dropout(0.25)\n        self.action_head, self.start_head, self.end_head = nn.Conv1d(96, 1, 1), nn.Conv1d(96, 1, 1), nn.Conv1d(96, 1, 1)\n\n    def forward(self, x):\n        z = self.dropout(self.tcn(self.cnn(x).squeeze(-1)))\n        return torch.cat([self.action_head(z), self.start_head(z), self.end_head(z)], dim=1)\n\n\ndef run_epoch(model, loader, device, optimizer=None):\n    model.train(optimizer is not None)\n    total, denominator = 0.0, 0.0\n    with torch.set_grad_enabled(optimizer is not None):\n        for x, target, weights in loader:\n            x, target, weights = x.to(device), target.to(device), weights.to(device)\n            logits = model(x)\n            loss_map = F.binary_cross_entropy_with_logits(logits, target, reduction="none",\n                       pos_weight=logits.new_tensor([3, 5, 5]).view(1, 3, 1))\n            numerator = (loss_map*logits.new_tensor([1, 0.5, 0.5]).view(1, 3, 1)*weights[:, None]).sum()\n            count = weights.sum()\n            loss = numerator/count.clamp_min(1)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite loss")\n            if optimizer is not None and float(count) > 0:\n                optimizer.zero_grad(set_to_none=True)\n                loss.backward()\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n            total += float(numerator.detach())\n            denominator += float(count)\n    return total/max(denominator, 1)\n\n\n# %% Sliding inference and event decoding\ndef infer_view(model, data, mean, scale, cfg, device):\n    features, times = data\n    starts = window_starts(len(times), cfg.window_frames, cfg.infer_hop_frames)\n    sums, counts = np.zeros((3, len(times)), np.float32), np.zeros(len(times), np.float32)\n    model.eval()\n    with torch.no_grad():\n        for offset in range(0, len(starts), cfg.batch_size):\n            batch_starts = starts[offset:offset+cfg.batch_size]\n            batch = []\n            for start in batch_starts:\n                piece = features[:, start:start+cfg.window_frames]\n                piece = np.pad(piece, ((0, 0), (0, cfg.window_frames-piece.shape[1]), (0, 0)), mode="edge")\n                batch.append(normalize_window(piece, mean, scale, cfg))\n            output = model(torch.from_numpy(np.stack(batch)).to(device)).sigmoid().cpu().numpy()\n            for row, start in enumerate(batch_starts):\n                n = min(cfg.window_frames, len(times)-start)\n                sums[:, start:start+n] += output[row, :, :n]\n                counts[start:start+n] += 1\n    if not np.all(counts > 0):\n        raise AssertionError("Inference left frames uncovered")\n    return times, sums/counts[None]\n\n\ndef infer_side(model, raw, views, side, mean, scale, cfg, device, verbose=False):\n    keys = [key for key, v in views.items() if v["side"] == side]\n    tracks = {}\n    for i, key in enumerate(keys, 1):\n        if verbose and (i == 1 or i % 25 == 0 or i == len(keys)):\n            print(f"Inference {side}: {i}/{len(keys)}", flush=True)\n        tracks[key] = infer_view(model, raw[key], mean, scale, cfg, device)\n    return tracks\n\n\ndef smooth(x, width):\n    return np.convolve(np.pad(x, (width//2, width//2), mode="edge"), np.ones(width)/width, mode="valid")\n\n\ndef decode(times, probs, duration, high=0.6, low=0.2, min_s=0.5, max_s=8.0, merge_gap=0.5):\n    action, pstart, pend = smooth(probs[0], 5), smooth(probs[1], 3), smooth(probs[2], 3)\n    proposals = []\n    for a, b in boolean_runs(action >= low):\n        if action[a:b].max() < high:\n            continue\n        left, right = max(0, a-10), min(len(times), b+10)\n        s = left+int(np.argmax(pstart[left:min(len(times), a+11)]))\n        base = max(0, b-11)\n        e = base+int(np.argmax(pend[base:right]))\n        start, end = float(times[s]), float(times[e]+0.05)\n        if not min_s <= end-start <= max_s:\n            start, end = float(times[a]-0.05), float(times[b-1]+0.05)\n        start, end = max(0, start), min(duration, end)\n        if not min_s-1e-6 <= end-start <= max_s+1e-6:\n            continue\n        proposals.append({"t_start": start, "t_end": end, "score": float(np.percentile(action[a:b], 90)), "t_alarm": None})\n    merged = []\n    for event in sorted(proposals, key=lambda p: p["t_start"]):\n        if merged and event["t_start"]-merged[-1]["t_end"] <= merge_gap and max(event["t_end"], merged[-1]["t_end"])-merged[-1]["t_start"] <= max_s:\n            merged[-1]["t_end"] = max(event["t_end"], merged[-1]["t_end"])\n            merged[-1]["score"] = max(event["score"], merged[-1]["score"])\n        else:\n            merged.append(event)\n    return merged\n\n\ndef predict_tracks(tracks, views, decoder):\n    return [dict(event, stream_id=key) for key, (times, probs) in tracks.items()\n            for event in decode(times, probs, views[key]["duration"], **decoder)]\n\n\n# %% Event metrics and frame metrics\ndef event_ground(views, keys, ignore_flagged=True):\n    ground = []\n    for key in keys:\n        for e in views[key]["events"]:\n            flagged = str(e["quality"]).lower() == "low" or truth(e["needs_review"])\n            ground.append({"stream_id": key, "t_start": e["t_start"], "t_end": e["t_end"],\n                           "ignore": truth(e["ignore"]) or (ignore_flagged and flagged)})\n    return ground\n\n\ndef event_report(evaluate, predictions, views, keys, ignore_flagged=True):\n    return evaluate(event_ground(views, keys, ignore_flagged), predictions,\n                    {key: views[key]["duration"] for key in keys})\n\n\ndef rank_key(report):\n    r = report["ranking_score"]\n    a, b = r["start_error_median_abs"], r["false_alarms_per_hour"]\n    return r["f1_mean"], -(a if a is not None else float("inf")), -(b if b is not None else float("inf"))\n\n\ndef frame_labels(times, events):\n    positive, ignored = np.zeros(len(times), bool), np.zeros(len(times), bool)\n    for e in events:\n        flagged = truth(e["ignore"]) or str(e["quality"]).lower() == "low" or truth(e["needs_review"])\n        if flagged:\n            ignored |= (times >= e["t_start"]-2) & (times <= e["t_end"]+2)\n        else:\n            positive |= (times >= e["t_start"]) & (times < e["t_end"])\n    return positive, ~ignored | positive\n\n\ndef frame_metrics(tracks, views):\n    tn = fp = fn = tp = 0\n    for key, (times, probs) in tracks.items():\n        y, mask = frame_labels(times, views[key]["events"])\n        pred = probs[0] >= 0.5\n        tn += int(np.sum(mask & ~y & ~pred))\n        fp += int(np.sum(mask & ~y & pred))\n        fn += int(np.sum(mask & y & ~pred))\n        tp += int(np.sum(mask & y & pred))\n    total = tn+fp+fn+tp\n    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp, "n_frames": total,\n            "accuracy": (tp+tn)/total if total else None,\n            "f1": 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0,\n            "balanced_accuracy": 0.5*(tp/(tp+fn)+tn/(tn+fp)) if tp+fn and tn+fp else None,\n            "threshold": 0.5, "sample_rate_hz": 10,\n            "scope": "unique pooled frames, ignore-flagged zones +/-2s excluded, evaluable positives take precedence"}\n\n\ndef export_split(out, name, tracks, views, decoder, evaluate, validate_predictions):\n    predictions = predict_tracks(tracks, views, decoder)\n    path = out/f"cnn_tcn_v4_{name}_submission.csv"\n    pd.DataFrame(predictions, columns=["stream_id", "t_start", "t_end", "score"]).to_csv(path, index=False)\n    checked = validate_predictions(path, {key: views[key]["duration"] for key in tracks})\n    report = {"split": name, "decoder": decoder,\n              "ignore_flagged": event_report(evaluate, checked, views, tracks),\n              "all_events": event_report(evaluate, checked, views, tracks, False),\n              "frame_metrics": frame_metrics(tracks, views)}\n    write_json(out/f"cnn_tcn_v4_{name}_report.json", report)\n    return report\n\n\n# %% Training and final evaluation orchestration\ndef prepare_fold(root, out, cache_root, holdout, cfg, restore=False):\n    cfg.validate()\n    seed_everything(cfg.seed)\n    if (out/"config.json").exists() and not restore:\n        raise FileExistsError(f"Existing experiment: {out}. Choose a fresh run directory.")\n    out.mkdir(parents=True, exist_ok=True)\n    sys.path.insert(0, str(root/"src"))\n    from abc2027.evaluate import evaluate, validate_predictions\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats()\n    env = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,\n           "device": str(device), "cuda": torch.version.cuda,\n           "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None}\n    write_json(out/"environment.json", env)\n    configuration = dict(asdict(cfg), holdout_groups=sorted(holdout), root=str(root),\n                         model="CNN24-48_TCN64-64-96", protocol="one source-group fold, isolated views",\n                         normalizer="fit-only per-channel z-score, std floor 1e-8",\n                         feature_dtype="float32", hard_negative_mining_enabled=False,\n                         frame_metric_threshold=0.5, official_metric="mean event F1 at tIoU 0.1/0.3/0.5")\n    if not restore:\n        write_json(out/"config.json", configuration)\n    views, raw, manifest, skipped = build_views(root, cache_root, set(holdout), cfg)\n    pd.DataFrame(manifest).to_csv(out/"split_manifest.csv", index=False)\n    pd.DataFrame(skipped, columns=["source_stream", "side", "start", "end", "reason"]).to_csv(out/"skipped_intervals.csv", index=False)\n    mean, scale = fit_normalizer(raw, views)\n    np.savez(out/"normalizer_stats.npz", mean=mean, scale=scale)\n    positives, negatives, fit_eval, val_eval = candidate_windows(raw, views, cfg)\n    audit = {"fit_views": sum(v["side"] == "fit" for v in views.values()),\n             "holdout_views": sum(v["side"] == "holdout" for v in views.values()),\n             "fit_hours": sum(v["duration"] for v in views.values() if v["side"] == "fit")/3600,\n             "holdout_hours": sum(v["duration"] for v in views.values() if v["side"] == "holdout")/3600,\n             "fit_event_owners": len(positives), "candidate_negative_windows": len(negatives),\n             "fixed_train_eval_windows": len(fit_eval), "holdout_eval_windows": len(val_eval),\n             "skipped_intervals": len(skipped), "truncated_events": sum(r["truncated_events"] for r in manifest),\n             "shared_fit_holdout_groups": [], "filter_state_reset_per_view": True,\n             "smoke_only": cfg.smoke}\n    write_json(out/"data_audit.json", audit)\n    print("Data audit:", audit, flush=True)\n    return dict(root=root, out=out, cache_root=cache_root, holdout=set(holdout), cfg=cfg,\n                views=views, raw=raw, mean=mean, scale=scale, positives=positives, negatives=negatives,\n                fit_eval=fit_eval, val_eval=val_eval, device=device, evaluate=evaluate, validator=validate_predictions,\n                config=configuration, audit=audit)\n\n\ndef make_loader(index, ctx, augment=False, epoch=0, shuffle=False):\n    dataset = WindowDataset(index, ctx["raw"], ctx["views"], ctx["mean"], ctx["scale"], ctx["cfg"], augment, epoch)\n    generator = torch.Generator().manual_seed(ctx["cfg"].seed+epoch)\n    return DataLoader(dataset, batch_size=ctx["cfg"].batch_size, shuffle=shuffle,\n                      num_workers=0, generator=generator, pin_memory=ctx["device"].type == "cuda")\n\n\ndef train_fold(ctx):\n    cfg, out = ctx["cfg"], ctx["out"]\n    model = CNNTCNV4().to(ctx["device"])\n    ctx["config"]["model_parameters"] = sum(p.numel() for p in model.parameters())\n    write_json(out/"config.json", ctx["config"])\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)\n    fit_loader, val_loader = make_loader(ctx["fit_eval"], ctx), make_loader(ctx["val_eval"], ctx)\n    fixed = {"high": 0.6, "low": 0.2, "min_s": 0.5, "max_s": 8.0, "merge_gap": 0.5}\n    history, sampling, checkpoint_rows, best_key, best_epoch, best_state, stale = [], [], [], None, 0, None, 0\n    for epoch in range(1, (1 if cfg.smoke else cfg.max_epochs)+1):\n        started = time.time()\n        index, manifest = sample_epoch(ctx["positives"], ctx["negatives"], ctx["raw"], ctx["views"], cfg, epoch)\n        sampling.extend(manifest)\n        augmented_loss = run_epoch(model, make_loader(index, ctx, True, epoch, True), ctx["device"], optimizer)\n        clean_loss = run_epoch(model, fit_loader, ctx["device"])\n        val_loss = run_epoch(model, val_loader, ctx["device"])\n        scheduler.step(val_loss)\n        tracks = infer_side(model, ctx["raw"], ctx["views"], "holdout", ctx["mean"], ctx["scale"], cfg, ctx["device"])\n        report = event_report(ctx["evaluate"], predict_tracks(tracks, ctx["views"], fixed), ctx["views"], tracks)\n        fm = frame_metrics(tracks, ctx["views"])\n        row = {"epoch": epoch, "augmented_train_loss": augmented_loss, "train_loss": clean_loss, "val_loss": val_loss,\n               "val_event_f1_mean": report["ranking_score"]["f1_mean"], "val_frame_f1": fm["f1"],\n               "val_frame_accuracy": fm["accuracy"], "lr": optimizer.param_groups[0]["lr"],\n               "train_windows": len(index), "seconds": time.time()-started}\n        history.append(row)\n        checkpoint_rows.append(dict(epoch=epoch, **report["ranking_score"]))\n        pd.DataFrame(history).to_csv(out/"training_history_v4.csv", index=False)\n        pd.DataFrame(checkpoint_rows).to_csv(out/"checkpoint_event_f1.csv", index=False)\n        pd.DataFrame(sampling).to_csv(out/"epoch_sampling_manifest.csv", index=False)\n        checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                      "epoch": epoch, "config": ctx["config"], "mean": ctx["mean"].tolist(),\n                      "scale": ctx["scale"].tolist(), "decoder": fixed, "history": history}\n        torch.save(checkpoint, out/f"checkpoint_epoch_{epoch:02d}.pt")\n        key = (*rank_key(report), -val_loss)\n        if best_key is None or key > best_key:\n            best_key, best_state, best_epoch, stale = key, copy.deepcopy(model.state_dict()), epoch, 0\n            torch.save(checkpoint, out/"best_checkpoint_fixed_decoder.pt")\n        else:\n            stale += 1\n        print(f"Epoch {epoch:02d}: train(clean)={clean_loss:.4f}, val={val_loss:.4f}, "\n              f"event F1={row[\'val_event_f1_mean\']:.4f}, frame acc={fm[\'accuracy\']}, {row[\'seconds\']:.1f}s", flush=True)\n        if stale >= cfg.patience:\n            print("Early stopping on holdout event metrics", flush=True)\n            break\n    model.load_state_dict(best_state)\n    ctx.update(model=model, history=history, best_epoch=best_epoch, fixed_decoder=fixed)\n    return ctx\n\n\ndef evaluate_fold(ctx):\n    cfg, out, model = ctx["cfg"], ctx["out"], ctx["model"]\n    tracks = infer_side(model, ctx["raw"], ctx["views"], "holdout", ctx["mean"], ctx["scale"], cfg, ctx["device"])\n    grid, best_decoder, best_key = [], None, None\n    for high in [0.4, 0.5, 0.6, 0.7, 0.8]:\n        for low in [0.1, 0.2, 0.3]:\n            decoder = dict(ctx["fixed_decoder"], high=high, low=low)\n            report = event_report(ctx["evaluate"], predict_tracks(tracks, ctx["views"], decoder), ctx["views"], tracks)\n            grid.append(dict(high=high, low=low, **report["ranking_score"]))\n            if best_key is None or rank_key(report) > best_key:\n                best_key, best_decoder = rank_key(report), decoder\n    pd.DataFrame(grid).sort_values(["f1_mean", "start_error_median_abs", "false_alarms_per_hour"],\n                                 ascending=[False, True, True]).to_csv(out/"cnn_tcn_v4_decoder_grid.csv", index=False)\n    reports = {"Holdout": export_split(out, "loso", tracks, ctx["views"], best_decoder, ctx["evaluate"], ctx["validator"])}\n    train_tracks = infer_side(model, ctx["raw"], ctx["views"], "fit", ctx["mean"], ctx["scale"], cfg, ctx["device"], True)\n    reports["Train"] = export_split(out, "train", train_tracks, ctx["views"], best_decoder, ctx["evaluate"], ctx["validator"])\n    # Model/decoder selection is complete before public data are processed.\n    torch.save({"model": model.state_dict(), "config": ctx["config"], "epoch": ctx["best_epoch"],\n                "mean": ctx["mean"].tolist(), "scale": ctx["scale"].tolist(), "decoder": best_decoder},\n               out/"cnn_tcn_v4_best_event_f1.pt")\n    pv, pr, pm, _ = build_views(ctx["root"], ctx["cache_root"], ctx["holdout"], cfg, public=True)\n    public_tracks = infer_side(model, pr, pv, "public", ctx["mean"], ctx["scale"], cfg, ctx["device"], True)\n    reports["Public"] = export_split(out, "public", public_tracks, pv, best_decoder, ctx["evaluate"], ctx["validator"])\n    for split, report in reports.items():\n        report.update(version="v4", best_epoch=ctx["best_epoch"], holdout_groups=sorted(ctx["holdout"]),\n                      normalization="fit-only zscore", smoke_only=cfg.smoke)\n        filename = {"Train": "train", "Holdout": "loso", "Public": "public"}[split]\n        write_json(out/f"cnn_tcn_v4_{filename}_report.json", report)\n    pd.DataFrame(pm).to_csv(out/"public_manifest.csv", index=False)\n    for key, (times, probs) in public_tracks.items():\n        np.savez_compressed(out/f"probabilities_{key}.npz", times=times, action=probs[0], start=probs[1], end=probs[2])\n    save_public_timeline(out, public_tracks, pv, best_decoder)\n    rows = []\n    for name in ["Train", "Holdout", "Public"]:\n        r = reports[name]\n        rows.append({"Split": name, "Event F1 mean": r["ignore_flagged"]["ranking_score"]["f1_mean"],\n                     "Event F1 @0.3": r["ignore_flagged"]["primary"]["f1"],\n                     "Frame F1": r["frame_metrics"]["f1"], "Frame Accuracy": r["frame_metrics"]["accuracy"],\n                     "Frame Balanced Acc": r["frame_metrics"]["balanced_accuracy"]})\n    matrix = pd.DataFrame(rows).set_index("Split")\n    matrix.to_csv(out/"metrics_matrix_v4.csv")\n    pd.DataFrame(columns=["score", "stream_id", "start"]).to_csv(out/"hard_negative_manifest.csv", index=False)\n    summary = {"version": "v4", "config": ctx["config"], "best_epoch": ctx["best_epoch"],\n               "decoder": best_decoder, "data_audit": ctx["audit"], "reports": reports,\n               "peak_gpu_allocated_mib": torch.cuda.max_memory_allocated()/2**20 if ctx["device"].type == "cuda" else 0,\n               "limitations": ["One source-group fold is not full LOSO CV.",\n                               "Holdout spans omit mixed crossfades and reset preprocessing; public remains continuous.",\n                               "Internal model/threshold selection makes holdout scores optimistic.",\n                               "Frame accuracy is not event accuracy and can be high due to background imbalance.",\n                               "CNN/TCN uses future context; no measured online alarm latency.",\n                               "Smoke runs evaluate a small data slice and are not benchmark scores."]}\n    write_json(out/"report_v4.json", summary)\n    print("Best epoch:", ctx["best_epoch"], "public:", reports["Public"]["ignore_flagged"]["ranking_score"], flush=True)\n    return {"out": out, "history": ctx["history"], "reports": reports, "matrix": matrix,\n            "best_epoch": ctx["best_epoch"], "smoke": cfg.smoke}\n\n\ndef zip_outputs(out):\n    archive = out/"cnn_tcn_v4_outputs.zip"\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:\n        for path in sorted(out.iterdir()):\n            if path.is_file() and path != archive and not path.name.startswith("checkpoint_epoch_"):\n                handle.write(path, arcname=path.name)\n    return archive\n\n\ndef save_public_timeline(out, tracks, views, decoder):\n    import matplotlib.pyplot as plt\n    key = sorted(tracks)[0]\n    times, probs = tracks[key]\n    fig, ax = plt.subplots(figsize=(15, 4))\n    ax.plot(times, probs[0], lw=0.7, label="Action probability")\n    for event in views[key]["events"]:\n        ax.axvspan(event["t_start"], event["t_end"], color="green", alpha=0.12)\n    for event in decode(times, probs, views[key]["duration"], **decoder):\n        ax.axvspan(event["t_start"], event["t_end"], color="red", alpha=0.18)\n    ax.set(xlabel="Seconds", ylabel="Probability",\n           title=f"{key}: green = annotation, red = prediction")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out/"public_timeline_v4.png", dpi=150)\n    plt.close(fig)\n\n\ndef load_result(out):\n    summary = json.loads((out/"report_v4.json").read_text(encoding="utf-8"))\n    return {"out": out, "history": pd.read_csv(out/"training_history_v4.csv").to_dict("records"),\n            "reports": summary["reports"], "matrix": pd.read_csv(out/"metrics_matrix_v4.csv").set_index("Split"),\n            "best_epoch": summary["best_epoch"], "smoke": summary["config"]["smoke"]}\n\n\ndef export_existing_fold(root, out, cache_root):\n    """Recover exports after an interrupted Colab run without calling train."""\n    if (out/"report_v4.json").exists():\n        return load_result(out)\n    checkpoint = out/"best_checkpoint_fixed_decoder.pt"\n    if not checkpoint.exists():\n        raise FileNotFoundError(f"No trained checkpoint to recover: {checkpoint}")\n    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    cfg = Config(**{k: v for k, v in payload["config"].items() if k in Config.__dataclass_fields__})\n    ctx = prepare_fold(root, out, cache_root, payload["config"]["holdout_groups"], cfg, restore=True)\n    if not (np.allclose(ctx["mean"], payload["mean"], rtol=1e-5, atol=1e-12) and\n            np.allclose(ctx["scale"], payload["scale"], rtol=1e-5, atol=1e-12)):\n        raise ValueError("Recovered data/normalizer differs from the trained checkpoint")\n    model = CNNTCNV4().to(ctx["device"])\n    model.load_state_dict(payload["model"])\n    ctx.update(model=model, best_epoch=payload["epoch"], history=pd.read_csv(out/"training_history_v4.csv").to_dict("records"),\n               fixed_decoder=payload["decoder"])\n    return evaluate_fold(ctx)\n'
base=types.ModuleType('cnn_tcn_v4')
sys.modules['cnn_tcn_v4']=base
exec(compile(V4_SOURCE,'cnn_tcn_v4.py','exec'),base.__dict__)
V5_SOURCE='"""V5 controlled window/model experiments. No training runs on import."""\nfrom __future__ import annotations\n\nimport gc\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport cnn_tcn_v4 as base\n\nnp, pd, torch, nn, F = base.np, base.pd, base.torch, base.nn, base.F\njson, time, copy = base.json, base.time, base.copy\n\n\n@dataclass\nclass ConfigV5(base.Config):\n    lr: float = 1e-4\n    patience: int = 7\n    min_epochs: int = 10\n    gain_augment: bool = False\n    model_size: str = "standard"\n    temporal_kernel: int = 5\n    steps_per_epoch: int = 110\n\n    def validate(self):\n        # The inherited validator intentionally fixes V4 at 8 seconds.\n        reference = base.Config(**{k: getattr(self, k) for k in base.Config.__dataclass_fields__})\n        reference.window_frames = 80\n        reference.validate()\n        if self.window_frames not in (40, 60, 80):\n            raise ValueError("Choose 40, 60 or 80 pooled frames (4/6/8 seconds).")\n        if self.model_size not in ("standard", "small") or self.temporal_kernel not in (3, 5):\n            raise ValueError("Invalid model size or temporal kernel.")\n        if self.steps_per_epoch < 1 or not 1 <= self.min_epochs <= self.max_epochs:\n            raise ValueError("Check steps_per_epoch, min_epochs and max_epochs.")\n        if self.lr <= 0 or self.weight_decay < 0:\n            raise ValueError("Invalid optimizer settings.")\n        if self.smoke:\n            raise ValueError("V5 is for full Colab experiments; smoke training is disabled.")\n\n\ndef experiment_suite():\n    return [\n        ("A_8s_gain_lr3e4", dict(window_frames=80, gain_augment=True, lr=3e-4)),\n        ("B_8s_no_gain_lr3e4", dict(window_frames=80, gain_augment=False, lr=3e-4)),\n        ("C_8s_no_gain_lr1e4", dict(window_frames=80, gain_augment=False, lr=1e-4)),\n        ("D_6s_no_gain_lr1e4", dict(window_frames=60, gain_augment=False, lr=1e-4)),\n        ("E_4s_no_gain_lr1e4", dict(window_frames=40, gain_augment=False, lr=1e-4)),\n        ("F_6s_small_lr1e4", dict(window_frames=60, gain_augment=False, lr=1e-4, model_size="small")),\n    ]\n\n\ndef group_count(channels, preferred):\n    return next(n for n in range(min(channels, preferred), 0, -1) if channels % n == 0)\n\n\nclass TemporalBlockV5(nn.Module):\n    def __init__(self, cin, cout, dilation, kernel):\n        super().__init__()\n        padding = dilation*(kernel-1)//2\n        self.net = nn.Sequential(\n            nn.Conv1d(cin, cout, kernel, padding=padding, dilation=dilation),\n            nn.GroupNorm(group_count(cout, 8), cout), nn.GELU(), nn.Dropout(0.20),\n            nn.Conv1d(cout, cout, kernel, padding=padding, dilation=dilation),\n            nn.GroupNorm(group_count(cout, 8), cout), nn.GELU(), nn.Dropout(0.20))\n        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()\n\n    def forward(self, x):\n        return self.net(x)+self.skip(x)\n\n\nclass CNNTCNV5(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        c1, c2, t1, t2, t3 = (24, 48, 64, 64, 96) if cfg.model_size == "standard" else (16, 32, 32, 48, 64)\n        self.cnn = nn.Sequential(\n            nn.Conv2d(6, c1, (1, 7), padding=(0, 3)),\n            nn.GroupNorm(group_count(c1, 6), c1), nn.GELU(), nn.Dropout2d(0.10),\n            nn.Conv2d(c1, c2, (1, 5), padding=(0, 2)),\n            nn.GroupNorm(group_count(c2, 6), c2), nn.GELU(), nn.AdaptiveAvgPool2d((None, 1)))\n        self.tcn = nn.Sequential(\n            TemporalBlockV5(c2, t1, 1, cfg.temporal_kernel),\n            TemporalBlockV5(t1, t2, 2, cfg.temporal_kernel),\n            TemporalBlockV5(t2, t3, 4, cfg.temporal_kernel))\n        self.dropout = nn.Dropout(0.25)\n        self.action_head = nn.Conv1d(t3, 1, 1)\n        self.start_head = nn.Conv1d(t3, 1, 1)\n        self.end_head = nn.Conv1d(t3, 1, 1)\n\n    def forward(self, x):\n        z = self.dropout(self.tcn(self.cnn(x).squeeze(-1)))\n        return torch.cat([self.action_head(z), self.start_head(z), self.end_head(z)], dim=1)\n\n\ndef loss_pass(model, loader, device, optimizer=None):\n    model.train(optimizer is not None)\n    sums, denominator, updates = np.zeros(3, dtype=np.float64), 0.0, 0\n    with torch.set_grad_enabled(optimizer is not None):\n        for x, target, weights in loader:\n            x, target, weights = x.to(device), target.to(device), weights.to(device)\n            logits = model(x)\n            loss_map = F.binary_cross_entropy_with_logits(\n                logits, target, reduction="none", pos_weight=logits.new_tensor([3, 5, 5]).view(1, 3, 1))\n            heads = (loss_map*weights[:, None]).sum(dim=(0, 2))\n            count = weights.sum()\n            loss = (heads*logits.new_tensor([1, .5, .5])).sum()/count.clamp_min(1)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite loss")\n            if optimizer is not None and float(count) > 0:\n                optimizer.zero_grad(set_to_none=True)\n                loss.backward()\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n                updates += 1\n            sums += heads.detach().cpu().numpy()\n            denominator += float(count)\n    components = sums/max(denominator, 1)\n    return dict(loss=float(components @ np.array([1, .5, .5])), action_loss=float(components[0]),\n                start_loss=float(components[1]), end_loss=float(components[2]),\n                valid_weighted_frames=denominator, optimizer_updates=updates)\n\n\ndef prepare_v5(root, out, cache_root, holdout, cfg, restore=False):\n    ctx = base.prepare_fold(root, out, cache_root, holdout, cfg, restore)\n    ctx["config"].update(version="v5", model=f"{cfg.model_size}_kernel{cfg.temporal_kernel}",\n                         window_seconds=cfg.window_frames/cfg.target_hz,\n                         sampling="V4 event candidate pool, then fixed replacement draw budget",\n                         conv_only_receptive_field_s=(1+2*(cfg.temporal_kernel-1)*7)/cfg.target_hz,\n                         normalization_note="GroupNorm also uses time/spatial dimensions; conv receptive field is not a strict context limit")\n    if not restore:\n        base.write_json(out/"config.json", ctx["config"])\n    return ctx\n\n\ndef train_v5(ctx):\n    cfg, out = ctx["cfg"], ctx["out"]\n    model = CNNTCNV5(cfg).to(ctx["device"])\n    ctx["config"]["model_parameters"] = sum(p.numel() for p in model.parameters())\n    base.write_json(out/"config.json", ctx["config"])\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=.5, patience=1)\n    fit_loader, val_loader = base.make_loader(ctx["fit_eval"], ctx), base.make_loader(ctx["val_eval"], ctx)\n    fixed = dict(high=.6, low=.2, min_s=.5, max_s=8.0, merge_gap=.5)\n    history, sampling, checkpoint_rows = [], [], []\n    best_key, best_epoch, stale, updates, best_loss = None, 0, 0, 0, float("inf")\n    for epoch in range(1, cfg.max_epochs+1):\n        started, lr_used = time.time(), optimizer.param_groups[0]["lr"]\n        pool, manifest = base.sample_epoch(ctx["positives"], ctx["negatives"], ctx["raw"], ctx["views"], cfg, epoch)\n        # The cap of three is a candidate-pool cap, not a cap on actual exposures.\n        lookup = {(r["view_id"], r["start_frame"]): r for r in manifest}\n        rng = np.random.default_rng(cfg.seed+200000+epoch)\n        indices = rng.integers(0, len(pool), cfg.steps_per_epoch*cfg.batch_size)\n        actual = [pool[i] for i in indices]\n        for draw, item in enumerate(actual):\n            sampling.append(dict(lookup[item], draw=draw, optimizer_step=draw//cfg.batch_size+1))\n        augmented = loss_pass(model, base.make_loader(actual, ctx, True, epoch, False), ctx["device"], optimizer)\n        clean, val = loss_pass(model, fit_loader, ctx["device"]), loss_pass(model, val_loader, ctx["device"])\n        updates += augmented["optimizer_updates"]\n        scheduler.step(val["loss"])\n        tracks = base.infer_side(model, ctx["raw"], ctx["views"], "holdout", ctx["mean"], ctx["scale"], cfg, ctx["device"])\n        report = base.event_report(ctx["evaluate"], base.predict_tracks(tracks, ctx["views"], fixed), ctx["views"], tracks)\n        fm = base.frame_metrics(tracks, ctx["views"])\n        row = dict(epoch=epoch, augmented_train_loss=augmented["loss"], train_loss=clean["loss"],\n                   val_loss=val["loss"], loss_gap=val["loss"]-clean["loss"],\n                   val_event_f1_mean=report["ranking_score"]["f1_mean"], val_frame_f1=fm["f1"],\n                   val_frame_accuracy=fm["accuracy"], val_frame_balanced_accuracy=fm["balanced_accuracy"],\n                   val_false_alarms_per_hour=report["primary"]["false_alarms_per_hour"],\n                   lr=lr_used, next_lr=optimizer.param_groups[0]["lr"],\n                   optimizer_updates=augmented["optimizer_updates"], cumulative_updates=updates,\n                   candidate_windows=len(pool), train_windows=len(actual), unique_drawn_windows=len(set(actual)),\n                   weighted_training_frames=augmented["valid_weighted_frames"], seconds=time.time()-started)\n        for label, metrics in [("train", clean), ("val", val)]:\n            row.update({f"{label}_{name}": metrics[name] for name in ("action_loss", "start_loss", "end_loss")})\n        history.append(row)\n        checkpoint_rows.append(dict(epoch=epoch, **report["ranking_score"]))\n        pd.DataFrame(history).to_csv(out/"training_history_v5.csv", index=False)\n        pd.DataFrame(sampling).to_csv(out/"epoch_sampling_manifest.csv", index=False)\n        pd.DataFrame(checkpoint_rows).to_csv(out/"checkpoint_event_f1.csv", index=False)\n        payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), config=ctx["config"],\n                       epoch=epoch, mean=ctx["mean"].tolist(), scale=ctx["scale"].tolist(), decoder=fixed)\n        torch.save(payload, out/f"checkpoint_epoch_{epoch:02d}.pt")\n        if val["loss"] < best_loss:\n            best_loss = val["loss"]\n            torch.save(payload, out/"best_checkpoint_val_loss.pt")\n        key = (*base.rank_key(report), -val["loss"])\n        if best_key is None or key > best_key:\n            best_key, best_epoch, stale = key, epoch, 0\n            torch.save(payload, out/"best_checkpoint_fixed_decoder.pt")\n        else:\n            stale += 1\n        print(f"Epoch {epoch:02d}: clean train={clean[\'loss\']:.4f} val={val[\'loss\']:.4f} "\n              f"F1={row[\'val_event_f1_mean\']:.4f} FA/h={row[\'val_false_alarms_per_hour\']:.2f}", flush=True)\n        if epoch >= cfg.min_epochs and stale >= cfg.patience:\n            break\n    base.write_json(out/"training_complete.json", dict(best_epoch=best_epoch, trained_epochs=len(history),\n                                                     cumulative_updates=updates))\n    return restore_v5(ctx["root"], out, ctx["cache_root"], ctx=ctx)\n\n\ndef restore_v5(root, out, cache_root, ctx=None):\n    checkpoint = out/"best_checkpoint_fixed_decoder.pt"\n    if not checkpoint.is_file():\n        raise FileNotFoundError(f"No checkpoint: {checkpoint}")\n    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    cfg = ConfigV5(**{k: v for k, v in payload["config"].items() if k in ConfigV5.__dataclass_fields__})\n    if ctx is None:\n        ctx = prepare_v5(root, out, cache_root, payload["config"]["holdout_groups"], cfg, restore=True)\n    if not (np.allclose(ctx["mean"], payload["mean"], rtol=1e-5, atol=1e-12) and\n            np.allclose(ctx["scale"], payload["scale"], rtol=1e-5, atol=1e-12)):\n        raise ValueError("Data/normalizer differs from checkpoint")\n    ctx["config"] = payload["config"]\n    ctx["model"] = CNNTCNV5(cfg).to(ctx["device"])\n    ctx["model"].load_state_dict(payload["model"])\n    ctx.update(best_epoch=payload["epoch"], fixed_decoder=payload["decoder"],\n               history=pd.read_csv(out/"training_history_v5.csv").to_dict("records"))\n    return ctx\n\n\ndef export_split_v5(ctx, name, tracks, views, decoder):\n    out = ctx["out"]\n    path = out/f"cnn_tcn_v5_{name}_submission.csv"\n    pd.DataFrame(base.predict_tracks(tracks, views, decoder),\n                 columns=["stream_id", "t_start", "t_end", "score"]).to_csv(path, index=False)\n    predictions = ctx["validator"](path, {key: views[key]["duration"] for key in tracks})\n    report = dict(split=name, version="v5", best_epoch=ctx["best_epoch"], decoder=decoder,\n                  holdout_groups=sorted(ctx["holdout"]), config=ctx["config"],\n                  ignore_flagged=base.event_report(ctx["evaluate"], predictions, views, tracks),\n                  all_events=base.event_report(ctx["evaluate"], predictions, views, tracks, False),\n                  frame_metrics=base.frame_metrics(tracks, views))\n    base.write_json(out/f"cnn_tcn_v5_{name}_report.json", report)\n    return report\n\n\ndef result_matrix(reports):\n    rows = []\n    for name in ("Train", "Holdout", "Public"):\n        if name not in reports:\n            continue\n        r, fm = reports[name]["ignore_flagged"], reports[name]["frame_metrics"]\n        rows.append({"Split": name, "Event F1 mean": r["ranking_score"]["f1_mean"],\n                     "Event F1 @0.3": r["primary"]["f1"], "Event Precision": r["primary"]["precision"],\n                     "Event Recall": r["primary"]["recall"], "Frame F1": fm["f1"],\n                     "Frame Accuracy": fm["accuracy"], "Frame Balanced Acc": fm["balanced_accuracy"]})\n    return pd.DataFrame(rows).set_index("Split")\n\n\ndef save_result_v5(ctx, reports, decoder, fixed_report):\n    out = ctx["out"]\n    summary = dict(version="v5", config=ctx["config"], best_epoch=ctx["best_epoch"],\n                   decoder=decoder, fixed_holdout_report=fixed_report, reports=reports, data_audit=ctx["audit"],\n                   training_complete=(out/"training_complete.json").is_file(),\n                   limitations=["Source-group holdout, not full LOSO unless all folds are supplied.",\n                                "Holdout selects checkpoints, thresholds and configurations; its scores are optimistic.",\n                                "Filter reset per isolated view differs from continuous public inference.",\n                                "Equal sample draws per epoch do not mean equal frames or identical class balance.",\n                                "Early stopping can change total optimizer updates between runs.",\n                                "Frame accuracy is dominated by background; event F1 is the ranking metric.",\n                                "Offline future context; no measured online alarm latency."])\n    base.write_json(out/"report_v5.json", summary)\n    result_matrix(reports).to_csv(out/"metrics_matrix_v5.csv")\n    return dict(out=out, history=ctx["history"], reports=reports, matrix=result_matrix(reports),\n                best_epoch=ctx["best_epoch"], config=ctx["config"],\n                fixed_holdout_report=fixed_report, training_complete=summary["training_complete"])\n\n\ndef load_result_v5(out):\n    summary = json.loads((out/"report_v5.json").read_text(encoding="utf-8"))\n    return dict(out=out, history=pd.read_csv(out/"training_history_v5.csv").to_dict("records"),\n                reports=summary["reports"], matrix=result_matrix(summary["reports"]),\n                best_epoch=summary["best_epoch"], config=summary["config"],\n                fixed_holdout_report=summary["fixed_holdout_report"],\n                training_complete=summary["training_complete"])\n\n\ndef evaluate_internal_v5(ctx):\n    cfg, model, out = ctx["cfg"], ctx["model"], ctx["out"]\n    tracks = base.infer_side(model, ctx["raw"], ctx["views"], "holdout", ctx["mean"], ctx["scale"], cfg, ctx["device"])\n    fixed_report = base.event_report(ctx["evaluate"], base.predict_tracks(tracks, ctx["views"], ctx["fixed_decoder"]), ctx["views"], tracks)\n    grid, best, best_key = [], None, None\n    for high in (.4, .5, .6, .7, .8):\n        for low in (.1, .2, .3):\n            decoder = dict(ctx["fixed_decoder"], high=high, low=low)\n            r = base.event_report(ctx["evaluate"], base.predict_tracks(tracks, ctx["views"], decoder), ctx["views"], tracks)\n            grid.append(dict(high=high, low=low, **r["ranking_score"]))\n            if best_key is None or base.rank_key(r) > best_key:\n                best, best_key = decoder, base.rank_key(r)\n    pd.DataFrame(grid).sort_values(["f1_mean", "start_error_median_abs", "false_alarms_per_hour"],\n                                  ascending=[False, True, True]).to_csv(out/"cnn_tcn_v5_decoder_grid.csv", index=False)\n    reports = {"Holdout": export_split_v5(ctx, "loso", tracks, ctx["views"], best)}\n    train_tracks = base.infer_side(model, ctx["raw"], ctx["views"], "fit", ctx["mean"], ctx["scale"], cfg, ctx["device"], True)\n    reports["Train"] = export_split_v5(ctx, "train", train_tracks, ctx["views"], best)\n    torch.save(dict(model=model.state_dict(), config=ctx["config"], epoch=ctx["best_epoch"],\n                    mean=ctx["mean"].tolist(), scale=ctx["scale"].tolist(), decoder=best), out/"cnn_tcn_v5_best_event_f1.pt")\n    pd.DataFrame(columns=["score", "stream_id", "start"]).to_csv(out/"hard_negative_manifest.csv", index=False)\n    return save_result_v5(ctx, reports, best, fixed_report)\n\n\ndef export_public_v5(root, out, cache_root):\n    result = load_result_v5(out)\n    if "Public" in result["reports"]:\n        return result\n    ctx = restore_v5(root, out, cache_root)\n    decoder = result["reports"]["Holdout"]["decoder"]\n    views, raw, manifest, _ = base.build_views(root, cache_root, ctx["holdout"], ctx["cfg"], public=True)\n    tracks = base.infer_side(ctx["model"], raw, views, "public", ctx["mean"], ctx["scale"], ctx["cfg"], ctx["device"], True)\n    result["reports"]["Public"] = export_split_v5(ctx, "public", tracks, views, decoder)\n    pd.DataFrame(manifest).to_csv(out/"public_manifest.csv", index=False)\n    for key, (times, probs) in tracks.items():\n        np.savez_compressed(out/f"probabilities_{key}.npz", times=times, action=probs[0], start=probs[1], end=probs[2])\n    import matplotlib.pyplot as plt\n    key = sorted(tracks)[0]\n    times, probs = tracks[key]\n    fig, ax = plt.subplots(figsize=(14, 4))\n    ax.plot(times, probs[0], lw=.7, label="Action probability")\n    for e in views[key]["events"]:\n        ax.axvspan(e["t_start"], e["t_end"], color="green", alpha=.12)\n    for e in base.decode(times, probs, views[key]["duration"], **decoder):\n        ax.axvspan(e["t_start"], e["t_end"], color="red", alpha=.18)\n    ax.set(xlabel="Seconds", ylabel="Probability", title=f"{key}: green annotation, red prediction")\n    ax.legend(); fig.tight_layout()\n    fig.savefig(out/"public_timeline_v5.png", dpi=150); plt.close(fig)\n    return save_result_v5(ctx, result["reports"], decoder, result["fixed_holdout_report"])\n\n\ndef release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef comparison_rows(records):\n    rows = []\n    for record in records:\n        r = load_result_v5(Path(record["out"]))\n        fixed = r["fixed_holdout_report"]\n        history = pd.DataFrame(r["history"])\n        selected = history.loc[history.epoch == r["best_epoch"]].iloc[0]\n        row = dict(record, window_s=r["config"]["window_seconds"], model=r["config"]["model_size"],\n                   parameters=r["config"]["model_parameters"], best_epoch=r["best_epoch"],\n                   trained_epochs=len(history), cumulative_updates=int(history.cumulative_updates.iloc[-1]),\n                   fixed_holdout_f1=fixed["ranking_score"]["f1_mean"],\n                   fixed_start_error=fixed["ranking_score"]["start_error_median_abs"],\n                   fixed_fa_hour=fixed["ranking_score"]["false_alarms_per_hour"],\n                   selected_loss_gap=float(selected.val_loss-selected.train_loss),\n                   final_loss_gap=float(history.val_loss.iloc[-1]-history.train_loss.iloc[-1]),\n                   training_complete=r["training_complete"])\n        for name, report in r["reports"].items():\n            label = name.lower()\n            row.update({f"{label}_event_f1": report["ignore_flagged"]["ranking_score"]["f1_mean"],\n                        f"{label}_frame_f1": report["frame_metrics"]["f1"],\n                        f"{label}_accuracy": report["frame_metrics"]["accuracy"],\n                        f"{label}_balanced_accuracy": report["frame_metrics"]["balanced_accuracy"]})\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef run_suite_v5(root, run_root, cache_root, common, experiments, folds, seeds,\n                 mode="train", public_policy="all"):\n    if mode not in ("train", "continue", "export_only") or public_policy not in ("all", "winner", "none"):\n        raise ValueError("Check mode/public_policy")\n    if not experiments or not folds or not seeds:\n        raise ValueError("At least one experiment, fold and seed are required")\n    names = [name for name, _ in experiments]\n    fold_names = [name for name, _ in folds]\n    if len(names) != len(set(names)) or len(fold_names) != len(set(fold_names)) or len(seeds) != len(set(seeds)):\n        raise ValueError("Experiment names, folds and seeds must be unique")\n    for name in names+fold_names:\n        if not name or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in name):\n            raise ValueError("Use letters, digits, underscore or hyphen in run names")\n    plan = dict(common=asdict(common), experiments=[dict(name=n, changes=v) for n,v in experiments],\n                folds=[dict(name=n, groups=sorted(g)) for n,g in folds], seeds=list(seeds))\n    run_root.mkdir(parents=True, exist_ok=True)\n    plan_path = run_root/"suite_plan.json"\n    if plan_path.exists():\n        if mode == "train":\n            raise FileExistsError("Existing suite. Use continue/export_only, or choose a new RUN_TAG.")\n        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:\n            raise ValueError("Existing suite has a different plan. Restore its config or choose a new RUN_TAG.")\n    else:\n        if mode == "export_only":\n            raise FileNotFoundError("No existing suite to export")\n        base.write_json(plan_path, plan)\n    records = []\n    for experiment, changes in experiments:\n        for fold, groups in folds:\n            for seed in seeds:\n                cfg = replace(common, **dict(changes, seed=seed))\n                cfg.validate()\n                out = run_root/experiment/fold/f"seed_{seed}"\n                print(f"\\n{experiment} | {fold} | seed {seed}", flush=True)\n                if (out/"report_v5.json").exists():\n                    pass\n                elif (out/"best_checkpoint_fixed_decoder.pt").exists():\n                    if mode != "export_only" and not (out/"training_complete.json").exists():\n                        raise RuntimeError(f"Interrupted training: {out}. Use export_only to export its best available epoch, or a new RUN_TAG to retrain.")\n                    ctx = restore_v5(root, out, cache_root)\n                    evaluate_internal_v5(ctx)\n                    del ctx\n                else:\n                    if mode == "export_only":\n                        raise FileNotFoundError(f"Missing checkpoint: {out}")\n                    ctx = prepare_v5(root, out, cache_root, groups, cfg)\n                    ctx = train_v5(ctx)\n                    evaluate_internal_v5(ctx)\n                    del ctx\n                records.append(dict(experiment=experiment, fold=fold, seed=seed, out=str(out)))\n                release_memory()\n    table = comparison_rows(records)\n    table.to_csv(run_root/"holdout_comparison_v5.csv", index=False)\n    groups = table.groupby("experiment", sort=False)\n    aggregate = groups.agg(fixed_holdout_f1_mean=("fixed_holdout_f1", "mean"),\n                           fixed_holdout_f1_std=("fixed_holdout_f1", "std"),\n                           mean_start_error=("fixed_start_error", "mean"), mean_fa_hour=("fixed_fa_hour", "mean"),\n                           runs=("seed", "size")).reset_index()\n    aggregate = aggregate.sort_values(["fixed_holdout_f1_mean", "mean_start_error", "mean_fa_hour"],\n                                      ascending=[False, True, True], na_position="last")\n    winner = str(aggregate.iloc[0].experiment)\n    lock = dict(selected_experiment=winner, selection="mean fixed-decoder holdout event F1, before public inference",\n                ranking=aggregate.astype(object).where(pd.notna(aggregate), None).to_dict("records"),\n                all_training_complete=bool(table.training_complete.all()))\n    lock_path = run_root/"selection_locked_before_public.json"\n    if lock_path.exists():\n        previous = json.loads(lock_path.read_text(encoding="utf-8"))\n        if previous != lock:\n            raise ValueError("Existing selection lock differs. Use a new RUN_TAG for a new suite.")\n    else:\n        base.write_json(lock_path, lock)\n    aggregate.to_csv(run_root/"configuration_ranking_holdout_v5.csv", index=False)\n    print("Locked selection before public:", winner, flush=True)\n    for record in records:\n        if public_policy == "all" or (public_policy == "winner" and record["experiment"] == winner):\n            export_public_v5(root, Path(record["out"]), cache_root)\n            release_memory()\n    table = comparison_rows(records)\n    table.to_csv(run_root/"experiment_comparison_v5.csv", index=False)\n    base.write_json(run_root/"suite_runs.json", dict(records=records, winner=winner, public_policy=public_policy))\n    return records, table, winner\n\n\ndef archive_run_v5(out):\n    archive = out/"cnn_tcn_v5_outputs.zip"\n    with base.zipfile.ZipFile(archive, "w", base.zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(out.iterdir()):\n            if path.is_file() and path != archive and not path.name.startswith("checkpoint_epoch_"):\n                z.write(path, path.name)\n    return archive\n\n\ndef plot_suite_v5(run_root, records):\n    from IPython.display import display\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import MaxNLocator\n    table = comparison_rows(records)\n    for record in records:\n        result = load_result_v5(Path(record["out"]))\n        out, history = result["out"], pd.DataFrame(result["history"])\n        title = f"{record[\'experiment\']} / {record[\'fold\']} / seed {record[\'seed\']}"\n        print("\\n", title, "| best epoch", result["best_epoch"])\n        if not result["training_complete"]:\n            print("RECOVERED INCOMPLETE TRAINING: not comparable to a completed run.")\n        fig, axes = plt.subplots(1, 2, figsize=(13, 4))\n        axes[0].plot(history.epoch, history.train_loss, "o-", ms=3, label="Train loss (clean)")\n        axes[0].plot(history.epoch, history.val_loss, "o-", ms=3, label="Validation loss (clean)")\n        axes[0].axvline(result["best_epoch"], color="gray", ls="--", label="Selected epoch")\n        axes[0].set(xlabel="Epoch", ylabel="Weighted loss")\n        axes[1].plot(history.epoch, history.val_event_f1_mean, "o-", ms=3, label="Holdout event F1 (fixed decoder)")\n        axes[1].plot(history.epoch, history.val_frame_f1, "o-", ms=3, label="Holdout frame F1")\n        axes[1].set(xlabel="Epoch", ylabel="F1", ylim=(0, 1.02))\n        for ax in axes:\n            ax.legend(fontsize=8); ax.grid(alpha=.2); ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n        fig.suptitle(title); fig.tight_layout()\n        fig.savefig(out/"training_curves_v5.png", dpi=160); plt.show(); plt.close(fig)\n        matrix = result["matrix"]\n        display(matrix.style.format("{:.2%}", na_rep="N/A"))\n        values = matrix.to_numpy(dtype=float)*100\n        fig, ax = plt.subplots(figsize=(13, 3.8))\n        heat = ax.imshow(np.ma.masked_invalid(values), cmap="YlGnBu", vmin=0, vmax=100, aspect="auto")\n        ax.set_xticks(np.arange(len(matrix.columns)), labels=matrix.columns, rotation=20, ha="right")\n        ax.set_yticks(np.arange(len(matrix.index)), labels=matrix.index)\n        for r in range(values.shape[0]):\n            for c in range(values.shape[1]):\n                ax.text(c,r,f"{values[r,c]:.2f}%" if np.isfinite(values[r,c]) else "N/A", ha="center", va="center",\n                        fontsize=9, color="white" if values[r,c]>=65 else "black")\n        ax.set_title("Event F1 and frame metrics (%)")\n        fig.colorbar(heat, ax=ax); fig.tight_layout()\n        fig.savefig(out/"f1_accuracy_matrix_v5.png", dpi=160); plt.show(); plt.close(fig)\n        fig, axes = plt.subplots(1, len(matrix), figsize=(4*len(matrix), 3.8), squeeze=False)\n        for ax, name in zip(axes[0], matrix.index):\n            m = result["reports"][name]["frame_metrics"]\n            counts = np.array([[m["tn"],m["fp"]],[m["fn"],m["tp"]]])\n            ax.imshow(counts, cmap="Blues")\n            ax.set(xticks=[0,1], yticks=[0,1], xticklabels=["Non-fall","Fall"], yticklabels=["Non-fall","Fall"],\n                   xlabel="Predicted", ylabel="Actual", title=name)\n            for r in range(2):\n                for c in range(2):\n                    ax.text(c,r,f"{counts[r,c]:,}",ha="center",va="center",color="white" if counts[r,c]>counts.max()/2 else "black")\n        fig.suptitle("Unique frame confusion matrices, threshold 0.5"); fig.tight_layout()\n        fig.savefig(out/"frame_confusion_matrices_v5.png",dpi=160); plt.show(); plt.close(fig)\n        archive_run_v5(out)\n    display(table.drop(columns=["out"]).round(4))\n    summary = table.groupby("experiment",sort=False).agg(\n        fixed_holdout_f1=("fixed_holdout_f1","mean"), tuned_holdout_f1=("holdout_event_f1","mean"),\n        holdout_frame_f1=("holdout_frame_f1","mean"), holdout_accuracy=("holdout_accuracy","mean"))\n    if "public_event_f1" in table:\n        summary["public_event_f1"] = table.groupby("experiment",sort=False).public_event_f1.mean()\n    summary.to_csv(run_root/"comparison_metrics_matrix_v5.csv")\n    display(summary.style.format("{:.2%}",na_rep="N/A"))\n    fig, ax = plt.subplots(figsize=(11, 5))\n    values=summary.to_numpy(dtype=float)*100\n    heat=ax.imshow(np.ma.masked_invalid(values),cmap="YlGnBu",vmin=0,vmax=100,aspect="auto")\n    ax.set_xticks(np.arange(len(summary.columns)),labels=summary.columns,rotation=20,ha="right")\n    ax.set_yticks(np.arange(len(summary.index)),labels=summary.index)\n    for r in range(values.shape[0]):\n        for c in range(values.shape[1]):\n            ax.text(c,r,f"{values[r,c]:.2f}%" if np.isfinite(values[r,c]) else "N/A",ha="center",va="center",\n                    color="white" if values[r,c]>=65 else "black")\n    ax.set_title("V5 configurations (mean across selected folds/seeds)")\n    fig.colorbar(heat,ax=ax); fig.tight_layout()\n    fig.savefig(run_root/"experiment_comparison_v5.png",dpi=160); plt.show(); plt.close(fig)\n    table.to_csv(run_root/"experiment_comparison_v5.csv",index=False)\n    archive=run_root/"cnn_tcn_v5_suite_outputs.zip"\n    with base.zipfile.ZipFile(archive,"w",base.zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(run_root.iterdir()):\n            if path.is_file() and path.suffix.lower() in (".csv",".json",".png"):\n                z.write(path,path.name)\n        for record in records:\n            path=Path(record["out"])/"cnn_tcn_v5_outputs.zip"\n            z.write(path,path.relative_to(run_root))\n    print("Outputs:",archive)\n    print("Event F1 ranks the competition. Frame accuracy can be high from background imbalance.")\n    print("Partial training and one-fold results do not establish an improvement over V4.")\n    return archive\n'
v5=types.ModuleType('cnn_tcn_v5')
sys.modules['cnn_tcn_v5']=v5
exec(compile(V5_SOURCE,'cnn_tcn_v5.py','exec'),v5.__dict__)
V6_SOURCE='"""V6 decoder study and optional nested source-group confirmation; inert on import."""\nfrom __future__ import annotations\n\nimport hashlib\nimport shutil\nfrom bisect import bisect_right\nfrom pathlib import Path\n\nimport cnn_tcn_v4 as base\nimport cnn_tcn_v5 as v5\nfrom scipy.signal import find_peaks\n\nnp, pd, torch, json = base.np, base.pd, base.torch, base.json\nDECODERS = ("legacy", "action_only", "joint_boundary")\nDECODER_VERSION = "v6-paired-boundaries-1"\n\n\ndef interval_iou(a, b):\n    intersection = max(0., min(a["t_end"], b["t_end"])-max(a["t_start"], b["t_start"]))\n    union = a["t_end"]-a["t_start"]+b["t_end"]-b["t_start"]-intersection\n    return intersection/union if union else 0.\n\n\ndef merge_action_events(events, gap, maximum):\n    merged = []\n    for event in sorted(events, key=lambda e: e["t_start"]):\n        if (merged and event["t_start"]-merged[-1]["t_end"] <= gap and\n                max(event["t_end"], merged[-1]["t_end"])-merged[-1]["t_start"] <= maximum):\n            merged[-1]["t_end"] = max(event["t_end"], merged[-1]["t_end"])\n            merged[-1]["score"] = max(event["score"], merged[-1]["score"])\n        else:\n            merged.append(dict(event))\n    return merged\n\n\ndef boundary_peaks(values, left, right, floor=.25, prominence=.08, distance=3):\n    """Include endpoint maxima so events at a stream boundary can be represented."""\n    if right <= left:\n        return []\n    segment = values[left:right]\n    positions, _ = find_peaks(segment, height=floor, prominence=prominence, distance=distance)\n    result = set((positions+left).tolist())\n    if segment[0] >= floor and (len(segment) == 1 or segment[0] > segment[1]):\n        result.add(left)\n    if len(segment) > 1 and segment[-1] >= floor and segment[-1] > segment[-2]:\n        result.add(right-1)\n    return sorted(result)\n\n\ndef weighted_nonoverlap(candidates, gap_s=.1):\n    """Maximum total utility among ordered, nonoverlapping start/end pairs."""\n    candidates = sorted(candidates, key=lambda e: (e["t_end"], e["t_start"]))\n    ends = [c["t_end"] for c in candidates]\n    previous = [bisect_right(ends, c["t_start"]-gap_s, 0, i)-1 for i,c in enumerate(candidates)]\n    best = [0.]*(len(candidates)+1)\n    for i,c in enumerate(candidates):\n        best[i+1] = max(best[i], c["utility"]+best[previous[i]+1])\n    chosen, i = [], len(candidates)\n    while i:\n        c = candidates[i-1]\n        if c["utility"]+best[previous[i-1]+1] > best[i-1]+1e-12:\n            chosen.append(c)\n            i = previous[i-1]+1\n        else:\n            i -= 1\n    return list(reversed(chosen))\n\n\ndef decode_v6(times, probs, duration, family, high=.6, low=.2,\n              min_s=.5, max_s=8., merge_gap=.5):\n    """Parameters other than high/low are predeclared, not fit on public labels."""\n    if family not in DECODERS or not 0 <= low < high <= 1:\n        raise ValueError("Invalid decoder/thresholds")\n    if not len(times):\n        return [], []\n    if family == "legacy":\n        return base.decode(times, probs, duration, high, low, min_s, max_s, merge_gap), []\n    action, sh, eh = base.smooth(probs[0],5), base.smooth(probs[1],3), base.smooth(probs[2],3)\n    proposals, audit = [], []\n    for a,b in base.boolean_runs(action >= low):\n        if float(action[a:b].max()) < high:\n            continue\n        fallback = dict(t_start=max(0.,float(times[a]-.05)),\n                        t_end=min(float(duration),float(times[b-1]+.05)),\n                        score=float(np.percentile(action[a:b],90)))\n        fallback_valid = min_s-1e-6 <= fallback["t_end"]-fallback["t_start"] <= max_s+1e-6\n        if family == "action_only":\n            if fallback_valid:\n                proposals.append(fallback)\n            continue\n        # Search the complete action proposal plus 0.3 s on each side, rather than onset +/-1 s.\n        left,right = max(0,a-3),min(len(times),b+3)\n        starts,ends = boundary_peaks(sh,left,right),boundary_peaks(eh,left,right)\n        candidates = []\n        for s in starts:\n            for e in ends:\n                start,end = max(0.,float(times[s]-.05)),min(float(duration),float(times[e]+.05))\n                if e <= s or not min_s-1e-6 <= end-start <= max_s+1e-6:\n                    continue\n                inside = action[s:e+1]\n                if float(inside.max()) < high or float(inside.mean()) < .3:\n                    continue\n                # Do not bridge a later, stronger onset, or a distinct earlier end of similar strength.\n                if any(s < q < e-2 and sh[q] >= .8*sh[s] for q in starts):\n                    continue\n                if any(s+2 < q < e and eh[q] >= .8*eh[e] for q in ends):\n                    continue\n                utility = float(np.sqrt(sh[s]*eh[e])*inside.mean()-.15)\n                if utility > 0:\n                    candidates.append(dict(t_start=start,t_end=end,score=float(np.percentile(inside,90)),utility=utility))\n        chosen = weighted_nonoverlap(candidates)\n        method = "paired_heads" if chosen else "action_fallback" if fallback_valid else "rejected_duration"\n        if chosen:\n            proposals.extend({k:c[k] for k in ("t_start","t_end","score")} for c in chosen)\n        elif fallback_valid:\n            proposals.append(fallback)\n        audit.append(dict(proposal_start=float(times[a]),proposal_end=float(times[b-1]+.05),\n                          start_peak_times=json.dumps([float(times[s]) for s in starts]),\n                          end_peak_times=json.dumps([float(times[e]) for e in ends]),\n                          candidate_pairs=len(candidates),selected_pairs=len(chosen),method=method))\n    if family == "action_only":\n        return merge_action_events(proposals,merge_gap,max_s),audit\n    # Suppress duplicate intervals; do not merge distinct refined events just for temporal proximity.\n    kept = []\n    for event in sorted(proposals,key=lambda e:e["score"],reverse=True):\n        if all(interval_iou(event,other) < .5 for other in kept):\n            kept.append(event)\n    return sorted(kept,key=lambda e:e["t_start"]),audit\n\n\ndef predictions_v6(tracks, views, family, parameters):\n    predictions,audit = [],[]\n    for key,(times,probs) in tracks.items():\n        events,diagnostics = decode_v6(times,probs,views[key]["duration"],family,**parameters)\n        predictions.extend(dict(event,stream_id=key) for event in events)\n        audit.extend(dict(row,stream_id=key) for row in diagnostics)\n    return predictions,audit\n\n\ndef decoded_frame_metrics(tracks, views, predictions):\n    converted = {}\n    for key,(times,_) in tracks.items():\n        occupied = np.zeros(len(times),dtype=np.float32)\n        for event in predictions:\n            if event["stream_id"] == key:\n                occupied[(times >= event["t_start"]) & (times < event["t_end"])] = 1\n        converted[key] = (times,np.stack([occupied,occupied,occupied]))\n    result = base.frame_metrics(converted,views)\n    result["scope"] = "unique 10Hz frames covered by decoded event intervals, same ignore mask as model frame metrics"\n    return result\n\n\ndef errors_v6(evaluate_module, predictions, views, tracks, theta=.3):\n    ground = base.event_ground(views,tracks,True)\n    rows = []\n    for key in tracks:\n        gs = [g for g in ground if g["stream_id"] == key]\n        ps = [p for p in predictions if p["stream_id"] == key]\n        matched = evaluate_module._match_stream(gs,ps,theta)\n        usedg = {b for _,b,_ in matched["pairs"]}\n        usedp = {a for a,_,_ in matched["pairs"]}\n        for a,b,iou in matched["pairs"]:\n            p,g = ps[a],matched["scored"][b]\n            rows.append(dict(stream_id=key,kind="TP",gt_start=g["t_start"],gt_end=g["t_end"],\n                             pred_start=p["t_start"],pred_end=p["t_end"],best_tiou=iou,\n                             start_error=p["t_start"]-g["t_start"],end_error=p["t_end"]-g["t_end"]))\n        for i,g in enumerate(matched["scored"]):\n            if i in usedg:\n                continue\n            best = max([evaluate_module.temporal_iou(g["t_start"],g["t_end"],p["t_start"],p["t_end"]) for p in ps]+[0.])\n            rows.append(dict(stream_id=key,kind="FN_localization" if best>0 else "FN_no_overlap",\n                             gt_start=g["t_start"],gt_end=g["t_end"],best_tiou=best))\n        for i,p in enumerate(ps):\n            if i in usedp or any(g["ignore"] and min(p["t_end"],g["t_end"]+2)>max(p["t_start"],g["t_start"]-2) for g in gs):\n                continue\n            best = max([evaluate_module.temporal_iou(g["t_start"],g["t_end"],p["t_start"],p["t_end"]) for g in matched["scored"]]+[0.])\n            rows.append(dict(stream_id=key,kind="FP_overlap" if best>0 else "FP_no_overlap",\n                             pred_start=p["t_start"],pred_end=p["t_end"],best_tiou=best))\n    return rows\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda:stream.read(1024*1024),b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef locked_json(path, data):\n    """A changed configuration must never silently reuse an existing output run."""\n    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != data:\n        raise ValueError(f"Existing run differs: {path}. Restore its plan or use a new RUN_TAG.")\n    if not path.exists():\n        base.write_json(path,data)\n\n\ndef nested_ranges(parts,length,inner,outer):\n    if not inner or not outer or set(inner)&set(outer):\n        raise ValueError("Inner/outer groups must be nonempty and disjoint")\n    masks = {side:np.zeros(length,bool) for side in ("fit","holdout","outer")}\n    for part in parts:\n        side = "outer" if part["group"] in outer else "holdout" if part["group"] in inner else "fit"\n        a,b = max(0,int(round(part["t_start"]*50))),min(length,int(round(part["t_end"]*50)))\n        masks[side][a:b] = True\n    return {side:base.boolean_runs(mask & ~np.logical_or.reduce([m for key,m in masks.items() if key != side]))\n            for side,mask in masks.items()}\n\n\ndef build_nested_views(root, cache_root, inner, outer, cfg):\n    if not inner or not outer or set(inner)&set(outer):\n        raise ValueError("Inner/outer groups must be nonempty and disjoint")\n    _,parts,eventmap,partmap = base.read_metadata(root)\n    known = set(parts.loc[parts.stream_id.str.startswith("train_"),"group"])\n    if not (set(inner)|set(outer)) <= known:\n        raise ValueError("Unknown training group")\n    views,raw,manifest,skipped = {},{},[],[]\n    cache_root.mkdir(parents=True,exist_ok=True)\n    for path in sorted((root/"streams/train").glob("stream_*.h5")):\n        sid = "train_"+path.stem.split("_")[1]\n        with base.h5py.File(path,"r") as h:\n            length = h["cir"].shape[1]\n            for side,spans in nested_ranges(partmap.get(sid,[]),length,inner,outer).items():\n                for number,(a,b) in enumerate(spans):\n                    if b-a < 50:\n                        skipped.append(dict(source_stream=sid,side=side,start=a/50,end=b/50,reason="shorter_than_1s"))\n                        continue\n                    identity = dict(version="v6-nested-f32-1",file=str(path.resolve()),size=path.stat().st_size,\n                                    mtime=path.stat().st_mtime_ns,start=a,end=b,tau=cfg.ema_tau_s)\n                    digest = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:24]\n                    cached = cache_root/f"{digest}.npz"\n                    if cached.exists():\n                        with np.load(cached) as p:\n                            x,times = p["features"],p["times"]\n                    else:\n                        x,times = base.extract_maps(h,a,b,cfg.ema_tau_s)\n                        np.savez_compressed(cached,features=x,times=times)\n                    if not np.isfinite(x).all():\n                        raise ValueError("Nonfinite features")\n                    present = {p["group"] for p in partmap.get(sid,[]) if base.overlap(a/50,b/50,p["t_start"],p["t_end"])>1e-6}\n                    allowed = known-(set(inner)|set(outer)) if side == "fit" else set(inner) if side == "holdout" else set(outer)\n                    if not present <= allowed:\n                        raise AssertionError("A group crossed a nested partition before preprocessing")\n                    key = f"{sid}__{side}_{number:04d}"\n                    events = base.local_events(eventmap.get(sid,[]),a/50,b/50)\n                    views[key] = dict(side=side,duration=(b-a)/50,events=events,groups=sorted(present),\n                                      source_stream=sid,source_start_s=a/50,source_end_s=b/50)\n                    raw[key] = (x,times)\n                    manifest.append(dict(view_id=key,source_stream=sid,side=side,source_start_s=a/50,\n                                         source_end_s=b/50,duration_s=(b-a)/50,groups=";".join(sorted(present)),\n                                         events=len(events),truncated_events=sum(e["truncated"] for e in events)))\n        print("Nested features:",sid,flush=True)\n    for side in ("fit","holdout","outer"):\n        events = base.event_ground(views,[k for k,v in views.items() if v["side"]==side],True)\n        if not any(not e["ignore"] for e in events):\n            raise ValueError(f"No evaluable events in {side}; choose different groups")\n    return views,raw,manifest,skipped\n\n\ndef evaluator_for(root):\n    import sys\n    sys.path.insert(0,str(root/"src"))\n    import abc2027.evaluate as module\n    return module\n\n\ndef prepare_nested(root,out,cache_root,inner,outer,cfg,restore=False):\n    cfg.validate()\n    base.seed_everything(cfg.seed)\n    out.mkdir(parents=True,exist_ok=True)\n    configuration = dict(v5.asdict(cfg),version="v6",mode="nested_train",inner_groups=sorted(inner),\n                         outer_groups=sorted(outer),holdout_groups=sorted(set(inner)|set(outer)),\n                         model="V5-B standard CNN24-48 TCN64-64-96",window_seconds=8.,\n                         protocol="source-exclusive fit/inner/outer; checkpoint and thresholds use inner only")\n    path = out/"config.json"\n    if not restore:\n        if path.exists():\n            raise FileExistsError("Existing training directory; use continue/export_only or a new RUN_TAG")\n        base.write_json(path,configuration)\n    else:\n        saved = json.loads(path.read_text(encoding="utf-8"))\n        for key in ("inner_groups","outer_groups"):\n            if saved[key] != configuration[key]:\n                raise ValueError("Nested group plan changed")\n        configuration = saved\n    views,raw,manifest,skipped = build_nested_views(root,cache_root,set(inner),set(outer),cfg)\n    mean,scale = base.fit_normalizer(raw,views)\n    positives,negatives,fit_eval,val_eval = base.candidate_windows(raw,views,cfg)\n    module = evaluator_for(root)\n    audit = dict(filter_state_reset_per_view=True,normalizer="fit-only zscore",shared_groups=[],\n                 fit_event_owners=len(positives),fit_hours=sum(v["duration"] for v in views.values() if v["side"]=="fit")/3600,\n                 inner_hours=sum(v["duration"] for v in views.values() if v["side"]=="holdout")/3600,\n                 outer_hours=sum(v["duration"] for v in views.values() if v["side"]=="outer")/3600,\n                 skipped_intervals=len(skipped),truncated_events=sum(m["truncated_events"] for m in manifest))\n    pd.DataFrame(manifest).to_csv(out/"split_manifest.csv",index=False)\n    pd.DataFrame(skipped,columns=["source_stream","side","start","end","reason"]).to_csv(out/"skipped_intervals.csv",index=False)\n    base.write_json(out/"data_audit.json",audit)\n    np.savez(out/"normalizer_stats.npz",mean=mean,scale=scale)\n    base.write_json(out/"environment.json",dict(torch=torch.__version__,numpy=np.__version__,cuda=torch.version.cuda,\n                    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))\n    return dict(root=root,out=out,cache_root=cache_root,holdout=set(inner)|set(outer),cfg=cfg,\n                views=views,raw=raw,mean=mean,scale=scale,positives=positives,negatives=negatives,\n                fit_eval=fit_eval,val_eval=val_eval,device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),\n                evaluate=module.evaluate,validator=module.validate_predictions,evaluator=module,config=configuration,audit=audit)\n\n\ndef find_v5_checkpoint(source):\n    source = Path(source)\n    if source.is_file():\n        return source\n    for name in ("cnn_tcn_v5_best_event_f1.pt","best_checkpoint_fixed_decoder.pt"):\n        if (source/name).is_file():\n            return source/name\n    raise FileNotFoundError(f"Upload the V5-B .pt checkpoint or fix V5_B_DIR: {source}")\n\n\ndef load_context_v6(root,out,cache_root,mode,source=None,inner=None,outer=None,cfg=None,run_mode="continue"):\n    out.mkdir(parents=True,exist_ok=True)\n    if mode == "reuse_v5":\n        checkpoint = find_v5_checkpoint(source)\n        payload = torch.load(checkpoint,map_location="cpu",weights_only=True)\n        cfg = v5.ConfigV5(**{k:v for k,v in payload["config"].items() if k in v5.ConfigV5.__dataclass_fields__})\n        cfg.validate()\n        if (cfg.window_frames != 80 or cfg.model_size != "standard" or cfg.temporal_kernel != 5 or\n                cfg.gain_augment or abs(cfg.lr-3e-4)>1e-12):\n            raise ValueError("This study requires the V5-B checkpoint: standard, 8s, no gain, initial LR3e-4")\n        groups = set(payload["config"]["holdout_groups"])\n        views,raw,manifest,skipped = base.build_views(root,cache_root,groups,cfg)\n        mean,scale = base.fit_normalizer(raw,views)\n        module = evaluator_for(root)\n        configuration = dict(payload["config"],version="v6",mode=mode,source_checkpoint=str(checkpoint),\n                             source_sha256=file_hash(checkpoint),selection_scope="reused V5 holdout: exploratory, not independent CV")\n        ctx = dict(root=root,out=out,cache_root=cache_root,cfg=cfg,views=views,raw=raw,mean=mean,scale=scale,\n                   device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),evaluate=module.evaluate,\n                   validator=module.validate_predictions,evaluator=module,config=configuration,holdout=groups,\n                   audit=dict(filter_state_reset_per_view=True,normalizer="fit-only zscore",mode=mode))\n        pd.DataFrame(manifest).to_csv(out/"split_manifest.csv",index=False)\n        pd.DataFrame(skipped,columns=["source_stream","side","start","end","reason"]).to_csv(out/"skipped_intervals.csv",index=False)\n        history_path = checkpoint.parent/"training_history_v5.csv"\n        if history_path.exists():\n            shutil.copyfile(history_path,out/"training_history_v6.csv")\n        base.write_json(out/"config.json",configuration)\n        base.write_json(out/"environment.json",dict(torch=torch.__version__,numpy=np.__version__,cuda=torch.version.cuda,\n                        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))\n        base.write_json(out/"data_audit.json",ctx["audit"])\n        np.savez(out/"normalizer_stats.npz",mean=mean,scale=scale)\n        source_report = checkpoint.parent/"report_v5.json"\n        completed = (checkpoint.parent/"training_complete.json").exists()\n        if source_report.exists():\n            completed = bool(json.loads(source_report.read_text(encoding="utf-8")).get("training_complete",False))\n        if not completed:\n            print("Source training completion is unverified. Keep report_v5.json with the B checkpoint for provenance.",flush=True)\n    elif mode == "nested_train":\n        checkpoint = out/"best_checkpoint_fixed_decoder.pt"\n        if checkpoint.exists():\n            if not (out/"training_complete.json").exists() and run_mode != "export_only":\n                raise RuntimeError("Interrupted training: use export_only for partial outputs or a new RUN_TAG to retrain")\n            ctx = prepare_nested(root,out,cache_root,inner,outer,cfg,restore=True)\n        else:\n            if run_mode == "export_only":\n                raise FileNotFoundError(f"Missing checkpoint: {checkpoint}")\n            ctx = prepare_nested(root,out,cache_root,inner,outer,cfg)\n            ctx = v5.train_v5(ctx)\n        payload = torch.load(checkpoint,map_location="cpu",weights_only=True)\n        ctx["config"] = payload["config"]\n        ctx["config"]["source_sha256"] = file_hash(checkpoint)\n        history_path = out/"training_history_v5.csv"\n        if history_path.exists():\n            shutil.copyfile(history_path,out/"training_history_v6.csv")\n        completed = (out/"training_complete.json").exists()\n    else:\n        raise ValueError("Choose reuse_v5 or nested_train")\n    if not (np.allclose(ctx["mean"],payload["mean"],rtol=1e-5,atol=1e-12) and\n            np.allclose(ctx["scale"],payload["scale"],rtol=1e-5,atol=1e-12)):\n        raise ValueError("Data/scaler mismatch with checkpoint; do not evaluate changed preprocessing")\n    model = v5.CNNTCNV5(cfg).to(ctx["device"])\n    model.load_state_dict(payload["model"])\n    ctx.update(model=model,best_epoch=payload["epoch"],checkpoint=checkpoint,training_complete=completed)\n    return ctx\n\n\ndef tracks_cached(ctx,split,views=None,raw=None):\n    views = ctx["views"] if views is None else views\n    raw = ctx["raw"] if raw is None else raw\n    side = dict(train="fit",inner="holdout",outer="outer",public="public")[split]\n    keys = sorted(k for k,v in views.items() if v["side"] == side)\n    if not keys:\n        return {}\n    metadata = dict(version="v6-track-cache-1",source_sha256=ctx["config"]["source_sha256"],\n                    split=split,config=v5.asdict(ctx["cfg"]),\n                    views={k:views[k] for k in keys},\n                    scaler=dict(mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist()),\n                    raw_hashes={k:hashlib.sha256(raw[k][0].tobytes()+raw[k][1].tobytes()).hexdigest() for k in keys})\n    # Metadata is normalized to JSON types, e.g. NumPy scalars in event rows.\n    metadata = json.loads(json.dumps(metadata,default=float,sort_keys=True))\n    path,meta_path = ctx["out"]/f"probabilities_{split}.npz",ctx["out"]/f"probabilities_{split}_index.json"\n    tracks = {}\n    if path.exists() and meta_path.exists() and json.loads(meta_path.read_text(encoding="utf-8")) == metadata:\n        with np.load(path) as p:\n            for i,key in enumerate(keys):\n                tracks[key] = (p[f"t{i}"],p[f"p{i}"])\n        return tracks\n    tracks = base.infer_side(ctx["model"],raw,views,side,ctx["mean"],ctx["scale"],ctx["cfg"],ctx["device"],True)\n    arrays = {}\n    for i,key in enumerate(keys):\n        arrays[f"t{i}"],arrays[f"p{i}"] = tracks[key]\n    np.savez_compressed(path,**arrays)\n    base.write_json(meta_path,metadata)\n    return tracks\n\n\ndef report_row(split,family,report):\n    r = report["ignore_flagged"]\n    primary = r["primary"]\n    frame,decoded = report["frame_metrics"],report["decoded_frame_metrics"]\n    return dict(split=split,decoder=family,event_f1_mean=r["ranking_score"]["f1_mean"],\n                event_f1_03=primary["f1"],event_f1_05=r["by_theta"]["0.5"]["f1"],\n                precision=primary["precision"],recall=primary["recall"],tp=primary["tp"],fp=primary["fp"],fn=primary["fn"],\n                false_alarms_per_hour=primary["false_alarms_per_hour"],\n                start_error_median=primary["start_error_median_abs"],end_error_median=primary["end_error_median_abs"],\n                model_frame_f1=frame["f1"],model_frame_accuracy=frame["accuracy"],\n                model_balanced_accuracy=frame["balanced_accuracy"],decoded_frame_f1=decoded["f1"],\n                decoded_frame_accuracy=decoded["accuracy"],decoded_balanced_accuracy=decoded["balanced_accuracy"],\n                exposure_hours=report["exposure_hours"])\n\n\ndef export_report_v6(ctx,split,tracks,views,family,parameters):\n    predictions,audit = predictions_v6(tracks,views,family,parameters)\n    out = ctx["out"]\n    prefix = f"cnn_tcn_v6_{family}_{split}"\n    path = out/f"{prefix}_submission.csv"\n    pd.DataFrame(predictions,columns=["stream_id","t_start","t_end","score"]).to_csv(path,index=False)\n    checked = ctx["validator"](path,{k:views[k]["duration"] for k in tracks})\n    report = dict(version="v6",decoder_version=DECODER_VERSION,split=split,decoder=family,parameters=parameters,\n                  best_epoch=ctx["best_epoch"],training_complete=ctx["training_complete"],\n                  mode=ctx["config"]["mode"],exposure_hours=sum(views[k]["duration"] for k in tracks)/3600,\n                  ignore_flagged=base.event_report(ctx["evaluate"],checked,views,tracks),\n                  all_events=base.event_report(ctx["evaluate"],checked,views,tracks,False),\n                  frame_metrics=base.frame_metrics(tracks,views),decoded_frame_metrics=decoded_frame_metrics(tracks,views,checked))\n    base.write_json(out/f"{prefix}_report.json",report)\n    pd.DataFrame(audit,columns=["stream_id","proposal_start","proposal_end","start_peak_times","end_peak_times",\n                               "candidate_pairs","selected_pairs","method"]).to_csv(out/f"{prefix}_boundary_audit.csv",index=False)\n    pd.DataFrame(errors_v6(ctx["evaluator"],checked,views,tracks),columns=["stream_id","kind","gt_start","gt_end",\n                 "pred_start","pred_end","best_tiou","start_error","end_error"]).to_csv(out/f"{prefix}_event_errors.csv",index=False)\n    return report\n\n\ndef export_fixed_comparison(ctx,split,tracks,views):\n    rows = []\n    parameters = dict(high=.6,low=.2,min_s=.5,max_s=8.,merge_gap=.5)\n    for family in DECODERS:\n        predictions,_ = predictions_v6(tracks,views,family,parameters)\n        report = base.event_report(ctx["evaluate"],predictions,views,tracks)\n        rows.append(dict(split=split,decoder=family,**report["ranking_score"],\n                         f1_03=report["primary"]["f1"],f1_05=report["by_theta"]["0.5"]["f1"],\n                         tp=report["primary"]["tp"],fp=report["primary"]["fp"],fn=report["primary"]["fn"]))\n    pd.DataFrame(rows).to_csv(ctx["out"]/f"fixed_threshold_decoder_comparison_{split}.csv",index=False)\n\n\ndef select_decoders_v6(ctx):\n    tracks = tracks_cached(ctx,"inner")\n    fixed = dict(high=.6,low=.2,min_s=.5,max_s=8.,merge_gap=.5)\n    grid,parameters,fixed_rows = [],{},[]\n    for family in DECODERS:\n        fixed_predictions,_ = predictions_v6(tracks,ctx["views"],family,fixed)\n        fixed_report = base.event_report(ctx["evaluate"],fixed_predictions,ctx["views"],tracks)\n        fixed_rows.append(dict(decoder=family,**fixed_report["ranking_score"]))\n        best_key,best = None,None\n        for high in (.4,.5,.6,.7,.8):\n            for low in (.1,.2,.3):\n                candidate = dict(fixed,high=high,low=low)\n                predictions,_ = predictions_v6(tracks,ctx["views"],family,candidate)\n                report = base.event_report(ctx["evaluate"],predictions,ctx["views"],tracks)\n                grid.append(dict(decoder=family,high=high,low=low,**report["ranking_score"]))\n                if best_key is None or base.rank_key(report)>best_key:\n                    best_key,best = base.rank_key(report),candidate\n        parameters[family] = best\n    pd.DataFrame(grid).to_csv(ctx["out"]/"cnn_tcn_v6_decoder_grid.csv",index=False)\n    pd.DataFrame(fixed_rows).to_csv(ctx["out"]/"fixed_threshold_decoder_comparison_inner.csv",index=False)\n    selection = dict(version=DECODER_VERSION,source_sha256=ctx["config"]["source_sha256"],parameters=parameters,\n                     mode=ctx["config"]["mode"],scope="inner only; reused holdout is exploratory",\n                     fixed_comparison=fixed_rows,training_complete=ctx["training_complete"])\n    locked_json(ctx["out"]/"decoder_selection_locked.json",selection)\n    return selection\n\n\ndef run_specs_v6(mode,folds,seeds):\n    if mode == "reuse_v5":\n        return [dict(name="reuse_v5_B",inner=[],outer=[],seed=None)]\n    if mode != "nested_train":\n        raise ValueError("Unknown mode")\n    specs = []\n    for name,inner,outer in folds:\n        if not name or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in name):\n            raise ValueError("Invalid fold name")\n        for seed in seeds:\n            specs.append(dict(name=f"{name}_seed_{seed}",inner=sorted(inner),outer=sorted(outer),seed=seed))\n    if not specs or len({s["name"] for s in specs}) != len(specs):\n        raise ValueError("Supply unique folds and seeds")\n    return specs\n\n\ndef aggregate_v6(table):\n    """Micro F1 pools counts per tIoU; FA/h pools exposure, not per-run rates."""\n    rows = []\n    for (split,family),group in table.groupby(["split","decoder"],sort=False):\n        summaries = [json.loads(Path(path).read_text(encoding="utf-8")) for path in group.report_path]\n        exposure = sum(r["exposure_hours"] for r in summaries)\n        micro = {}\n        for theta in ("0.1","0.3","0.5","0.7"):\n            counts = {key:sum(r["ignore_flagged"]["by_theta"][theta][key] for r in summaries) for key in ("tp","fp","fn")}\n            den = 2*counts["tp"]+counts["fp"]+counts["fn"]\n            micro[theta] = 2*counts["tp"]/den if den else 0.\n            if theta == "0.3":\n                main = counts\n        rows.append(dict(split=split,decoder=family,runs=len(group),micro_event_f1_mean=float(np.mean([micro[t] for t in ("0.1","0.3","0.5")])),\n                         micro_f1_03=micro["0.3"],micro_f1_05=micro["0.5"],macro_f1_mean=float(group.event_f1_mean.mean()),\n                         run_f1_std=float(group.event_f1_mean.std()) if len(group)>1 else None,\n                         tp=main["tp"],fp=main["fp"],fn=main["fn"],exposure_hours=exposure,\n                         false_alarms_per_hour=main["fp"]/exposure if exposure else None,\n                         note="Repeated seeds repeat the same subjects/exposure; these are not independent samples"))\n    return pd.DataFrame(rows)\n\n\ndef collect_results_v6(run_root,specs):\n    rows = []\n    for spec in specs:\n        out = run_root/spec["name"]\n        for path in sorted(out.glob("cnn_tcn_v6_*_report.json")):\n            report = json.loads(path.read_text(encoding="utf-8"))\n            if (report.get("decoder") not in DECODERS or\n                    path.name != f"cnn_tcn_v6_{report[\'decoder\']}_{report[\'split\']}_report.json"):\n                continue\n            row = report_row(report["split"],report["decoder"],report)\n            row.update(run=spec["name"],seed=spec["seed"],report_path=str(path),training_complete=report["training_complete"])\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef run_v6(root,run_root,cache_root,source,mode,run_mode,cfg,folds,seeds,public=True):\n    if run_mode not in ("train","continue","export_only"):\n        raise ValueError("Invalid RUN_MODE")\n    specs = run_specs_v6(mode,folds,seeds)\n    source_checkpoint = find_v5_checkpoint(source) if mode == "reuse_v5" else None\n    plan = dict(version=DECODER_VERSION,mode=mode,specs=specs,config=v5.asdict(cfg),\n                source=str(source_checkpoint) if source_checkpoint else None,\n                source_sha256=file_hash(source_checkpoint) if source_checkpoint else None,\n                public=public,decoders=list(DECODERS))\n    run_root.mkdir(parents=True,exist_ok=True)\n    if (run_root/"suite_plan_v6.json").exists() and run_mode == "train":\n        raise FileExistsError("Existing suite; use continue/export_only or a new RUN_TAG")\n    locked_json(run_root/"suite_plan_v6.json",plan)\n\n    def context(spec):\n        conf = cfg if spec["seed"] is None else v5.replace(cfg,seed=spec["seed"])\n        return load_context_v6(root,run_root/spec["name"],cache_root,mode,source_checkpoint,\n                               set(spec["inner"]),set(spec["outer"]),conf,run_mode)\n\n    selections = []\n    # All checkpoint and decoder selection finishes before ANY outer/public evaluation in this suite.\n    for spec in specs:\n        out = run_root/spec["name"]\n        if (out/"decoder_selection_locked.json").exists():\n            selection = json.loads((out/"decoder_selection_locked.json").read_text(encoding="utf-8"))\n        else:\n            ctx = context(spec)\n            selection = select_decoders_v6(ctx)\n            del ctx\n            v5.release_memory()\n        selections.append(selection)\n    averages = {family:float(np.mean([next(r["f1_mean"] for r in s["fixed_comparison"] if r["decoder"]==family) for s in selections])) for family in DECODERS}\n    winner = max(DECODERS,key=lambda family:averages[family])\n    lock = dict(selected_decoder=winner,criterion="mean inner event F1 with fixed thresholds .6/.2, before outer/public",\n                fixed_inner_f1=averages,all_training_complete=all(s["training_complete"] for s in selections),\n                scope="exploratory reused holdout" if mode=="reuse_v5" else "inner-selected rules; outer untouched by this suite selection")\n    locked_json(run_root/"decoder_family_locked_before_outer_public.json",lock)\n    print("Locked decoder family:",winner,flush=True)\n    for spec,selection in zip(specs,selections):\n        out = run_root/spec["name"]\n        marker = out/"run_complete_v6.json"\n        if marker.exists():\n            continue\n        ctx = context(spec)\n        reports = {}\n        for split in ("train","inner","outer"):\n            tracks = tracks_cached(ctx,split)\n            if not tracks:\n                continue\n            for family in DECODERS:\n                reports[f"{family}:{split}"] = export_report_v6(ctx,split,tracks,ctx["views"],family,selection["parameters"][family])\n            export_fixed_comparison(ctx,split,tracks,ctx["views"])\n        if public:\n            pv,pr,manifest,_ = base.build_views(root,cache_root,ctx["holdout"],ctx["cfg"],public=True)\n            pd.DataFrame(manifest).to_csv(out/"public_manifest.csv",index=False)\n            tracks = tracks_cached(ctx,"public",pv,pr)\n            for family in DECODERS:\n                reports[f"{family}:public"] = export_report_v6(ctx,"public",tracks,pv,family,selection["parameters"][family])\n            export_fixed_comparison(ctx,"public",tracks,pv)\n            del pr\n        for split in ("train","inner","outer","public"):\n            if f"{winner}:{split}" in reports:\n                alias = "loso" if split == ("outer" if mode=="nested_train" else "inner") else split\n                for suffix in ("submission.csv","report.json"):\n                    shutil.copyfile(out/f"cnn_tcn_v6_{winner}_{split}_{suffix}",out/f"cnn_tcn_v6_{alias}_{suffix}")\n        torch.save(dict(model=ctx["model"].state_dict(),config=ctx["config"],epoch=ctx["best_epoch"],\n                        mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),decoder_family=winner,\n                        decoder=selection["parameters"][winner]),out/"cnn_tcn_v6_selected_model.pt")\n        pd.DataFrame(columns=["score","stream_id","start"]).to_csv(out/"hard_negative_manifest.csv",index=False)\n        base.write_json(out/"report_v6.json",dict(version="v6",best_epoch=ctx["best_epoch"],mode=mode,\n                        selected_decoder=winner,selection=selection,training_complete=ctx["training_complete"],reports=reports,\n                        limitations=["Reused V5 holdout is exploratory; nested outer groups are excluded from fit and inner selection.",\n                                     "The public split has been inspected in earlier research and is a development benchmark.",\n                                     "Decoder changes do not alter model frame probabilities or model frame accuracy.",\n                                     "No claim of reduced neural-network overfitting from postprocessing alone.",\n                                     "Several folds/seeds are not full LOSO and seeds are not independent subjects.",\n                                     "Filters reset per source-isolated view; public streams remain continuous.",\n                                     "Offline future context; no measured online alarm latency."]))\n        base.write_json(marker,dict(decoder_version=DECODER_VERSION,selected_decoder=winner))\n        del ctx\n        v5.release_memory()\n    table = collect_results_v6(run_root,specs)\n    table.to_csv(run_root/"decoder_comparison_v6.csv",index=False)\n    aggregate_v6(table).to_csv(run_root/"decoder_aggregate_v6.csv",index=False)\n    base.write_json(run_root/"suite_runs_v6.json",dict(specs=specs,winner=winner,mode=mode))\n    return specs,table,winner\n\n\ndef probability_plots_v6(out):\n    import matplotlib.pyplot as plt\n    split = "public" if (out/"probabilities_public.npz").exists() else "outer" if (out/"probabilities_outer.npz").exists() else "inner"\n    index = json.loads((out/f"probabilities_{split}_index.json").read_text(encoding="utf-8"))\n    key = sorted(index["views"])[0]\n    view = index["views"][key]\n    with np.load(out/f"probabilities_{split}.npz") as data:\n        times,probs = data["t0"],data["p0"]\n    fig,ax = plt.subplots(figsize=(14,4))\n    ax.plot(times,probs[0],lw=.8,label="Model action probability")\n    for event in view["events"]:\n        ax.axvspan(event["t_start"],event["t_end"],color="green",alpha=.15)\n    selection=json.loads((out/"decoder_selection_locked.json").read_text(encoding="utf-8"))\n    colors={"legacy":"#dc2626","action_only":"#2563eb","joint_boundary":"#7c3aed"}\n    for i,family in enumerate(DECODERS):\n        events,_=decode_v6(times,probs,view["duration"],family,**selection["parameters"][family])\n        for j,event in enumerate(events):\n            ax.plot([event["t_start"],event["t_end"]],[-.08-.08*i]*2,lw=4,color=colors[family],label=family if j==0 else None)\n    ax.set(xlabel="Seconds",ylabel="Probability",title=f"{key}: green annotations; decoded intervals below")\n    ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/f"{split}_timeline_v6.png",dpi=160);plt.show();plt.close(fig)\n    events=[e for e in view["events"] if not base.truth(e.get("ignore",False))]\n    if not events:\n        return\n    # A deterministic first-event diagnostic, not a public-error-picked training target.\n    event=events[0]\n    mask=(times>=max(0,event["t_start"]-5))&(times<=min(view["duration"],event["t_end"]+5))\n    fig,ax=plt.subplots(figsize=(11,4))\n    for channel,label in enumerate(("Action","Start head","End head")):\n        ax.plot(times[mask],probs[channel,mask],label=label)\n    ax.axvspan(event["t_start"],event["t_end"],color="green",alpha=.12,label="Annotation")\n    ax.set(xlabel="Seconds",ylabel="Probability",title=f"{key}: first annotated event (fixed diagnostic)")\n    ax.ticklabel_format(useOffset=False,axis="x");ax.legend();fig.tight_layout()\n    fig.savefig(out/"boundary_probabilities_example_v6.png",dpi=160);plt.show();plt.close(fig)\n\n\ndef plot_v6(run_root):\n    import matplotlib.pyplot as plt\n    from IPython.display import display\n    index = json.loads((run_root/"suite_runs_v6.json").read_text(encoding="utf-8"))\n    table = collect_results_v6(run_root,index["specs"])\n    display_columns = ["split","decoder","event_f1_mean","event_f1_03","event_f1_05","model_frame_f1",\n                       "model_frame_accuracy","decoded_frame_f1","decoded_frame_accuracy"]\n    for spec in index["specs"]:\n        out = run_root/spec["name"]\n        report = json.loads((out/"report_v6.json").read_text(encoding="utf-8"))\n        print("\\nRun:",spec["name"],"| mode:",index["mode"],"| selected epoch:",report["best_epoch"])\n        history_path = out/"training_history_v6.csv"\n        if history_path.exists():\n            h = pd.read_csv(history_path)\n            fig,axes = plt.subplots(1,2,figsize=(12,4))\n            axes[0].plot(h.epoch,h.train_loss,"o-",ms=3,label="Train loss (clean)")\n            axes[0].plot(h.epoch,h.val_loss,"o-",ms=3,label="Selection-set loss (clean)")\n            axes[0].axvline(report["best_epoch"],ls="--",color="gray",label="Selected epoch")\n            axes[0].set(xlabel="Epoch",ylabel="Weighted loss")\n            axes[1].plot(h.epoch,h.val_event_f1_mean,"o-",ms=3,label="Selection-set event F1, legacy fixed decoder")\n            axes[1].set(xlabel="Epoch",ylabel="F1",ylim=(0,1.02))\n            for ax in axes:\n                ax.grid(alpha=.2);ax.legend(fontsize=8)\n            fig.suptitle("V5 training history reused" if index["mode"]=="reuse_v5" else "V6 training: validation = INNER only")\n            fig.tight_layout();fig.savefig(out/"training_curves_v6.png",dpi=160);plt.show();plt.close(fig)\n        else:\n            print("No history CSV alongside the checkpoint; no loss curve was invented.")\n        rows = table[table.run==spec["name"]]\n        matrix = rows[display_columns].set_index(["split","decoder"])\n        matrix.to_csv(out/"metrics_matrix_v6.csv")\n        display(matrix.style.format("{:.2%}",na_rep="N/A"))\n        for name,columns in [("f1_accuracy_matrix_v6",["event_f1_mean","event_f1_05","model_frame_accuracy","decoded_frame_f1","decoded_frame_accuracy"])]:\n            values=matrix[columns].to_numpy(dtype=float)*100\n            fig,ax=plt.subplots(figsize=(11,max(4,len(matrix)*.4+1.5)))\n            heat=ax.imshow(np.ma.masked_invalid(values),cmap="YlGnBu",vmin=0,vmax=100,aspect="auto")\n            ax.set_xticks(np.arange(len(columns)),labels=columns,rotation=18,ha="right")\n            ax.set_yticks(np.arange(len(matrix)),labels=[f"{a} / {b}" for a,b in matrix.index])\n            for r in range(values.shape[0]):\n                for c in range(values.shape[1]):\n                    ax.text(c,r,f"{values[r,c]:.2f}%",ha="center",va="center",fontsize=9,color="white" if values[r,c]>=65 else "black")\n            ax.set_title("Same checkpoint; event metrics vs model/decoded frame metrics")\n            fig.colorbar(heat,ax=ax);fig.tight_layout();fig.savefig(out/f"{name}.png",dpi=160);plt.show();plt.close(fig)\n        selected=[(key.split(":")[1],r) for key,r in report["reports"].items() if key.startswith(index["winner"]+":")]\n        fig,axes=plt.subplots(2,len(selected),figsize=(4*len(selected),7),squeeze=False)\n        for c,(split,r) in enumerate(selected):\n            for row,metric in enumerate(("frame_metrics","decoded_frame_metrics")):\n                m=r[metric];counts=np.array([[m["tn"],m["fp"]],[m["fn"],m["tp"]]])\n                ax=axes[row,c];ax.imshow(counts,cmap="Blues")\n                ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Non-fall","Fall"],yticklabels=["Non-fall","Fall"],\n                       xlabel="Predicted",ylabel="Actual",title=f"{split}: {\'model\' if row==0 else \'decoded\'}")\n                for a in range(2):\n                    for b in range(2):\n                        ax.text(b,a,f"{counts[a,b]:,}",ha="center",va="center",color="white" if counts[a,b]>counts.max()/2 else "black")\n        fig.suptitle(f"Frame confusion matrices, selected decoder: {index[\'winner\']}")\n        fig.tight_layout();fig.savefig(out/"frame_confusion_matrices_v6.png",dpi=160);plt.show();plt.close(fig)\n        probability_plots_v6(out)\n        archive=out/"cnn_tcn_v6_outputs.zip"\n        with base.zipfile.ZipFile(archive,"w",base.zipfile.ZIP_DEFLATED) as z:\n            for p in sorted(out.iterdir()):\n                if p.is_file() and p!=archive and not p.name.startswith("checkpoint_epoch_"):\n                    z.write(p,p.name)\n    aggregate=aggregate_v6(table)\n    table.to_csv(run_root/"decoder_comparison_v6.csv",index=False)\n    aggregate.to_csv(run_root/"decoder_aggregate_v6.csv",index=False)\n    print("Aggregate across selected folds/seeds (seeds repeat the same subjects):")\n    display(aggregate.drop(columns=["note"]).round(4))\n    archive=run_root/"cnn_tcn_v6_suite_outputs.zip"\n    with base.zipfile.ZipFile(archive,"w",base.zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(run_root.iterdir()):\n            if p.is_file() and p.suffix in (".csv",".json",".png"):\n                z.write(p,p.name)\n        for spec in index["specs"]:\n            p=run_root/spec["name"]/"cnn_tcn_v6_outputs.zip"\n            z.write(p,p.relative_to(run_root))\n    print("Locked decoder family:",index["winner"])\n    print("Model frame accuracy is identical across decoders for the same split/checkpoint.")\n    print("Decoded frame metrics measure exported interval coverage; they are not the official event score.")\n    print("Outputs:",archive)\n    return archive\n'
v6=types.ModuleType('cnn_tcn_v6')
sys.modules['cnn_tcn_v6']=v6
exec(compile(V6_SOURCE,'cnn_tcn_v6.py','exec'),v6.__dict__)
V7_SOURCE='"""V7 controlled sampling study, nested evaluation and all-train final refit.\n\nImporting this module only defines functions. Training is started explicitly.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nimport random\nimport shutil\nimport time\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\n\nimport cnn_tcn_v4 as base\nimport cnn_tcn_v5 as v5\nimport cnn_tcn_v6 as v6\n\nnp, pd, torch = base.np, base.pd, base.torch\nVERSION = "v7-sampling-nested-refit-1"\nDECODERS = ("legacy", "joint_boundary", "conservative")\nEXPERIMENTS = ("T0_baseline", "T1_balanced_unique", "T2_weight_decay")\nFIXED = dict(high=.6, low=.2, min_s=.5, max_s=8., merge_gap=.5)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    base.write_json(temporary, value)\n    temporary.replace(path)\n\n\ndef write_csv(path, rows, columns=None):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    pd.DataFrame(rows, columns=columns).to_csv(temporary, index=False)\n    temporary.replace(path)\n\n\ndef save_checkpoint(path, payload):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef cpu_state(model):\n    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n\n\ndef event_metadata(views):\n    result = {}\n    for view in views.values():\n        if view["side"] != "fit":\n            continue\n        for event in view["events"]:\n            result[event["event_id"]] = dict(event, source_group=base.source_group(event["source_path"], "fall"))\n    return result\n\n\ndef candidate_indices(raw, views, cfg, require_inner=True):\n    """Same V5 candidate definitions, also allowing an all-train refit."""\n    positives, negatives, fit_eval, val_eval = defaultdict(list), [], [], []\n    for key, view in views.items():\n        times = raw[key][1]\n        for start in base.window_starts(len(times), cfg.window_frames, cfg.train_hop_frames):\n            item = (key, start)\n            if view["side"] == "holdout":\n                val_eval.append(item)\n            if view["side"] != "fit":\n                continue\n            fit_eval.append(item)\n            a, b = float(times[start]), float(times[min(len(times), start+cfg.window_frames)-1]+.1)\n            hits = [(base.overlap(a, b, e["t_start"], e["t_end"]), e["event_id"])\n                    for e in view["events"] if not base.truth(e["ignore"])]\n            hits = [hit for hit in hits if hit[0] > 0]\n            if hits:\n                positives[max(hits)[1]].append(item)\n            else:\n                negatives.append(item)\n    if not positives or not negatives or (require_inner and not val_eval):\n        raise ValueError("Need fit fall/background candidates and, for nested training, inner windows")\n    if len(fit_eval) > cfg.train_eval_max_windows:\n        rng = np.random.default_rng(cfg.seed)\n        fit_eval = [fit_eval[i] for i in sorted(rng.choice(len(fit_eval), cfg.train_eval_max_windows, replace=False))]\n    return dict(positives), negatives, fit_eval, val_eval\n\n\ndef balanced_unique_sample(positives, negatives, metadata, raw, cfg, epoch):\n    """Cover every owner, then allocate remaining slots with capped soft balancing.\n\n    The budget is fixed. Owner exposures are capped at three; actual overlaps with\n    every event are audited separately because one window can contain many events.\n    """\n    rng = np.random.default_rng(cfg.seed+300000+epoch)\n    owners = sorted(positives)\n    budget = cfg.batch_size*cfg.steps_per_epoch\n    if len(owners) > budget:\n        raise ValueError("Budget cannot cover all owner events. Increase steps_per_epoch.")\n    cap = cfg.windows_per_event\n    groups = {owner:(str(metadata[owner].get("dataset", "unknown")),\n                    str(metadata[owner].get("fall_type", "unknown"))) for owner in owners}\n    counts = Counter(groups.values())\n    subject_counts = Counter((groups[o], metadata[o]["source_group"]) for o in owners)\n    weights = {o:counts[groups[o]]**-.5 * subject_counts[(groups[o], metadata[o]["source_group"])]**-.25 for o in owners}\n    ceiling = 3*float(np.median(list(weights.values())))\n    weights = {o:min(w, ceiling) for o,w in weights.items()}\n    queues = {o:[positives[o][i] for i in rng.permutation(len(positives[o]))] for o in owners}\n    original_owner = {item:o for o in owners for item in positives[o]}\n    selected, seen, exposure = [], set(), Counter()\n\n    def take(owner):\n        while queues[owner] and exposure[owner] < cap:\n            key, start = queues[owner].pop()\n            original = (key, int(start))\n            if cfg.jitter_frames:\n                shifted = int(np.clip(start+rng.integers(-cfg.jitter_frames, cfg.jitter_frames+1),\n                                      0, max(0, len(raw[key][1])-cfg.window_frames)))\n                times = raw[key][1]\n                event = metadata[owner]\n                if original_owner.get((key,shifted),owner)==owner and base.overlap(\n                        float(times[shifted]), float(times[min(len(times), shifted+cfg.window_frames)-1]+.1),\n                        event["t_start"], event["t_end"]) > 0:\n                    start = shifted\n            item = (key, int(start))\n            if item in seen:\n                item = original\n            if item in seen:\n                continue\n            seen.add(item)\n            exposure[owner] += 1\n            selected.append((item, owner, "event"))\n            return True\n        return False\n\n    # A coverage pass keeps rare and common types present; no Chute04-only filter.\n    for owner in rng.permutation(owners):\n        take(owner)\n    if set(exposure) != set(owners):\n        raise ValueError("Coverage pass did not include every event owner")\n    target = min(budget//(1+cfg.negative_ratio), 2*len(owners))\n    target = max(target, len(selected))\n    while len(selected) < target:\n        available = [o for o in owners if queues[o] and exposure[o] < cap]\n        if not available:\n            break\n        probability = np.array([weights[o] for o in available], float)\n        owner = available[int(rng.choice(len(available), p=probability/probability.sum()))]\n        take(owner)\n    for i in rng.permutation(len(negatives)):\n        if len(selected) >= budget:\n            break\n        item = negatives[i]\n        if item not in seen:\n            seen.add(item)\n            selected.append((item, "", "background"))\n    if len(selected) != budget:\n        raise ValueError(f"Only {len(selected)} unique windows for budget {budget}. "\n                         "Lower steps_per_epoch for BOTH profiles and use a new RUN_TAG.")\n    selected = [selected[i] for i in rng.permutation(len(selected))]\n    index, manifest = [], []\n    for draw, (item, owner, kind) in enumerate(selected):\n        key, start = item\n        index.append(item)\n        manifest.append(dict(epoch=epoch, draw=draw, view_id=key, start_frame=start,\n                             owner_event=owner, kind=kind, optimizer_step=draw//cfg.batch_size+1))\n    return index, manifest\n\n\ndef sample_training(ctx, epoch):\n    cfg = ctx["cfg"]\n    if ctx["strategy"] == "T1_balanced_unique":\n        return balanced_unique_sample(ctx["positives"], ctx["negatives"], event_metadata(ctx["views"]),\n                                      ctx["raw"], cfg, epoch)\n    pool, manifest = base.sample_epoch(ctx["positives"], ctx["negatives"], ctx["raw"], ctx["views"], cfg, epoch)\n    lookup = {(row["view_id"], row["start_frame"]):row for row in manifest}\n    rng = np.random.default_rng(cfg.seed+200000+epoch)\n    actual = [pool[i] for i in rng.integers(0, len(pool), cfg.batch_size*cfg.steps_per_epoch)]\n    manifest = [dict(lookup[item], draw=i, optimizer_step=i//cfg.batch_size+1) for i,item in enumerate(actual)]\n    return actual, manifest\n\n\ndef exposure_audit(ctx, index, manifest, epoch):\n    owners = Counter(row["owner_event"] for row in manifest if row["kind"] == "event")\n    overlaps = Counter()\n    by_view = defaultdict(list)\n    for key, start in index:\n        by_view[key].append(start)\n    records = []\n    for key, starts in by_view.items():\n        times = ctx["raw"][key][1]\n        for event in ctx["views"][key]["events"]:\n            if base.truth(event["ignore"]):\n                continue\n            for start in starts:\n                if base.overlap(float(times[start]), float(times[min(len(times), start+ctx["cfg"].window_frames)-1]+.1),\n                                event["t_start"], event["t_end"]) > 0:\n                    overlaps[event["event_id"]] += 1\n    for owner,event in event_metadata(ctx["views"]).items():\n        records.append(dict(epoch=epoch, event_id=owner, dataset=event.get("dataset", "unknown"),\n                            fall_type=event.get("fall_type", "unknown"), source_group=event["source_group"],\n                            owner_draws=owners[owner], actual_overlapping_windows=overlaps[owner]))\n    summary = dict(train_windows=len(index), unique_drawn_windows=len(set(index)),\n                   positive_owner_draws=sum(owners.values()), covered_owner_events=len(owners),\n                   max_owner_draws=max(owners.values(), default=0), covered_actual_events=len(overlaps),\n                   max_actual_event_exposure=max(overlaps.values(), default=0))\n    return records, summary\n\n\ndef conservative_decode(times, probs, duration, **parameters):\n    legacy = base.decode(times, probs, duration, **parameters)\n    if not len(times):\n        return legacy, []\n    action, sh, eh = base.smooth(probs[0], 5), base.smooth(probs[1], 3), base.smooth(probs[2], 3)\n    spans = base.boolean_runs(action >= parameters["low"])\n    result, audit = [], []\n    for number, old in enumerate(legacy):\n        hits = [(base.overlap(float(times[a]-.05), float(times[b-1]+.05), old["t_start"], old["t_end"]), a, b)\n                for a,b in spans]\n        if not hits or max(hits)[0] <= 0:\n            result.append(old)\n            continue\n        _, a, b = max(hits)\n        left, right = max(0,a-3), min(len(times),b+3)\n        starts, ends = v6.boundary_peaks(sh,left,right), v6.boundary_peaks(eh,left,right)\n\n        def utility(start, end):\n            s = int(np.argmin(np.abs(times-start)))\n            e = int(np.argmin(np.abs(times-(end-.05))))\n            if e <= s:\n                return -1.\n            inside = action[s:e+1]\n            skipped = max([float(sh[q]) for q in starts if a <= q < s-2]+[0.])\n            return float(.5*np.sqrt(sh[s]*eh[e])+.5*inside.mean()-.15*skipped\n                         -.2*np.mean(inside < parameters["low"]))\n\n        original_utility = utility(old["t_start"], old["t_end"])\n        candidates = []\n        for s in starts:\n            for e in ends:\n                start,end = max(0.,float(times[s]-.05)),min(float(duration),float(times[e]+.05))\n                if e <= s or not parameters["min_s"]-1e-6 <= end-start <= parameters["max_s"]+1e-6:\n                    continue\n                if float(action[s:e+1].max()) < parameters["high"] or float(action[s:e+1].mean()) < .3:\n                    continue\n                if any(base.overlap(start,end,other["t_start"],other["t_end"])>0\n                       for i,other in enumerate(legacy) if i != number):\n                    continue\n                candidates.append((utility(start,end),start,end))\n        candidates.sort(reverse=True)\n        best = candidates[0] if candidates else None\n        margin = best[0]-candidates[1][0] if len(candidates)>1 else 1.\n        accept = best is not None and best[0] >= original_utility+.08 and margin >= .04\n        event = dict(old)\n        if accept:\n            event.update(t_start=best[1], t_end=best[2])\n        # Never split a legacy event or create an extra event in this family.\n        if result and event["t_start"] < result[-1]["t_end"]:\n            event = dict(old)\n            accept = False\n        result.append(event)\n        audit.append(dict(legacy_start=old["t_start"],legacy_end=old["t_end"],\n                          selected_start=event["t_start"],selected_end=event["t_end"],\n                          candidate_pairs=len(candidates),original_utility=original_utility,\n                          candidate_margin=margin,method="refined" if accept else "kept_legacy"))\n    return result, audit\n\n\ndef predictions(tracks, views, family, parameters):\n    events, audits = [], []\n    for key,(times,probs) in tracks.items():\n        if family == "conservative":\n            decoded,audit = conservative_decode(times,probs,views[key]["duration"],**parameters)\n        else:\n            decoded,audit = v6.decode_v6(times,probs,views[key]["duration"],family,**parameters)\n        events.extend(dict(e,stream_id=key) for e in decoded)\n        audits.extend(dict(row,stream_id=key) for row in audit)\n    return events, audits\n\n\ndef subtype_recall(module, events, tracks, views):\n    rows = []\n    for key in tracks:\n        grounds = base.event_ground(views,[key],True)\n        matched = module._match_stream(grounds,[p for p in events if p["stream_id"]==key],.3)\n        used = {i for _,i,_ in matched["pairs"]}\n        for i,g in enumerate(matched["scored"]):\n            source = next(e for e in views[key]["events"] if abs(e["t_start"]-g["t_start"])<1e-6\n                          and abs(e["t_end"]-g["t_end"])<1e-6 and not base.truth(e["ignore"])\n                          and str(e["quality"]).lower()!="low" and not base.truth(e["needs_review"]))\n            rows.append(dict(dataset=source.get("dataset","unknown"),fall_type=source.get("fall_type","unknown"),\n                             source_group=base.source_group(source["source_path"],"fall"),\n                             tp=int(i in used),fn=int(i not in used)))\n    if not rows:\n        return []\n    table = pd.DataFrame(rows).groupby(["dataset","fall_type","source_group"],as_index=False)[["tp","fn"]].sum()\n    table["n_events"] = table.tp+table.fn\n    table["recall"] = table.tp/table.n_events\n    return table.to_dict("records")\n\n\ndef pooled_loss(tracks, views):\n    sums, denominator = np.zeros(3), 0.\n    for key,(times,probs) in tracks.items():\n        y,w = base.targets_for(times,views[key]["events"])\n        p = np.clip(probs.astype(np.float64),1e-7,1-1e-7)\n        loss = -(np.array([3,5,5])[:,None]*y*np.log(p)+(1-y)*np.log1p(-p))\n        sums += (loss*w[None]).sum(axis=1)\n        denominator += float(w.sum())\n    values = sums/max(denominator,1)\n    return dict(loss=float(values @ np.array([1,.5,.5])),action_loss=float(values[0]),\n                start_loss=float(values[1]),end_loss=float(values[2]),\n                scope="weighted BCE on unique 10Hz frames after overlap probability averaging; not window loss")\n\n\ndef signal_audit(ctx):\n    rows = []\n    for key,view in ctx["views"].items():\n        # Outer is not used to investigate signals during fitting/selection.\n        if view["side"] not in ("fit","holdout"):\n            continue\n        features,times = ctx["raw"][key]\n        for event in view["events"]:\n            inside = (times>=event["t_start"]) & (times<event["t_end"])\n            if not inside.any():\n                continue\n            context = (times>=event["t_start"]-2) & (times<event["t_end"]+2) & ~inside\n            row = dict(view_id=key,side=view["side"],event_id=event["event_id"],\n                       dataset=event.get("dataset","unknown"),fall_type=event.get("fall_type","unknown"),\n                       source_path=event["source_path"],source_group=base.source_group(event["source_path"],"fall"),\n                       duration_s=event["t_end"]-event["t_start"],quality=event["quality"],\n                       ignore=base.truth(event["ignore"]),needs_review=base.truth(event["needs_review"]))\n            for radar in range(3):\n                energy = features[radar,inside].mean(axis=0)\n                row[f"radar{radar+1}_residual_mean"] = float(energy.mean())\n                row[f"radar{radar+1}_range_peak_bin"] = int(energy.argmax())\n                row[f"radar{radar+1}_delta_mean"] = float(features[radar+3,inside].mean())\n                row[f"radar{radar+1}_context_residual_mean"] = float(features[radar,context].mean()) if context.any() else None\n            rows.append(row)\n    write_csv(ctx["out"]/"event_signal_audit_fit_inner.csv",rows)\n\n\ndef prepare_context(root, out, cache, spec, cfg, full=False):\n    cfg = replace(cfg,seed=int(spec["seed"]),weight_decay=1e-3 if spec["strategy"]=="T2_weight_decay" else cfg.weight_decay)\n    cfg.validate()\n    if not full:\n        ctx = v6.prepare_nested(root,out,cache,set(spec["inner"]),set(spec["outer"]),cfg,\n                                restore=(out/"config.json").exists())\n    else:\n        base.seed_everything(cfg.seed)\n        out.mkdir(parents=True,exist_ok=True)\n        views,raw,manifest,skipped = base.build_views(root,cache,set(),cfg)\n        mean,scale = base.fit_normalizer(raw,views)\n        module = v6.evaluator_for(root)\n        ctx = dict(root=root,out=out,cache_root=cache,views=views,raw=raw,mean=mean,scale=scale,cfg=cfg,\n                   device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),evaluate=module.evaluate,\n                   validator=module.validate_predictions,evaluator=module,holdout=set())\n        write_csv(out/"split_manifest.csv",manifest)\n        write_csv(out/"skipped_intervals.csv",skipped,columns=["source_stream","side","start","end","reason"])\n        np.savez(out/"normalizer_stats.npz",mean=mean,scale=scale)\n    ctx["positives"],ctx["negatives"],ctx["fit_eval"],ctx["val_eval"] = candidate_indices(ctx["raw"],ctx["views"],cfg,not full)\n    ctx.update(strategy=spec["strategy"],spec=spec,full=full,cfg=cfg)\n    ctx["config"] = dict(asdict(cfg),version=VERSION,mode="all_train_refit" if full else "nested_train",\n                         strategy=spec["strategy"],inner_groups=spec["inner"],outer_groups=spec["outer"],\n                         all_fall_types_included=True,adl_included=True,\n                         protocol="refit all official train; fixed epochs/LR/thresholds from inner" if full else\n                         "source-exclusive fit/inner/outer before filtering; selection uses inner only")\n    write_json(out/"config.json",ctx["config"])\n    write_json(out/"data_audit.json",dict(normalizer="fit-only per-channel zscore",filter_reset_per_view=True,\n               event_owners=len(ctx["positives"]),all_fall_types_included=True,\n               hours={side:sum(v["duration"] for v in ctx["views"].values() if v["side"]==side)/3600\n                      for side in ("fit","holdout","outer")},\n               fall_types=sorted({str(e.get("fall_type","unknown")) for e in event_metadata(ctx["views"]).values()})))\n    write_json(out/"environment.json",dict(torch=str(torch.__version__),numpy=str(np.__version__),\n                  cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))\n    if not (out/"event_signal_audit_fit_inner.csv").exists():\n        signal_audit(ctx)\n    return ctx\n\n\ndef rng_state():\n    state = np.random.get_state()\n    return dict(python=random.getstate(),numpy=[state[0],state[1].tolist(),state[2],state[3],state[4]],\n                torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(saved):\n    random.setstate(saved["python"])\n    state = saved["numpy"]\n    np.random.set_state((state[0],np.array(state[1],dtype=np.uint32),state[2],state[3],state[4]))\n    torch.set_rng_state(saved["torch"])\n    if torch.cuda.is_available() and saved["cuda"]:\n        torch.cuda.set_rng_state_all(saved["cuda"])\n\n\ndef load_selected(ctx):\n    payload = torch.load(ctx["out"]/"best_checkpoint_fixed_decoder.pt",map_location="cpu",weights_only=True)\n    if payload["config"] != ctx["config"]:\n        raise ValueError("Checkpoint configuration differs; use a new RUN_TAG")\n    if not (np.allclose(payload["mean"],ctx["mean"],rtol=1e-5,atol=1e-12) and\n            np.allclose(payload["scale"],ctx["scale"],rtol=1e-5,atol=1e-12)):\n        raise ValueError("Normalizer/data differs from saved checkpoint")\n    model = v5.CNNTCNV5(ctx["cfg"]).to(ctx["device"])\n    model.load_state_dict(payload["model"])\n    ctx.update(model=model,best_epoch=payload["epoch"],training_complete=True)\n    ctx["config"]["source_sha256"] = v6.file_hash(ctx["out"]/"best_checkpoint_fixed_decoder.pt")\n    return ctx\n\n\ndef train_context(ctx, run_mode, fixed_schedule=None):\n    out,cfg = ctx["out"],ctx["cfg"]\n    if (out/"training_complete.json").exists():\n        return load_selected(ctx)\n    if run_mode == "export_only":\n        raise ValueError(f"Training incomplete: {out}. Use continue to resume before export.")\n    model = v5.CNNTCNV5(cfg).to(ctx["device"])\n    optimizer = torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=.5,patience=1)\n    history,sampling,exposures,types = [],[],[],[]\n    best_key,best_epoch,stale,best_model = None,0,0,None\n    last = out/"last_checkpoint.pt"\n    if last.exists():\n        saved = torch.load(last,map_location="cpu",weights_only=True)\n        if saved["config"] != ctx["config"] or saved["schedule"] != fixed_schedule:\n            raise ValueError("Resume configuration changed; use a new RUN_TAG")\n        if not (np.allclose(saved["mean"],ctx["mean"],rtol=1e-5,atol=1e-12) and\n                np.allclose(saved["scale"],ctx["scale"],rtol=1e-5,atol=1e-12)):\n            raise ValueError("Resume normalizer changed")\n        model.load_state_dict(saved["model"])\n        optimizer.load_state_dict(saved["optimizer"])\n        scheduler.load_state_dict(saved["scheduler"])\n        best_key,best_epoch,stale,best_model = saved["best_key"],saved["best_epoch"],saved["stale"],saved["best_model"]\n        for filename,destination in [("training_history_v7.csv",history),("epoch_sampling_manifest.csv",sampling),\n                                     ("epoch_event_exposure.csv",exposures),("epoch_inner_subtype_recall.csv",types)]:\n            if (out/filename).exists():\n                rows = pd.read_csv(out/filename).fillna("").to_dict("records")\n                destination.extend(r for r in rows if r["epoch"] <= saved["epoch"])\n        if len(history) != saved["epoch"]:\n            raise ValueError("Resume history/checkpoint mismatch")\n        restore_rng(saved["rng"])\n        print("Resuming after epoch",saved["epoch"],flush=True)\n    fit_loader = base.make_loader(ctx["fit_eval"],ctx)\n    val_loader = base.make_loader(ctx["val_eval"],ctx) if not ctx["full"] else None\n    start_epoch = len(history)+1\n    for epoch in range(start_epoch,cfg.max_epochs+1):\n        if not ctx["full"] and history and epoch>cfg.min_epochs and stale>=cfg.patience:\n            break\n        started = time.time()\n        if fixed_schedule is not None:\n            for group in optimizer.param_groups:\n                group["lr"] = fixed_schedule[epoch-1]\n        lr = optimizer.param_groups[0]["lr"]\n        index,manifest = sample_training(ctx,epoch)\n        exposure,summary = exposure_audit(ctx,index,manifest,epoch)\n        sampling.extend(manifest);exposures.extend(exposure)\n        augmented = v5.loss_pass(model,base.make_loader(index,ctx,True,epoch),ctx["device"],optimizer)\n        clean = v5.loss_pass(model,fit_loader,ctx["device"])\n        row = dict(epoch=epoch,augmented_train_loss=augmented["loss"],train_loss=clean["loss"],lr=lr,\n                   optimizer_updates=augmented["optimizer_updates"],cumulative_updates=epoch*cfg.steps_per_epoch,\n                   weighted_training_frames=augmented["valid_weighted_frames"],**summary)\n        if not ctx["full"]:\n            val = v5.loss_pass(model,val_loader,ctx["device"])\n            scheduler.step(val["loss"])\n            tracks = base.infer_side(model,ctx["raw"],ctx["views"],"holdout",ctx["mean"],ctx["scale"],cfg,ctx["device"])\n            events = base.predict_tracks(tracks,ctx["views"],FIXED)\n            report = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n            fm = base.frame_metrics(tracks,ctx["views"])\n            row.update(val_loss=val["loss"],loss_gap=val["loss"]-clean["loss"],\n                       val_event_f1_mean=report["ranking_score"]["f1_mean"],val_frame_f1=fm["f1"],\n                       val_frame_accuracy=fm["accuracy"],val_frame_balanced_accuracy=fm["balanced_accuracy"],\n                       val_false_alarms_per_hour=report["primary"]["false_alarms_per_hour"])\n            types.extend(dict(r,epoch=epoch) for r in subtype_recall(ctx["evaluator"],events,tracks,ctx["views"]))\n            for label,metric in (("train",clean),("val",val)):\n                row.update({f"{label}_{k}":metric[k] for k in ("action_loss","start_loss","end_loss")})\n            key = (*base.rank_key(report),-val["loss"])\n            if best_key is None or key>tuple(best_key):\n                best_key,best_epoch,stale,best_model = key,epoch,0,cpu_state(model)\n            else:\n                stale += 1\n        else:\n            best_epoch,best_model = epoch,cpu_state(model)\n        row.update(next_lr=optimizer.param_groups[0]["lr"],seconds=time.time()-started)\n        history.append(row)\n        write_csv(out/"training_history_v7.csv",history)\n        write_csv(out/"epoch_sampling_manifest.csv",sampling)\n        write_csv(out/"epoch_event_exposure.csv",exposures)\n        write_csv(out/"epoch_inner_subtype_recall.csv",types,\n                  columns=["dataset","fall_type","source_group","tp","fn","n_events","recall","epoch"])\n        write_csv(out/"checkpoint_event_f1.csv",[r for r in history if "val_event_f1_mean" in r],\n                  columns=["epoch","val_event_f1_mean","val_false_alarms_per_hour","val_loss"])\n        payload = dict(model=best_model,config=ctx["config"],epoch=best_epoch,\n                       mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist())\n        save_checkpoint(out/"best_checkpoint_fixed_decoder.pt",payload)\n        save_checkpoint(last,dict(model=cpu_state(model),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),\n                        config=ctx["config"],mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),epoch=epoch,\n                        best_key=best_key,best_epoch=best_epoch,stale=stale,best_model=best_model,\n                        rng=rng_state(),schedule=fixed_schedule))\n        print(f"{ctx[\'spec\'][\'name\']} epoch {epoch:02d}: train(clean)={clean[\'loss\']:.4f} "\n              +(f"inner={row[\'val_loss\']:.4f} F1={row[\'val_event_f1_mean\']:.4f} " if not ctx["full"] else "all-train fixed schedule ")\n              +f"unique={summary[\'unique_drawn_windows\']}/{len(index)}",flush=True)\n    write_json(out/"training_complete.json",dict(best_epoch=best_epoch,trained_epochs=len(history),\n                 reason="fixed inner-derived schedule" if ctx["full"] else "early stopping or max epochs"))\n    return load_selected(ctx)\n\n\ndef select_decoders(ctx):\n    tracks = v6.tracks_cached(ctx,"inner")\n    rows, fixed_rows, parameters = [],[],{}\n    for family in DECODERS:\n        events,_ = predictions(tracks,ctx["views"],family,FIXED)\n        fixed = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n        fixed_rows.append(dict(decoder=family,**fixed["ranking_score"]))\n        best_key,best = None,None\n        for high in (.4,.5,.6,.7,.8):\n            for low in (.1,.2,.3):\n                candidate = dict(FIXED,high=high,low=low)\n                events,_ = predictions(tracks,ctx["views"],family,candidate)\n                report = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n                rows.append(dict(decoder=family,high=high,low=low,**report["ranking_score"]))\n                if best_key is None or base.rank_key(report)>best_key:\n                    best_key,best = base.rank_key(report),candidate\n        parameters[family] = best\n    write_csv(ctx["out"]/"cnn_tcn_v7_decoder_grid.csv",rows)\n    selection = dict(version=VERSION,checkpoint_sha256=ctx["config"]["source_sha256"],\n                     parameters=parameters,fixed_comparison=fixed_rows,best_epoch=ctx["best_epoch"],\n                     scope="inner only; no outer/public selection",training_complete=True)\n    v6.locked_json(ctx["out"]/"decoder_selection_locked.json",selection)\n    return selection\n\n\ndef export_report(ctx, split, tracks, views, family, parameters):\n    events,audit = predictions(tracks,views,family,parameters)\n    prefix = f"cnn_tcn_v7_{family}_{split}"\n    out = ctx["out"]\n    path = out/f"{prefix}_submission.csv"\n    write_csv(path,events,columns=["stream_id","t_start","t_end","score"])\n    checked = ctx["validator"](path,{k:views[k]["duration"] for k in tracks})\n    frame = base.frame_metrics(tracks,views)\n    frame["all_background_accuracy"] = (frame["tn"]+frame["fp"])/max(frame["n_frames"],1)\n    report = dict(version=VERSION,split=split,decoder=family,parameters=parameters,strategy=ctx["strategy"],\n                  best_epoch=ctx["best_epoch"],training_complete=True,mode=ctx["config"]["mode"],\n                  exposure_hours=sum(views[k]["duration"] for k in tracks)/3600,\n                  ignore_flagged=base.event_report(ctx["evaluate"],checked,views,tracks),\n                  all_events=base.event_report(ctx["evaluate"],checked,views,tracks,False),\n                  frame_metrics=frame,decoded_frame_metrics=v6.decoded_frame_metrics(tracks,views,checked),\n                  pooled_probability_loss=pooled_loss(tracks,views),\n                  note="Offline; no online alarm latency measurement. loso aliases are not full LOSO.")\n    write_json(out/f"{prefix}_report.json",report)\n    write_csv(out/f"{prefix}_boundary_audit.csv",audit if audit else [],\n              columns=None if audit else ["stream_id","method"])\n    errors = v6.errors_v6(ctx["evaluator"],checked,views,tracks)\n    for row in errors:\n        if "gt_start" in row:\n            row["gt_duration"] = row["gt_end"]-row["gt_start"]\n        if "pred_start" in row:\n            row["pred_duration"] = row["pred_end"]-row["pred_start"]\n        if row["kind"]=="TP":\n            row["duration_error"] = row["pred_duration"]-row["gt_duration"]\n    write_csv(out/f"{prefix}_event_errors.csv",errors,\n              columns=["stream_id","kind","gt_start","gt_end","pred_start","pred_end","best_tiou","start_error","end_error",\n                       "gt_duration","pred_duration","duration_error"])\n    write_csv(out/f"{prefix}_subtype_recall.csv",subtype_recall(ctx["evaluator"],checked,tracks,views),\n              columns=["dataset","fall_type","source_group","tp","fn","n_events","recall"])\n    return report\n\n\ndef export_context(ctx, selection, family, public):\n    out = ctx["out"]\n    reports, fixed_rows = {},[]\n    for split in ("train","inner","outer","public"):\n        views = ctx["views"]\n        if split == "public":\n            if not public:\n                continue\n            views,raw,manifest,_ = base.build_views(ctx["root"],ctx["cache_root"],set(),ctx["cfg"],public=True)\n            write_csv(out/"public_manifest.csv",manifest)\n            tracks = v6.tracks_cached(ctx,split,views,raw)\n        else:\n            tracks = v6.tracks_cached(ctx,split)\n        if not tracks:\n            continue\n        for decoder in DECODERS:\n            report = export_report(ctx,split,tracks,views,decoder,selection["parameters"][decoder])\n            reports[f"{decoder}:{split}"] = report\n            events,_ = predictions(tracks,views,decoder,FIXED)\n            fixed = base.event_report(ctx["evaluate"],events,views,tracks)\n            fixed_rows.append(dict(split=split,decoder=decoder,**fixed["ranking_score"]))\n        alias = "loso" if split == "outer" else split\n        for suffix in ("submission.csv","report.json"):\n            shutil.copyfile(out/f"cnn_tcn_v7_{family}_{split}_{suffix}",out/f"cnn_tcn_v7_{alias}_{suffix}")\n        if split == "public":\n            del raw\n    write_csv(out/"fixed_threshold_decoder_comparison_v7.csv",fixed_rows)\n    save_checkpoint(out/"cnn_tcn_v7_selected_model.pt",dict(model=cpu_state(ctx["model"]),epoch=ctx["best_epoch"],\n                    config=ctx["config"],mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),\n                    decoder_family=family,decoder=selection["parameters"][family]))\n    write_csv(out/"hard_negative_manifest.csv",[],columns=["stream_id","start","score"])\n    write_json(out/"report_v7.json",dict(version=VERSION,selected_decoder=family,strategy=ctx["strategy"],\n                 best_epoch=ctx["best_epoch"],mode=ctx["config"]["mode"],reports=reports,\n                 limitations=["Only supplied data used; all fall types and ADL included.",\n                              "Public has been inspected in earlier research and remains a development benchmark.",\n                              "Three nested folds are not full LOSO; seeds repeat the same subjects.",\n                              "Sampling changes may alter class proportions; inspect exposure logs.",\n                              "Decoder metrics do not imply reduced neural-network overfitting."]))\n    write_json(out/"run_complete_v7.json",dict(version=VERSION,selected_decoder=family,public=public))\n\n\ndef make_specs(folds, seeds, experiments):\n    if not experiments or len(set(experiments)) != len(experiments) or any(e not in EXPERIMENTS for e in experiments):\n        raise ValueError("Choose unique known experiment profiles")\n    if not seeds or len(set(seeds)) != len(seeds):\n        raise ValueError("Choose unique seeds")\n    specs = []\n    for strategy in experiments:\n        for name,inner,outer in folds:\n            if not inner or not outer or set(inner)&set(outer):\n                raise ValueError("Inner and outer must be nonempty and disjoint")\n            if not name or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in name):\n                raise ValueError("Use a simple ASCII fold name")\n            for seed in seeds:\n                specs.append(dict(name=f"{strategy}/{name}/seed_{int(seed)}",strategy=strategy,fold=name,\n                                  inner=sorted(inner),outer=sorted(outer),seed=int(seed)))\n    if not specs or len({s["name"] for s in specs}) != len(specs):\n        raise ValueError("Duplicate or empty run plan")\n    return specs\n\n\ndef choose_suite(specs, selections, run_root):\n    rows = []\n    for spec,selection in zip(specs,selections):\n        rows.extend(dict(strategy=spec["strategy"],run=spec["name"],fold=spec["fold"],**r) for r in selection["fixed_comparison"])\n    ranking = pd.DataFrame(rows).groupby(["strategy","decoder"],sort=False,as_index=False).f1_mean.mean()\n    ranking = ranking.sort_values("f1_mean",ascending=False,kind="stable")\n    strategy,family = ranking.iloc[0][["strategy","decoder"]]\n    relevant = [(s,d) for s,d in zip(specs,selections) if s["strategy"]==strategy]\n    epochs = max(1,int(np.floor(np.median([d["best_epoch"] for _,d in relevant])+.5)))\n    schedule = []\n    histories = [pd.read_csv(run_root/s["name"]/"training_history_v7.csv") for s,_ in relevant]\n    for epoch in range(1,epochs+1):\n        schedule.append(float(np.median([h.loc[h.epoch==min(epoch,int(h.epoch.max())),"lr"].iloc[0] for h in histories])))\n    parameters = {decoder:dict(FIXED,\n                 high=float(np.median([d["parameters"][decoder]["high"] for _,d in relevant])),\n                 low=float(np.median([d["parameters"][decoder]["low"] for _,d in relevant]))) for decoder in DECODERS}\n    best_spec,best_selection = max(relevant,key=lambda sd:next(r["f1_mean"] for r in sd[1]["fixed_comparison"] if r["decoder"]==family))\n    fold_locks = {}\n    for fold,group in pd.DataFrame(rows).groupby("fold",sort=False):\n        r = group.groupby(["strategy","decoder"],sort=False,as_index=False).f1_mean.mean()\n        r = r.sort_values("f1_mean",ascending=False,kind="stable")\n        fold_locks[fold] = dict(strategy=r.iloc[0].strategy,selected_decoder=r.iloc[0].decoder,\n                                criterion="only this outer fold\'s INNER groups across planned seeds")\n    return dict(version=VERSION,strategy=strategy,selected_decoder=family,selected_run=best_spec["name"],\n                criterion="mean INNER fixed-threshold F1; all choices locked before outer/public",\n                ranking=ranking.to_dict("records"),refit_epochs=epochs,refit_lr_schedule=schedule,\n                outer_selection_by_fold=fold_locks,\n                refit_parameters=parameters,refit_seed=2027,\n                refit_note="Full-train scaler is refit; transferred thresholds may shift operating point. No public tuning.")\n\n\ndef collect_results(run_root, specs, final_refit):\n    rows = []\n    locations = [(run_root/s["name"],s) for s in specs]\n    if final_refit:\n        locations.append((run_root/"final_all_train",dict(name="final_all_train",strategy="final_refit",fold="all",seed=2027)))\n    for out,spec in locations:\n        for path in sorted(out.glob("cnn_tcn_v7_*_report.json")):\n            r = read_json(path)\n            if path.name != f"cnn_tcn_v7_{r[\'decoder\']}_{r[\'split\']}_report.json":\n                continue\n            row = v6.report_row(r["split"],r["decoder"],r)\n            row.update(strategy=spec["strategy"],run=spec["name"],fold=spec["fold"],seed=spec["seed"],\n                       report_path=str(path),pooled_loss=r["pooled_probability_loss"]["loss"],\n                       all_background_accuracy=r["frame_metrics"]["all_background_accuracy"])\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef aggregate_results(table):\n    rows = []\n    for (strategy,split,family),group in table.groupby(["strategy","split","decoder"],sort=False):\n        reports = [read_json(p) for p in group.report_path]\n        hours = sum(r["exposure_hours"] for r in reports)\n        f1 = {}\n        for theta in ("0.1","0.3","0.5","0.7"):\n            count = {k:sum(r["ignore_flagged"]["by_theta"][theta][k] for r in reports) for k in ("tp","fp","fn")}\n            den = 2*count["tp"]+count["fp"]+count["fn"]\n            f1[theta] = 2*count["tp"]/den if den else 0.\n            if theta=="0.3":\n                main = count\n        frames = {k:sum(r["frame_metrics"][k] for r in reports) for k in ("tn","fp","fn","tp")}\n        n = sum(frames.values())\n        rows.append(dict(strategy=strategy,split=split,decoder=family,runs=len(group),\n                         micro_event_f1_mean=float(np.mean([f1[t] for t in ("0.1","0.3","0.5")])),\n                         micro_f1_03=f1["0.3"],micro_f1_05=f1["0.5"],macro_f1_mean=float(group.event_f1_mean.mean()),\n                         run_f1_std=float(group.event_f1_mean.std()) if len(group)>1 else None,\n                         tp=main["tp"],fp=main["fp"],fn=main["fn"],exposure_hours=hours,\n                         false_alarms_per_hour=main["fp"]/hours if hours else None,\n                         model_frame_accuracy=(frames["tn"]+frames["tp"])/n if n else None,\n                         note="Repeated seeds/models repeat subjects and exposure; not independent observations."))\n    return pd.DataFrame(rows)\n\n\ndef run_suite(root, run_root, cache, cfg, folds, seeds, experiments, run_mode="continue",\n              public=True, final_refit=True, implementation_hash=""):\n    root,run_root,cache = Path(root),Path(run_root),Path(cache)\n    if run_mode not in ("train","continue","export_only"):\n        raise ValueError("Unknown RUN_MODE")\n    if cfg.window_frames!=80 or cfg.model_size!="standard" or cfg.gain_augment or cfg.feature_noise_std:\n        raise ValueError("Keep 8s standard model and gain/noise off for this controlled V7 study")\n    specs = make_specs(folds,seeds,experiments)\n    plan = dict(version=VERSION,specs=specs,config=asdict(cfg),public=public,final_refit=final_refit,\n                implementation_hash=implementation_hash,\n                data_root=str(root.resolve()),metadata_hashes={p:v6.file_hash(root/p) for p in ("streams_events.csv","streams_composition.csv")},\n                streams=[dict(path=str(p.relative_to(root)),size=p.stat().st_size,mtime=p.stat().st_mtime_ns)\n                         for p in sorted((root/"streams").glob("*/*.h5"))])\n    run_root.mkdir(parents=True,exist_ok=True)\n    if run_mode=="train" and (run_root/"suite_plan_v7.json").exists():\n        raise FileExistsError("Existing run; use continue or a NEW RUN_TAG")\n    v6.locked_json(run_root/"suite_plan_v7.json",plan)\n    selections = []\n    # Phase 1: fit and inner only. No outer/public scoring before the suite lock.\n    for spec in specs:\n        out = run_root/spec["name"]\n        if (out/"decoder_selection_locked.json").exists():\n            selection = read_json(out/"decoder_selection_locked.json")\n            if selection["checkpoint_sha256"] != v6.file_hash(out/"best_checkpoint_fixed_decoder.pt"):\n                raise ValueError("Checkpoint changed after calibration")\n        else:\n            ctx = prepare_context(root,out,cache,spec,cfg)\n            ctx = train_context(ctx,run_mode)\n            selection = select_decoders(ctx)\n            del ctx\n            v5.release_memory()\n        selections.append(selection)\n    lock = choose_suite(specs,selections,run_root)\n    v6.locked_json(run_root/"selection_locked_before_outer_public.json",lock)\n    print("Locked strategy/decoder:",lock["strategy"],lock["selected_decoder"],flush=True)\n\n    # Phase 2: selected training recipe refit on ALL official train; no val labels.\n    if final_refit:\n        full_spec = dict(name="final_all_train",strategy=lock["strategy"],inner=[],outer=[],seed=lock["refit_seed"],fold="all")\n        full_cfg = replace(cfg,max_epochs=lock["refit_epochs"],min_epochs=lock["refit_epochs"])\n        out = run_root/"final_all_train"\n        if not (out/"run_complete_v7.json").exists():\n            ctx = prepare_context(root,out,cache,full_spec,full_cfg,full=True)\n            ctx = train_context(ctx,run_mode,fixed_schedule=lock["refit_lr_schedule"])\n            selection = dict(parameters=lock["refit_parameters"],scope="thresholds transferred from inner folds")\n            write_json(out/"decoder_selection_locked.json",selection)\n            export_context(ctx,selection,lock["selected_decoder"],public)\n            del ctx\n            v5.release_memory()\n    # Phase 3: frozen rules evaluated on outer and public for all planned runs.\n    for spec,selection in zip(specs,selections):\n        out = run_root/spec["name"]\n        if (out/"run_complete_v7.json").exists():\n            continue\n        ctx = prepare_context(root,out,cache,spec,cfg)\n        ctx = load_selected(ctx)\n        export_context(ctx,selection,lock["outer_selection_by_fold"][spec["fold"]]["selected_decoder"],public)\n        del ctx\n        v5.release_memory()\n    selected = run_root/("final_all_train" if final_refit else lock["selected_run"])\n    for split in ("train","public"):\n        for suffix in ("submission.csv","report.json"):\n            p = selected/f"cnn_tcn_v7_{lock[\'selected_decoder\']}_{split}_{suffix}"\n            if p.exists():\n                shutil.copyfile(p,run_root/f"cnn_tcn_v7_{split}_{suffix}")\n    payload = torch.load(selected/"cnn_tcn_v7_selected_model.pt",map_location="cpu",weights_only=True)\n    payload["decoder_family"] = lock["selected_decoder"]\n    payload["decoder"] = read_json(selected/"decoder_selection_locked.json")["parameters"][lock["selected_decoder"]]\n    save_checkpoint(run_root/"cnn_tcn_v7_selected_model.pt",payload)\n    del payload\n    table = collect_results(run_root,specs,final_refit)\n    table.to_csv(run_root/"experiment_comparison_v7.csv",index=False)\n    aggregate_results(table).to_csv(run_root/"experiment_aggregate_v7.csv",index=False)\n    nested = selected_outer_rows(table,lock)\n    nested.to_csv(run_root/"nested_selected_outer_v7.csv",index=False)\n    aggregate_results(nested).to_csv(run_root/"nested_selected_outer_aggregate_v7.csv",index=False)\n    write_json(run_root/"suite_runs_v7.json",dict(specs=specs,final_refit=final_refit,selected=lock,\n                                               final_output=str(selected),public=public))\n    return table,lock\n\n\ndef selected_outer_rows(table, lock):\n    chosen = []\n    for fold,selection in lock["outer_selection_by_fold"].items():\n        subset = table[(table.fold==fold)&(table.split=="outer")&\n                       (table.strategy==selection["strategy"])&(table.decoder==selection["selected_decoder"])].copy()\n        subset["source_strategy"] = subset.strategy\n        subset["source_decoder"] = subset.decoder\n        subset["strategy"] = "nested_inner_selected"\n        subset["decoder"] = "per_fold_inner_choice"\n        chosen.append(subset)\n    return pd.concat(chosen,ignore_index=True)\n\n\ndef plots_and_archive(run_root):\n    """Read saved V7 results only; this function never runs a model."""\n    import matplotlib.pyplot as plt\n    from IPython.display import display\n    run_root = Path(run_root)\n    index = read_json(run_root/"suite_runs_v7.json")\n    lock = index["selected"]\n    table = collect_results(run_root,index["specs"],index["final_refit"])\n    display_columns = ["split","decoder","event_f1_mean","event_f1_03","event_f1_05","model_frame_f1",\n                       "model_frame_accuracy","model_balanced_accuracy","decoded_frame_f1","decoded_frame_accuracy"]\n    locations = [(run_root/s["name"],s["name"]) for s in index["specs"]]\n    if index["final_refit"]:\n        locations.append((run_root/"final_all_train","final_all_train"))\n    for out,name in locations:\n        report = read_json(out/"report_v7.json")\n        selected_family = report["selected_decoder"]\n        history = pd.read_csv(out/"training_history_v7.csv")\n        nested = "val_loss" in history\n        fig,axes = plt.subplots(1,2,figsize=(12,4))\n        axes[0].plot(history.epoch,history.train_loss,"o-",ms=3,label="Train loss (clean eval)")\n        axes[0].plot(history.epoch,history.augmented_train_loss,alpha=.4,label="Training loss (dropout on)")\n        if nested:\n            axes[0].plot(history.epoch,history.val_loss,"o-",ms=3,label="Inner validation loss (clean eval)")\n            axes[1].plot(history.epoch,history.val_event_f1_mean,"o-",ms=3,label="Inner event F1, fixed legacy decoder")\n            axes[1].plot(history.epoch,history.val_frame_f1,alpha=.7,label="Inner frame F1")\n            axes[1].set_ylim(0,1.02)\n        else:\n            axes[1].plot(history.epoch,history.unique_drawn_windows,"o-",label="Unique sampled windows")\n            axes[1].plot(history.epoch,history.train_windows,label="Window draws")\n            axes[1].set_title("All-train refit: epochs/LR locked from inner folds")\n        axes[0].axvline(report["best_epoch"],ls="--",color="gray",label="Selected epoch" if nested else "Final fixed epoch")\n        axes[0].set(xlabel="Epoch",ylabel="Weighted window BCE")\n        axes[1].set_xlabel("Epoch")\n        for ax in axes:\n            ax.grid(alpha=.2);ax.legend(fontsize=8)\n        fig.suptitle(name+ (" | validation = inner only" if nested else " | all official train"),fontsize=11)\n        fig.tight_layout();fig.savefig(out/"training_curves_v7.png",dpi=150);plt.show();plt.close(fig)\n        rows = table[table.run==name]\n        matrix = rows[display_columns].set_index(["split","decoder"])\n        matrix.to_csv(out/"metrics_matrix_v7.csv")\n        print("\\n",name,"| Selected decoder:",selected_family)\n        display(matrix.style.format("{:.2%}",na_rep="N/A"))\n        columns = ["event_f1_mean","event_f1_05","model_frame_f1","model_frame_accuracy","decoded_frame_f1"]\n        values = matrix[columns].to_numpy(dtype=float)*100\n        fig,ax = plt.subplots(figsize=(10,max(3,len(matrix)*.35+1.8)))\n        heat = ax.imshow(np.ma.masked_invalid(values),cmap="YlGnBu",vmin=0,vmax=100,aspect="auto")\n        ax.set_xticks(range(len(columns)),columns,rotation=18,ha="right")\n        ax.set_yticks(range(len(matrix)),[f"{a} / {b}" for a,b in matrix.index])\n        for a in range(len(matrix)):\n            for b in range(len(columns)):\n                ax.text(b,a,f"{values[a,b]:.2f}%",ha="center",va="center",fontsize=8,\n                        color="white" if values[a,b]>65 else "black")\n        ax.set_title("Event F1 and frame F1 / accuracy (different definitions)")\n        fig.colorbar(heat,ax=ax);fig.tight_layout();fig.savefig(out/"f1_accuracy_matrix_v7.png",dpi=150)\n        plt.show();plt.close(fig)\n        selected = [(key.split(":")[1],r) for key,r in report["reports"].items() if key.startswith(selected_family+":")]\n        fig,axes = plt.subplots(1,len(selected),figsize=(3.6*len(selected),3.6),squeeze=False)\n        for i,(split,r) in enumerate(selected):\n            m = r["frame_metrics"]\n            counts = np.array([[m["tn"],m["fp"]],[m["fn"],m["tp"]]])\n            ax = axes[0,i];ax.imshow(counts,cmap="Blues")\n            ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Non-fall","Fall"],yticklabels=["Non-fall","Fall"],\n                   xlabel="Predicted",ylabel="Actual",title=split+": model frames")\n            for a in range(2):\n                for b in range(2):\n                    ax.text(b,a,f"{counts[a,b]:,}",ha="center",va="center",color="white" if counts[a,b]>counts.max()/2 else "black")\n        fig.tight_layout();fig.savefig(out/"frame_confusion_matrices_v7.png",dpi=150);plt.show();plt.close(fig)\n        subtype_rows = []\n        for split,r in selected:\n            p = out/f"cnn_tcn_v7_{selected_family}_{split}_subtype_recall.csv"\n            s = pd.read_csv(p)\n            if len(s):\n                s = s.groupby(["dataset","fall_type"],as_index=False)[["tp","fn","n_events"]].sum()\n                s["recall"] = s.tp/s.n_events\n                s["split"] = split\n                subtype_rows.extend(s.to_dict("records"))\n        subtype = pd.DataFrame(subtype_rows)\n        write_csv(out/"selected_subtype_recall_v7.csv",subtype_rows)\n        print("Recall by type; denominators matter. All types were included during fitting:")\n        display(subtype)\n        # A deterministic timeline uses the first scored event, not a selected error.\n        split = next(s for s in ("public","outer","inner","train") if (out/f"probabilities_{s}.npz").exists())\n        meta = read_json(out/f"probabilities_{split}_index.json")\n        with np.load(out/f"probabilities_{split}.npz") as saved:\n            first = next(((i,key,v,e) for i,(key,v) in enumerate(sorted(meta["views"].items())) for e in v["events"]\n                          if not base.truth(e["ignore"]) and str(e["quality"]).lower()!="low" and not base.truth(e["needs_review"])),None)\n            if first is not None:\n                i,key,view,event = first\n                times,probs = saved[f"t{i}"],saved[f"p{i}"]\n                mask = (times>=event["t_start"]-5)&(times<=event["t_end"]+5)\n                fig,ax = plt.subplots(figsize=(11,4))\n                for c,label in enumerate(("Action","Start","End")):\n                    ax.plot(times[mask],probs[c,mask],label=label)\n                ax.axvspan(event["t_start"],event["t_end"],color="green",alpha=.15,label="Ground truth")\n                colors = ("#dc2626","#7c3aed","#0f766e")\n                parameters = read_json(out/"decoder_selection_locked.json")["parameters"]\n                for d,family in enumerate(DECODERS):\n                    es,_ = predictions({key:(times,probs)},{key:view},family,parameters[family])\n                    for n,e in enumerate([e for e in es if e["t_end"]>event["t_start"]-5 and e["t_start"]<event["t_end"]+5]):\n                        ax.plot([e["t_start"],e["t_end"]],[-.08-d*.08]*2,lw=4,color=colors[d],label=family if n==0 else None)\n                ax.set(xlabel="Seconds",ylabel="Probability",title=f"{split}: {key}, first scored event")\n                ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/"probability_timeline_v7.png",dpi=150)\n                plt.show();plt.close(fig)\n    aggregate = aggregate_results(table)\n    table.to_csv(run_root/"experiment_comparison_v7.csv",index=False)\n    aggregate.to_csv(run_root/"experiment_aggregate_v7.csv",index=False)\n    print("Pooled event counts by profile/split/decoder; repeated models are not independent participants:")\n    display(aggregate.drop(columns="note").round(4))\n    nested = selected_outer_rows(table,lock)\n    nested.to_csv(run_root/"nested_selected_outer_v7.csv",index=False)\n    nested_summary = aggregate_results(nested)\n    nested_summary.to_csv(run_root/"nested_selected_outer_aggregate_v7.csv",index=False)\n    print("Nested procedure: each outer fold uses its OWN inner-selected profile and decoder:")\n    display(nested_summary.drop(columns="note").round(4))\n    print("INNER selected:",lock["strategy"],lock["selected_decoder"])\n    print("Final refit epochs:",lock["refit_epochs"],"| Public is evaluated after all selections are locked.")\n    print("Accuracy has a strong all-background baseline. Use event F1, FP/hour and subtype recall.")\n    archive = run_root/"cnn_tcn_v7_suite_outputs.zip"\n    with base.zipfile.ZipFile(archive,"w",base.zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(run_root.rglob("*")):\n            if path.is_file() and path.suffix in (".csv",".json",".png",".npz",".pt") and path.name!="last_checkpoint.pt":\n                z.write(path,path.relative_to(run_root))\n    print("Saved:",archive)\n    return archive\n'
v7=types.ModuleType('cnn_tcn_v7')
sys.modules['cnn_tcn_v7']=v7
exec(compile(V7_SOURCE,'cnn_tcn_v7.py','exec'),v7.__dict__)
V9_SOURCE='"""V9 factorial study of frame normalization and EMA-residual input.\n\nImport defines functions only. Colab explicitly calls run_suite to train.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport shutil\nimport time\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport cnn_tcn_v4 as base\nimport cnn_tcn_v5 as v5\nimport cnn_tcn_v6 as v6\nimport cnn_tcn_v7 as v7\n\nnp, pd, torch = base.np, base.pd, base.torch\nVERSION = "v9-frame-normalization-residual-ablation-1"\nDECODERS = ("conservative",)\nPROFILES = ("A_residual_delta", "B_frame_norm", "C_delta_only", "D_delta_frame_norm")\nFIXED = dict(v7.FIXED)\nwrite_json, write_csv, save_checkpoint = v7.write_json, v7.write_csv, v7.save_checkpoint\nread_json = v7.read_json\n\n\n@dataclass\nclass ConfigV9(v5.ConfigV5):\n    lr: float = 3e-4\n    jitter_frames: int = 0\n    train_hop_frames: int = 10\n    low_positive_fraction: float = .14\n    owner_draw_cap: int = 4\n    lr_floor_ratio: float = .1\n    frame_floor_quantile: float = .10\n    floor_sample_frames: int = 100000\n    floor_absolute_min: float = 1e-12\n\n    def validate(self):\n        super().validate()\n        if self.window_frames != 80 or self.model_size != "standard" or self.temporal_kernel != 5:\n            raise ValueError("Keep the standard 8s kernel-5 model for V9")\n        if self.gain_augment or self.feature_noise_std or self.jitter_frames:\n            raise ValueError("Turn gain, noise and jitter off for this controlled study")\n        if self.low_positive_fraction != .14:\n            raise ValueError("V9 fixes the positive-owner fraction at 14%")\n        if self.owner_draw_cap < 1 or not 0 < self.lr_floor_ratio <= 1:\n            raise ValueError("Check owner_draw_cap and lr_floor_ratio")\n        if not 0 < self.frame_floor_quantile < 1 or self.floor_sample_frames < 100:\n            raise ValueError("Check fit-only floor quantile and sample size")\n        if not np.isfinite(self.floor_absolute_min) or self.floor_absolute_min <= 0:\n            raise ValueError("A positive finite numeric floor is required")\n\n\ndef profile_flags(profile):\n    if profile not in PROFILES:\n        raise ValueError(f"Unknown V9 profile: {profile}")\n    return dict(frame_normalize=profile in (PROFILES[1],PROFILES[3]),\n                drop_residual=profile in (PROFILES[2],PROFILES[3]))\n\n\ndef motion_chunks(handle,start,end,tau_s=3.,chunk_frames=5000):\n    """Unpooled six-channel features at 50 Hz; state crosses I/O chunks only."""\n    if tuple(handle["cir"].shape[::2]) != (3,368) or end <= start:\n        raise ValueError("Expected nonempty CIR (3,T,368)")\n    alpha = math.exp(-1/(50*tau_s))\n    previous = state = None\n    for offset in range(start,end,chunk_frames):\n        x = handle["cir"][:,offset:min(offset+chunk_frames,end)].astype(np.float64)\n        n = x.shape[1]\n        if previous is None:\n            previous = x[:,0].copy()\n            state = (alpha*previous).reshape(-1,1)\n        flat = x.transpose(0,2,1).reshape(-1,n)\n        background,state = base.lfilter([1-alpha],[1,-alpha],flat,axis=1,zi=state)\n        residual = np.abs((flat-background).reshape(3,368,n).transpose(0,2,1))\n        delta = np.abs(np.diff(x,axis=1,prepend=previous[:,None]))\n        previous = x[:,-1].copy()\n        yield offset-start,np.concatenate([residual,delta],axis=0)\n\n\ndef normalize_frames(features,floors):\n    floors = np.asarray(floors,dtype=np.float64)\n    if floors.shape != (6,) or not np.isfinite(floors).all() or np.any(floors<=0):\n        raise ValueError("Six finite positive fit-only amplitude floors required")\n    denominator = np.maximum(np.max(np.abs(features),axis=2,keepdims=True),floors[:,None,None])\n    return features/denominator\n\n\ndef extract_maps_v9(handle,start,end,tau_s=3.,floors=None,chunk_frames=5000):\n    """Optional frame max normalization BEFORE pooling 368->92 and 50->10 Hz."""\n    pending = None\n    features_out,times_out,processed = [],[],0\n    for _,features in motion_chunks(handle,start,end,tau_s,chunk_frames):\n        if floors is not None:\n            features = normalize_frames(features,floors)\n        n = features.shape[1]\n        z = features.reshape(6,n,92,4).mean(axis=3)\n        if pending is not None:\n            z = np.concatenate([pending,z],axis=1)\n        blocks = z.shape[1]//5\n        usable = blocks*5\n        if blocks:\n            features_out.append(z[:,:usable].reshape(6,blocks,5,92).mean(axis=2).astype(np.float32))\n            times_out.append((processed+np.arange(blocks)*5+2)/50)\n            processed += usable\n        pending = z[:,usable:]\n    if pending is not None and pending.shape[1]:\n        features_out.append(pending.mean(axis=1,keepdims=True).astype(np.float32))\n        times_out.append(np.array([(processed+(pending.shape[1]-1)/2)/50]))\n    return np.concatenate(features_out,axis=1),np.concatenate(times_out).astype(np.float32)\n\n\ndef floor_from_samples(samples,cfg):\n    if samples.ndim!=2 or samples.shape[0]!=6 or samples.shape[1]==0:\n        raise ValueError("Need six-channel fit-only frame maxima")\n    if not np.isfinite(samples).all():\n        raise ValueError("Nonfinite amplitude samples")\n    return np.maximum(np.quantile(samples,cfg.frame_floor_quantile,axis=1),cfg.floor_absolute_min)\n\n\ndef fit_amplitude_floors(root,cache,cfg,inner=(),outer=()):\n    """Uniform deterministic time sampling from fit spans only, without labels."""\n    _,_,_,part_map = read_metadata(root)\n    spans,identity = [],[]\n    for path in sorted((root/"streams/train").glob("stream_*.h5")):\n        sid = "train_"+path.stem.split("_")[1]\n        with base.h5py.File(path,"r") as handle:\n            length = handle["cir"].shape[1]\n        for a,b in split_ranges(part_map.get(sid,[]),length,set(inner),set(outer))["fit"]:\n            if b-a<50:\n                continue\n            spans.append((path,a,b))\n            identity.append(dict(path=str(path.resolve()),size=path.stat().st_size,\n                mtime=path.stat().st_mtime_ns,start=a,end=b))\n    config = dict(version=VERSION,spans=identity,tau=cfg.ema_tau_s,quantile=cfg.frame_floor_quantile,\n        limit=cfg.floor_sample_frames,numeric_floor=cfg.floor_absolute_min)\n    digest = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()\n    cache.mkdir(parents=True,exist_ok=True)\n    saved = cache/f"floor_{digest[:28]}.json"\n    if saved.exists():\n        result = read_json(saved)\n        if result["fit_source_sha256"]!=digest:\n            raise ValueError("Amplitude floor cache mismatch")\n        return result\n    total = sum(b-a for _,a,b in spans)\n    if not total:\n        raise ValueError("No fit frames for amplitude floors")\n    selected = np.linspace(0,total-1,min(total,cfg.floor_sample_frames),dtype=np.int64)\n    samples = np.empty((6,len(selected)),np.float64)\n    cursor = 0\n    for path,a,b in spans:\n        lo,hi = np.searchsorted(selected,[cursor,cursor+b-a])\n        local = selected[lo:hi]-cursor\n        if len(local):\n            with base.h5py.File(path,"r") as handle:\n                for offset,features in motion_chunks(handle,a,b,cfg.ema_tau_s):\n                    left,right = np.searchsorted(local,[offset,offset+features.shape[1]])\n                    if right>left:\n                        samples[:,lo+left:lo+right] = np.abs(features[:,local[left:right]-offset]).max(axis=2)\n        cursor += b-a\n    floors = floor_from_samples(samples,cfg)\n    result = dict(version=VERSION,scope="fit-only, labels not used",fit_source_sha256=digest,\n        floors=floors.tolist(),quantile=cfg.frame_floor_quantile,numeric_floor=cfg.floor_absolute_min,\n        fit_raw_frames=total,sampled_frames=len(selected),inner=sorted(inner),outer=sorted(outer),\n        frame_max_quantiles={str(q):np.quantile(samples,q,axis=1).tolist() for q in (0,.1,.5,.9,1)},\n        channels=["residual_r1","residual_r2","residual_r3","delta_r1","delta_r2","delta_r3"])\n    write_json(saved,result)\n    return result\n\n\ndef source_family(path, kind):\n    """Fall/Autre/Empty from the same source family share one split group."""\n    path = str(path).replace("\\\\", "/")\n    if path.startswith("falls/"):\n        match = re.search(r"Test_P\\d+_(\\d+)(?:/|$)", path)\n        corpus = "fall"\n    elif kind == "adl" and path.startswith("activities/"):\n        match = re.search(r"_(\\d+)\\.json$", path)\n        corpus = "adl"\n    else:\n        raise ValueError(f"Unknown source corpus: {path} ({kind})")\n    if match is None:\n        raise ValueError(f"Cannot parse source family: {path}")\n    return f"{corpus}:{int(match.group(1))}"\n\n\ndef read_metadata(root):\n    events = pd.read_csv(root/"streams_events.csv").fillna("")\n    parts = pd.read_csv(root/"streams_composition.csv").fillna("")\n    parts["group"] = [source_family(p, k) for p, k in zip(parts.path, parts.kind)]\n    return events, parts, {s:g.to_dict("records") for s,g in events.groupby("stream_id")}, {\n        s:g.to_dict("records") for s,g in parts.groupby("stream_id")}\n\n\ndef split_ranges(parts, length, inner, outer, public=False):\n    if public:\n        return {"public": [(0, length)]}\n    if set(inner) & set(outer):\n        raise ValueError("Inner and outer source families must be disjoint")\n    masks = {side:np.zeros(length, bool) for side in ("fit", "holdout", "outer")}\n    for part in parts:\n        side = "outer" if part["group"] in outer else "holdout" if part["group"] in inner else "fit"\n        a = max(0, int(round(float(part["t_start"])*50)))\n        b = min(length, int(round(float(part["t_end"])*50)))\n        masks[side][a:b] = True\n    # Crossfades between different split groups are excluded from BOTH sides.\n    return {side:base.boolean_runs(mask & ~np.logical_or.reduce([\n        other for name,other in masks.items() if name != side])) for side,mask in masks.items()}\n\n\ndef build_views(root, cache, cfg, inner=(), outer=(), public=False,profile=PROFILES[0],floors=None):\n    _, parts, event_map, part_map = read_metadata(root)\n    known = set(parts.loc[parts.stream_id.str.startswith("train_"), "group"])\n    if not (set(inner) | set(outer)) <= known:\n        raise ValueError("Unknown source families in split plan")\n    cache.mkdir(parents=True, exist_ok=True)\n    flags = profile_flags(profile)\n    if flags["frame_normalize"] and floors is None:\n        raise ValueError("Normalized input needs locked fit-only floors")\n    folder = root/"streams"/("val" if public else "train")\n    views, raw, manifest, skipped, source_rows = {}, {}, [], [], []\n    for path in sorted(folder.glob("stream_*.h5")):\n        sid = folder.name+"_"+path.stem.split("_")[1]\n        rows = part_map.get(sid, [])\n        with base.h5py.File(path, "r") as handle:\n            n = handle["cir"].shape[1]\n            ranges = split_ranges(rows, n, set(inner), set(outer), public)\n            for side, spans in ranges.items():\n                for number, (a,b) in enumerate(spans):\n                    if b-a < 50:\n                        skipped.append(dict(source_stream=sid, side=side, start=a/50, end=b/50,\n                                            reason="shorter_than_1s"))\n                        continue\n                    key = sid if public else f"{sid}__{side}_{number:04d}"\n                    identity = dict(version=VERSION, file=str(path.resolve()), size=path.stat().st_size,\n                        mtime=path.stat().st_mtime_ns,start=a,end=b,tau=cfg.ema_tau_s,\n                        normalization_floors=list(map(float,floors)) if flags["frame_normalize"] else None)\n                    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:28]\n                    cached = cache/f"{digest}.npz"\n                    if cached.exists():\n                        with np.load(cached) as saved:\n                            x, times = saved["features"], saved["times"]\n                    else:\n                        x, times = extract_maps_v9(handle,a,b,cfg.ema_tau_s,\n                            floors=floors if flags["frame_normalize"] else None)\n                        np.savez_compressed(cached, features=x, times=times)\n                    if not np.isfinite(x).all():\n                        raise ValueError(f"Nonfinite features: {key}")\n                    if flags["drop_residual"]:\n                        x[:3] = 0\n                    present = set()\n                    for part in rows:\n                        hit = base.overlap(a/50,b/50,float(part["t_start"]),float(part["t_end"]))\n                        if hit <= 1e-6:\n                            continue\n                        present.add(part["group"])\n                        source_rows.append(dict(view_id=key, side=side, path=part["path"], kind=part["kind"],\n                            source_family=part["group"], segment_id=part["segment_id"], overlap_s=hit))\n                    expected = set(inner) if side == "holdout" else set(outer) if side == "outer" else set()\n                    if side == "fit" and present & (set(inner)|set(outer)):\n                        raise AssertionError("Held source family reached fit")\n                    if side in ("holdout","outer") and not present <= expected:\n                        raise AssertionError("Unexpected source family in held split")\n                    events = base.local_events(event_map.get(sid, []), a/50, b/50)\n                    views[key] = dict(side=side, duration=(b-a)/50, events=events, groups=sorted(present),\n                        source_stream=sid, source_start_s=a/50, source_end_s=b/50)\n                    raw[key] = (x,times)\n                    manifest.append(dict(view_id=key, source_stream=sid, side=side, source_start_s=a/50,\n                        source_end_s=b/50, duration_s=(b-a)/50, groups=";".join(sorted(present)),\n                        events=len(events), truncated_events=sum(e["truncated"] for e in events)))\n        print("Features:",sid,"| canonical source-family split",flush=True)\n    if not raw:\n        raise ValueError(f"No usable data: {folder}")\n    return views, raw, manifest, skipped, source_rows\n\n\ndef window_interval(ctx, key, start):\n    return start/ctx["cfg"].target_hz, min(ctx["views"][key]["duration"],\n        (start+ctx["cfg"].window_frames)/ctx["cfg"].target_hz)\n\n\ndef candidate_indices(raw, views, cfg, require_inner=True):\n    positive, negative, fit_eval, val_eval = defaultdict(list), [], [], []\n    ctx = dict(cfg=cfg, views=views)\n    for key, view in views.items():\n        if view["side"] not in ("fit","holdout"):\n            continue\n        for start in base.window_starts(len(raw[key][1]),cfg.window_frames,cfg.train_hop_frames):\n            item = (key,int(start))\n            if view["side"] == "holdout":\n                val_eval.append(item)\n                continue\n            fit_eval.append(item)\n            a,b = window_interval(ctx,key,start)\n            hits = [(base.overlap(a,b,e["t_start"],e["t_end"]),e["event_id"])\n                    for e in view["events"] if not base.truth(e["ignore"])]\n            hits = [hit for hit in hits if hit[0]>0]\n            if hits:\n                positive[max(hits)[1]].append(item)\n            else:\n                negative.append(item)\n    if not positive or not negative or (require_inner and not val_eval):\n        raise ValueError("Need fit positive/background candidates and inner evaluation windows")\n    if len(fit_eval)>cfg.train_eval_max_windows:\n        rng = np.random.default_rng(cfg.seed)\n        fit_eval = [fit_eval[i] for i in sorted(rng.choice(len(fit_eval),cfg.train_eval_max_windows,replace=False))]\n    return dict(positive),negative,fit_eval,val_eval\n\n\ndef sample_training(ctx, epoch):\n    """Every V9 profile replays the same V8-B 14% with-replacement sampler."""\n    cfg, profile = ctx["cfg"], ctx["strategy"]\n    if profile not in PROFILES:\n        raise ValueError(f"Unknown profile: {profile}")\n    budget = cfg.batch_size*cfg.steps_per_epoch\n    fraction = cfg.low_positive_fraction\n    npositive = int(math.floor(budget*fraction+.5))\n    owners = sorted(ctx["positives"])\n    if npositive < len(owners):\n        raise ValueError("Fixed positive budget cannot cover every owner; increase common budget or fractions")\n    capacities = {o:min(len(ctx["positives"][o]),cfg.owner_draw_cap) for o in owners}\n    if sum(capacities.values())<npositive:\n        raise ValueError("Common owner capacity too small: lower the relevant positive fraction for all "\n                         "matched profiles or expand the common candidate pool/cap; use a new RUN_TAG")\n    weights = {o:1. for o in owners}\n    rng = np.random.default_rng(cfg.seed+700000+epoch)\n    owner_draws = list(rng.permutation(owners))\n    allocated = Counter(owner_draws)\n    for _ in range(npositive-len(owner_draws)):\n        available = [o for o in owners if allocated[o]<capacities[o]]\n        p = np.array([weights[o] for o in available],dtype=float)\n        o = available[int(rng.choice(len(available),p=p/p.sum()))]\n        allocated[o] += 1\n        owner_draws.append(o)\n    unique = False\n    positive_rng = np.random.default_rng(cfg.seed+710000+epoch)\n    queues = {o:list(positive_rng.permutation(len(ctx["positives"][o]))) for o in owners}\n    rows = []\n    for owner in owner_draws:\n        choices = ctx["positives"][owner]\n        i = queues[owner].pop() if unique else int(positive_rng.integers(len(choices)))\n        key,start = choices[i]\n        rows.append((key,start,owner,"event"))\n    nnegative = budget-npositive\n    if unique and len(ctx["negatives"])<nnegative:\n        raise ValueError("Not enough unique backgrounds; reduce steps_per_epoch for ALL profiles with new RUN_TAG")\n    negative_rng = np.random.default_rng(cfg.seed+720000+epoch)\n    for i in negative_rng.choice(len(ctx["negatives"]),nnegative,replace=not unique):\n        key,start = ctx["negatives"][int(i)]\n        rows.append((key,start,"","background"))\n    order = np.random.default_rng(cfg.seed+730000+epoch).permutation(len(rows))\n    manifest = [dict(epoch=epoch,draw=i,view_id=rows[j][0],start_frame=rows[j][1],owner_event=rows[j][2],\n        kind=rows[j][3],optimizer_step=i//cfg.batch_size+1) for i,j in enumerate(order)]\n    index = [(r["view_id"],r["start_frame"]) for r in manifest]\n    if unique and len(set(index))!=budget:\n        raise AssertionError("Unique sampler produced a duplicate")\n    held = set(ctx["spec"]["inner"]) | set(ctx["spec"]["outer"])\n    if any(ctx["views"][key]["side"]!="fit" or set(ctx["views"][key]["groups"]) & held for key,_ in index):\n        raise AssertionError("Held source family reached actual training samples")\n    return index,manifest\n\n\ndef sampling_audit(ctx,index,manifest,epoch):\n    exposures,summary = v7.exposure_audit(ctx,index,manifest,epoch)\n    target_cache = {key:base.targets_for(ctx["raw"][key][1],ctx["views"][key]["events"]) for key in {k for k,_ in index}}\n    weighted_total,weighted_action,hard_positive,valid = 0.,0.,0,0\n    for row,(key,start) in zip(manifest,index):\n        times = ctx["raw"][key][1][start:start+ctx["cfg"].window_frames]\n        y,w = target_cache[key]\n        weights = w[start:start+len(times)]\n        action = y[0,start:start+len(times)]\n        hard = np.zeros(len(times),bool)\n        for event in ctx["views"][key]["events"]:\n            if not base.truth(event["ignore"]):\n                hard |= (times>=event["t_start"]) & (times<event["t_end"])\n        row.update(valid_frames=len(times),weighted_frames=float(weights.sum()),\n            weighted_action_mass=float((weights*action).sum()),hard_positive_frames=int(hard.sum()),\n            source_families=";".join(ctx["views"][key]["groups"]))\n        weighted_total += row["weighted_frames"]\n        weighted_action += row["weighted_action_mass"]\n        hard_positive += row["hard_positive_frames"]\n        valid += len(times)\n    summary.update(positive_owner_fraction=summary["positive_owner_draws"]/len(index),\n        hard_positive_frame_fraction=hard_positive/max(valid,1),\n        weighted_action_frame_fraction=weighted_action/max(weighted_total,1),\n        sampled_valid_frames=valid,sampled_weighted_frames=weighted_total)\n    return exposures,summary\n\n\ndef lr_schedule(cfg, epochs=None):\n    """Every refit replays an exact prefix of the same predeclared cosine schedule."""\n    count = cfg.max_epochs if epochs is None else epochs\n    if not 1<=count<=cfg.max_epochs:\n        raise ValueError("Refit epochs must be a prefix of the original schedule")\n    return [cfg.lr*(cfg.lr_floor_ratio+(1-cfg.lr_floor_ratio)*.5*(1+math.cos(\n        math.pi*(e-1)/max(cfg.max_epochs-1,1)))) for e in range(1,count+1)]\n\n\ndef physical_summary(features, mask):\n    """CIR descriptors in signal/bin units, not calibrated body kinematics."""\n    result = {}\n    if not np.any(mask):\n        return result\n    energies = []\n    for radar in range(3):\n        x = features[radar,mask].astype(np.float64)\n        dx = features[radar+3,mask].astype(np.float64)\n        profile = x.mean(axis=0)\n        total = profile.sum()\n        centroid = float(profile@np.arange(x.shape[1])/total) if total>1e-20 else None\n        prefix = f"radar{radar+1}"\n        result.update({prefix+"_residual_mean":float(x.mean()), prefix+"_residual_rms":float(np.sqrt((x*x).mean())),\n            prefix+"_delta_mean":float(dx.mean()), prefix+"_centroid_bin":centroid,\n            prefix+"_spread_bin":float(np.sqrt(profile@((np.arange(x.shape[1])-centroid)**2)/total)) if centroid is not None else None,\n            prefix+"_peak_bin":int(profile.argmax()) if total>1e-20 else None})\n        energies.append(float(x.mean()))\n    result["strongest_radar"] = int(np.argmax(energies)+1) if max(energies)>1e-20 else None\n    result["max_radar_residual_mean"] = max(energies)\n    return result\n\n\ndef physics_audit(ctx, sides):\n    rows = []\n    for key,view in ctx["views"].items():\n        if view["side"] not in sides:\n            continue\n        features,times = ctx["raw"][key]\n        # Selection-blind windows cover falls and non-falls; event windows aid diagnosis.\n        intervals = [("regular",a,min(a+4.,view["duration"]),None)\n                     for a in np.arange(0,view["duration"],4.)]\n        for event in view["events"]:\n            a,b = event["t_start"],event["t_end"]\n            intervals.extend((label,left,right,event) for label,left,right in\n                [("pre",max(0,a-2),max(0,a-.5)),("event",a,b),("post",min(view["duration"],b+.5),min(view["duration"],b+2))])\n        for label,a,b,event in intervals:\n            mask = (times>=a)&(times<b)\n            if not mask.any():\n                continue\n            rows.append(dict(view_id=key,side=view["side"],interval_kind=label,start=a,end=b,\n                event_id=event["event_id"] if event else "",dataset=event.get("dataset","") if event else "",\n                fall_type=event.get("fall_type","") if event else "",quality=event.get("quality","") if event else "",\n                source_path=event.get("source_path","") if event else "",**physical_summary(features,mask)))\n    return rows\n\n\ndef prepare_context(root,out,cache,spec,cfg,mode="nested",epochs=None):\n    cfg.validate()\n    base.seed_everything(cfg.seed)\n    out.mkdir(parents=True,exist_ok=True)\n    inner = set(spec["inner"]) if mode in ("nested","confirmation") else set()\n    outer = set(spec["outer"]) if mode!="all_train" else set()\n    floor_report = fit_amplitude_floors(root,cache,cfg,inner,outer)\n    views,raw,manifest,skipped,sources = build_views(root,cache,cfg,inner,outer,\n        profile=spec["strategy"],floors=floor_report["floors"])\n    mean,scale = base.fit_normalizer(raw,views)\n    if profile_flags(spec["strategy"])["drop_residual"]:\n        mean[:3],scale[:3] = 0,1\n    module = v6.evaluator_for(root)\n    config = dict(asdict(cfg),version=VERSION,mode=mode,strategy=spec["strategy"],\n        inner_groups=sorted(inner),outer_groups=sorted(outer),fixed_epochs=epochs,\n        selection_decoder="conservative",checkpoint_rule="inner fixed-threshold event F1",\n        schedule="cosine prefix, horizon=max_epochs",filter_reset_per_view=True,\n        normalizer="fit-only channel z-score",source_family_rule="fall + Autre + Empty share fall session ID",\n        preprocessing=profile_flags(spec["strategy"]),amplitude_floor=floor_report,\n        confirmation_role=spec.get("confirmation_role",""),\n        dropped_channels="residual channels zero before z-score; mean=0 scale=1",\n        ablation_note="No spatial sum, bandpass or frame trimming; normalized maps retain range bins")\n    v6.locked_json(out/"config.json",config)\n    ctx = dict(root=root,out=out,cache_root=cache,views=views,raw=raw,mean=mean,scale=scale,cfg=cfg,\n        spec=dict(spec,inner=sorted(inner),outer=sorted(outer)),strategy=spec["strategy"],mode=mode,\n        full=mode=="all_train",config=config,device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),\n        evaluator=module,evaluate=module.evaluate,validator=module.validate_predictions)\n    ctx["positives"],ctx["negatives"],ctx["fit_eval"],ctx["val_eval"] = candidate_indices(raw,views,cfg,not ctx["full"])\n    for side in ("fit","holdout","outer"):\n        if side=="holdout" and ctx["full"] or side=="outer" and mode=="all_train":\n            continue\n        grounds = base.event_ground(views,[k for k,v in views.items() if v["side"]==side],True)\n        if not any(not e["ignore"] for e in grounds):\n            raise ValueError(f"No scored events in {side}")\n    write_csv(out/"split_manifest.csv",manifest)\n    write_csv(out/"source_family_manifest.csv",sources)\n    write_csv(out/"skipped_intervals.csv",skipped,columns=["source_stream","side","start","end","reason"])\n    np.savez(out/"normalizer_stats.npz",mean=mean,scale=scale)\n    write_json(out/"amplitude_floor_fit_only.json",floor_report)\n    write_json(out/"environment.json",dict(torch=str(torch.__version__),numpy=str(np.__version__),cuda=torch.version.cuda,\n        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))\n    index,manifest_sample = sample_training(ctx,1)\n    _,summary = sampling_audit(ctx,index,manifest_sample,1)\n    sample_signature = hashlib.sha256(json.dumps(index,separators=(",",":")).encode()).hexdigest()\n    sample_identity = dict(mode=mode,fold=spec["fold"],seed=cfg.seed,inner=sorted(inner),outer=sorted(outer),\n        budget=cfg.batch_size*cfg.steps_per_epoch,owner_cap=cfg.owner_draw_cap,hop=cfg.train_hop_frames,\n        metadata={name:v6.file_hash(root/name) for name in ("streams_events.csv","streams_composition.csv")})\n    sample_key = hashlib.sha256(json.dumps(sample_identity,sort_keys=True).encode()).hexdigest()[:24]\n    v6.locked_json(cache/f"paired_samples_{sample_key}.json",dict(epoch1_sha256=sample_signature))\n    write_json(out/"data_audit.json",dict(version=VERSION,split_check="passed for views and actual sampled windows",\n        groups_by_side={side:sorted({g for v in views.values() if v["side"]==side for g in v["groups"]})\n                        for side in ("fit","holdout","outer")},event_owners=len(ctx["positives"]),\n        unique_positive_candidates=sum(map(len,ctx["positives"].values())),negative_candidates=len(ctx["negatives"]),\n        all_fall_types_included=True,epoch1_sampling=summary,epoch1_sample_sha256=sample_signature,\n        preprocessing_context_note="EMA/difference reset at isolated view starts; no frame trimming",\n        physics_scope="Model input before z-score; normalized profiles are dimensionless, C/D have zero residual"))\n    if not (out/"physics_fit_inner.csv").exists():\n        write_csv(out/"physics_fit_inner.csv",physics_audit(ctx,{"fit","holdout"}))\n    return ctx\n\n\ndef load_selected(ctx):\n    payload = torch.load(ctx["out"]/"best_checkpoint_fixed_decoder.pt",map_location="cpu",weights_only=True)\n    if payload["config"] != ctx["config"]:\n        raise ValueError("Checkpoint configuration differs; use a new RUN_TAG")\n    for name in ("mean","scale"):\n        if not np.allclose(payload[name],ctx[name],rtol=1e-5,atol=1e-12):\n            raise ValueError("Normalizer/data differs from checkpoint")\n    model = v5.CNNTCNV5(ctx["cfg"]).to(ctx["device"])\n    model.load_state_dict(payload["model"])\n    ctx.update(model=model,best_epoch=payload["epoch"],training_complete=True)\n    ctx["config"] = dict(ctx["config"],source_sha256=v6.file_hash(ctx["out"]/"best_checkpoint_fixed_decoder.pt"))\n    return ctx\n\n\ndef train_context(ctx,run_mode):\n    out,cfg = ctx["out"],ctx["cfg"]\n    if (out/"training_complete.json").exists():\n        return load_selected(ctx)\n    if run_mode=="export_only":\n        raise ValueError(f"Training incomplete: {out}; use continue")\n    model = v5.CNNTCNV5(cfg).to(ctx["device"])\n    optimizer = torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)\n    history,sampling,exposures,types = [],[],[],[]\n    best_key,best_epoch,stale,best_state = None,0,0,None\n    count = ctx["config"]["fixed_epochs"] if ctx["full"] else cfg.max_epochs\n    schedule = lr_schedule(cfg,count)\n    last = out/"last_checkpoint.pt"\n    if last.exists():\n        saved = torch.load(last,map_location="cpu",weights_only=True)\n        if saved["config"]!=ctx["config"] or saved["schedule"]!=schedule:\n            raise ValueError("Resume configuration/schedule changed")\n        for name in ("mean","scale"):\n            if not np.allclose(saved[name],ctx[name],rtol=1e-5,atol=1e-12):\n                raise ValueError("Resume data/scaler changed")\n        model.load_state_dict(saved["model"])\n        optimizer.load_state_dict(saved["optimizer"])\n        history = saved["history"]\n        best_key,best_epoch,stale,best_state = saved["best_key"],saved["best_epoch"],saved["stale"],saved["best_model"]\n        for filename,destination in [("epoch_sampling_manifest.csv",sampling),("epoch_event_exposure.csv",exposures),\n                                     ("epoch_inner_subtype_recall.csv",types)]:\n            if (out/filename).exists():\n                destination.extend(r for r in pd.read_csv(out/filename).fillna("").to_dict("records")\n                                   if r["epoch"]<=saved["epoch"])\n        if len(history)!=saved["epoch"] or len(sampling)!=len(history)*cfg.steps_per_epoch*cfg.batch_size:\n            raise ValueError("Resume logs incomplete; restore logs from Drive")\n        v7.restore_rng(saved["rng"])\n        print("Resume after epoch",saved["epoch"],flush=True)\n    fit_loader = base.make_loader(ctx["fit_eval"],ctx)\n    val_loader = base.make_loader(ctx["val_eval"],ctx) if not ctx["full"] else None\n    for epoch in range(len(history)+1,count+1):\n        if not ctx["full"] and epoch>cfg.min_epochs and stale>=cfg.patience:\n            break\n        started = time.time()\n        for group in optimizer.param_groups:\n            group["lr"] = schedule[epoch-1]\n        index,manifest = sample_training(ctx,epoch)\n        exposure,summary = sampling_audit(ctx,index,manifest,epoch)\n        augmented = v5.loss_pass(model,base.make_loader(index,ctx,True,epoch),ctx["device"],optimizer)\n        clean = v5.loss_pass(model,fit_loader,ctx["device"])\n        row = dict(epoch=epoch,augmented_train_loss=augmented["loss"],train_loss=clean["loss"],\n            lr=schedule[epoch-1],optimizer_updates=augmented["optimizer_updates"],\n            cumulative_updates=epoch*cfg.steps_per_epoch,\n            sampled_window_sha256=hashlib.sha256(json.dumps(index,separators=(",",":")).encode()).hexdigest(),**summary)\n        row.update({f"train_{k}":clean[k] for k in ("action_loss","start_loss","end_loss")})\n        if not ctx["full"]:\n            val = v5.loss_pass(model,val_loader,ctx["device"])\n            tracks = base.infer_side(model,ctx["raw"],ctx["views"],"holdout",ctx["mean"],ctx["scale"],cfg,ctx["device"])\n            events,_ = v7.predictions(tracks,ctx["views"],"conservative",FIXED)\n            report = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n            frame = base.frame_metrics(tracks,ctx["views"])\n            pooled = v7.pooled_loss(tracks,ctx["views"])\n            row.update(val_loss=val["loss"],loss_gap=val["loss"]-clean["loss"],\n                val_unique_frame_loss=pooled["loss"],val_event_f1_mean=report["ranking_score"]["f1_mean"],\n                val_frame_f1=frame["f1"],val_frame_accuracy=frame["accuracy"],val_frame_balanced_accuracy=frame["balanced_accuracy"],\n                val_false_alarms_per_hour=report["primary"]["false_alarms_per_hour"])\n            row.update({f"val_{k}":val[k] for k in ("action_loss","start_loss","end_loss")})\n            types.extend(dict(r,epoch=epoch) for r in v7.subtype_recall(ctx["evaluator"],events,tracks,ctx["views"]))\n            key = (*base.rank_key(report),-val["loss"])\n            if best_key is None or key>tuple(best_key):\n                best_key,best_epoch,stale,best_state = key,epoch,0,v7.cpu_state(model)\n            else:\n                stale += 1\n        else:\n            best_epoch,best_state = epoch,v7.cpu_state(model)\n        row["seconds"] = time.time()-started\n        history.append(row)\n        sampling.extend(manifest)\n        exposures.extend(exposure)\n        write_csv(out/"training_history_v9.csv",history)\n        write_csv(out/"checkpoint_event_f1.csv",[r for r in history if "val_event_f1_mean" in r],\n            columns=["epoch","val_event_f1_mean","val_false_alarms_per_hour","val_loss"])\n        write_csv(out/"epoch_sampling_manifest.csv",sampling)\n        write_csv(out/"epoch_event_exposure.csv",exposures)\n        write_csv(out/"epoch_inner_subtype_recall.csv",types,\n            columns=["dataset","fall_type","source_group","tp","fn","n_events","recall","epoch"])\n        save_checkpoint(last,dict(model=v7.cpu_state(model),optimizer=optimizer.state_dict(),history=history,\n            config=ctx["config"],mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),epoch=epoch,\n            best_key=best_key,best_epoch=best_epoch,stale=stale,best_model=best_state,rng=v7.rng_state(),schedule=schedule))\n        save_checkpoint(out/"best_checkpoint_fixed_decoder.pt",dict(model=best_state,config=ctx["config"],\n            mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),epoch=best_epoch))\n        print(f"{ctx[\'spec\'][\'name\']} epoch {epoch:02d} train={clean[\'loss\']:.4f} "\n            +(f"inner={row[\'val_loss\']:.4f} F1={row[\'val_event_f1_mean\']:.4f} " if not ctx["full"] else "fixed refit ")\n            +f"positive={summary[\'positive_owner_fraction\']:.2%} unique={summary[\'unique_drawn_windows\']}/{len(index)}",flush=True)\n    # Rebuild the best checkpoint from the committed last state after an interrupted save.\n    if best_state is None:\n        raise ValueError("No completed training epoch")\n    save_checkpoint(out/"best_checkpoint_fixed_decoder.pt",dict(model=best_state,config=ctx["config"],\n        mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),epoch=best_epoch))\n    write_json(out/"training_complete.json",dict(version=VERSION,best_epoch=best_epoch,trained_epochs=len(history),\n        reason="fixed exact-prefix refit" if ctx["full"] else "inner event F1 early stopping or max_epochs"))\n    return load_selected(ctx)\n\n\ndef select_decoders(ctx):\n    tracks = v6.tracks_cached(ctx,"inner")\n    rows,fixed,parameters = [],[],{}\n    for family in DECODERS:\n        events,_ = v7.predictions(tracks,ctx["views"],family,FIXED)\n        r = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n        fixed.append(dict(decoder=family,**r["ranking_score"]))\n        best_key,best = None,None\n        for high in (.4,.5,.6,.7,.8):\n            for low in (.1,.2,.3):\n                p = dict(FIXED,high=high,low=low)\n                events,_ = v7.predictions(tracks,ctx["views"],family,p)\n                report = base.event_report(ctx["evaluate"],events,ctx["views"],tracks)\n                rows.append(dict(decoder=family,high=high,low=low,**report["ranking_score"]))\n                if best_key is None or base.rank_key(report)>best_key:\n                    best_key,best = base.rank_key(report),p\n        parameters[family] = best\n    write_csv(ctx["out"]/"cnn_tcn_v9_decoder_grid.csv",rows)\n    selection = dict(version=VERSION,checkpoint_sha256=ctx["config"]["source_sha256"],best_epoch=ctx["best_epoch"],\n        parameters=parameters,fixed_comparison=fixed,scope="inner only",selected_decoder="conservative")\n    v6.locked_json(ctx["out"]/"decoder_selection_locked.json",selection)\n    return selection\n\n\ndef export_report(ctx,split,tracks,views,family,parameters):\n    decoding_family = "conservative" if family=="conservative_fixed" else family\n    events,audit = v7.predictions(tracks,views,decoding_family,parameters)\n    prefix = f"cnn_tcn_v9_{family}_{split}"\n    out = ctx["out"]\n    path = out/f"{prefix}_submission.csv"\n    write_csv(path,events,columns=["stream_id","t_start","t_end","score"])\n    checked = ctx["validator"](path,{k:views[k]["duration"] for k in tracks})\n    frame = base.frame_metrics(tracks,views)\n    frame["all_background_accuracy"] = (frame["tn"]+frame["fp"])/max(frame["n_frames"],1)\n    report = dict(version=VERSION,split=split,decoder=family,parameters=parameters,strategy=ctx["strategy"],\n        best_epoch=ctx["best_epoch"],training_complete=True,mode=ctx["mode"],\n        fold=ctx["spec"]["fold"],seed=int(ctx["cfg"].seed),\n        confirmation_role=ctx["spec"].get("confirmation_role",""),\n        exposure_hours=sum(views[k]["duration"] for k in tracks)/3600,\n        ignore_flagged=base.event_report(ctx["evaluate"],checked,views,tracks),\n        all_events=base.event_report(ctx["evaluate"],checked,views,tracks,False),\n        frame_metrics=frame,decoded_frame_metrics=v6.decoded_frame_metrics(tracks,views,checked),\n        pooled_probability_loss=v7.pooled_loss(tracks,views),note="Offline source-family CV; not full LOSO or independent test")\n    write_json(out/f"{prefix}_report.json",report)\n    write_csv(out/f"{prefix}_boundary_audit.csv",audit,columns=None if audit else ["stream_id","method"])\n    errors = v6.errors_v6(ctx["evaluator"],checked,views,tracks)\n    for row in errors:\n        if "gt_start" in row:\n            row["gt_duration"] = row["gt_end"]-row["gt_start"]\n        if "pred_start" in row:\n            row["pred_duration"] = row["pred_end"]-row["pred_start"]\n        if row["kind"]=="TP":\n            row["duration_error"] = row["pred_duration"]-row["gt_duration"]\n    write_csv(out/f"{prefix}_event_errors.csv",errors,columns=["stream_id","kind","gt_start","gt_end","pred_start",\n        "pred_end","best_tiou","start_error","end_error","gt_duration","pred_duration","duration_error"])\n    write_csv(out/f"{prefix}_subtype_recall.csv",v7.subtype_recall(ctx["evaluator"],checked,tracks,views),\n        columns=["dataset","fall_type","source_group","tp","fn","n_events","recall"])\n    event_rows = []\n    for key,(times,probs) in tracks.items():\n        smooth = base.smooth(probs[0],5)\n        for e in views[key]["events"]:\n            inside = (times>=e["t_start"]) & (times<e["t_end"])\n            if not inside.any():\n                continue\n            match = next((r for r in errors if r["stream_id"]==key and "gt_start" in r\n                and abs(r["gt_start"]-e["t_start"])<1e-6 and abs(r["gt_end"]-e["t_end"])<1e-6),None)\n            event_rows.append(dict(stream_id=key,event_id=e["event_id"],dataset=e.get("dataset",""),\n                fall_type=e.get("fall_type",""),source_path=e["source_path"],quality=e["quality"],\n                ignore=base.truth(e["ignore"]),needs_review=base.truth(e["needs_review"]),\n                kind=match["kind"] if match else "ignored",t_start=e["t_start"],t_end=e["t_end"],\n                peak_action=float(probs[0,inside].max()),peak_action_smooth=float(smooth[inside].max())))\n    write_csv(out/f"{prefix}_event_probabilities.csv",event_rows)\n    return report\n\n\ndef export_context(ctx,selection,splits,public=False):\n    out = ctx["out"]\n    reports,fixed_rows = {},[]\n    for split in splits:\n        views,raw = ctx["views"],ctx["raw"]\n        if split=="public":\n            if not public:\n                continue\n            # Public inference is on entire streams, with no composition cuts or labels used as inputs.\n            views,raw,manifest,_,_ = build_views(ctx["root"],ctx["cache_root"],ctx["cfg"],public=True,\n                profile=ctx["strategy"],floors=ctx["config"]["amplitude_floor"]["floors"])\n            write_csv(out/"public_manifest.csv",manifest)\n            tracks = v6.tracks_cached(ctx,split,views,raw)\n        else:\n            tracks = v6.tracks_cached(ctx,split)\n        if not tracks:\n            continue\n        for family in DECODERS:\n            r = export_report(ctx,split,tracks,views,family,selection["parameters"][family])\n            reports[f"{family}:{split}"] = r\n            events,_ = v7.predictions(tracks,views,family,FIXED)\n            rfixed = base.event_report(ctx["evaluate"],events,views,tracks)\n            fixed_rows.append(dict(split=split,decoder=family,**rfixed["ranking_score"]))\n        fixed_report = export_report(ctx,split,tracks,views,"conservative_fixed",FIXED)\n        reports[f"conservative_fixed:{split}"] = fixed_report\n        for suffix in ("submission.csv","report.json"):\n            shutil.copyfile(out/f"cnn_tcn_v9_conservative_{split}_{suffix}",out/f"cnn_tcn_v9_{split}_{suffix}")\n        if split=="public":\n            del raw\n    write_csv(out/"fixed_threshold_decoder_comparison_v9.csv",fixed_rows)\n    write_csv(out/"hard_negative_manifest.csv",[],columns=["stream_id","start","score"])\n    save_checkpoint(out/"cnn_tcn_v9_selected_model.pt",dict(model=v7.cpu_state(ctx["model"]),epoch=ctx["best_epoch"],\n        config=ctx["config"],mean=ctx["mean"].tolist(),scale=ctx["scale"].tolist(),decoder_family="conservative",\n        decoder=selection["parameters"]["conservative"]))\n    write_json(out/"report_v9.json",dict(version=VERSION,strategy=ctx["strategy"],selected_decoder="conservative",\n        mode=ctx["mode"],best_epoch=ctx["best_epoch"],reports=reports))\n    if "outer" in splits:\n        write_csv(out/"physics_outer_after_lock.csv",physics_audit(ctx,{"outer"}))\n    write_json(out/"run_complete_v9.json",dict(version=VERSION,splits=list(splits),public=public))\n\n\n\n\ndef choose_suite(specs,selections,run_root,cfg):\n    rows = []\n    for spec,selection in zip(specs,selections):\n        score = next(r["f1_mean"] for r in selection["fixed_comparison"] if r["decoder"]=="conservative")\n        rows.append(dict(strategy=spec["strategy"],fold=spec["fold"],seed=spec["seed"],score=score))\n    frame = pd.DataFrame(rows)\n    ranking = frame.groupby("strategy",as_index=False,sort=False).score.mean().sort_values("score",ascending=False,kind="stable")\n    strategy = str(ranking.iloc[0].strategy)\n    fold_choice = {}\n    for fold,group in frame.groupby("fold",sort=False):\n        ranked = group.groupby("strategy",as_index=False,sort=False).score.mean().sort_values("score",ascending=False,kind="stable")\n        fold_choice[fold] = str(ranked.iloc[0].strategy)\n    relevant = [(s,d) for s,d in zip(specs,selections) if s["strategy"]==strategy]\n    epochs = max(1,int(math.floor(float(np.median([d["best_epoch"] for _,d in relevant]))+.5)))\n    parameters = {}\n    grids = pd.concat([pd.read_csv(run_root/s["name"]/"cnn_tcn_v9_decoder_grid.csv") for s,_ in relevant],ignore_index=True)\n    for family in DECODERS:\n        group = grids[grids.decoder==family].groupby(["high","low"],as_index=False,sort=True).f1_mean.mean()\n        best = group.sort_values("f1_mean",ascending=False,kind="stable").iloc[0]\n        parameters[family] = dict(FIXED,high=float(best.high),low=float(best.low))\n    return dict(version=VERSION,strategy=strategy,selected_decoder="conservative",ranking=ranking.to_dict("records"),\n        outer_selection_by_fold=fold_choice,refit_epochs=epochs,refit_lr_schedule=lr_schedule(cfg,epochs),\n        refit_parameters=parameters,refit_seed=int(specs[0]["seed"]),\n        criterion="mean INNER fixed-threshold conservative F1; each outer fold uses only its own inner",\n        final_threshold_rule="mean INNER grid F1 across chosen profile; stable ties prefer lower high then low",\n        refit_rule="Optional all-train refit; exact schedule prefix, median screening checkpoint epoch; refit may degrade",\n        final_selection_scope="Screening INNER only; confirmation reports reproducibility without changing selection")\n\n\n\n\ndef collect_results(run_root):\n    rows = []\n    for path in sorted(run_root.rglob("cnn_tcn_v9_*_report.json")):\n        report = read_json(path)\n        if path.name!=f"cnn_tcn_v9_{report[\'decoder\']}_{report[\'split\']}_report.json":\n            continue\n        row = v6.report_row(report["split"],report["decoder"],report)\n        row.update(run=path.parent.relative_to(run_root).as_posix(),strategy=report["strategy"],\n            fold=report["fold"],seed=report["seed"],mode=report["mode"],report_path=str(path),\n            pooled_loss=report["pooled_probability_loss"]["loss"],\n            all_background_accuracy=report["frame_metrics"]["all_background_accuracy"],\n            confirmation_role=report.get("confirmation_role",""))\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef selected_outer_rows(table,lock):\n    chosen = []\n    for fold,strategy in lock["outer_selection_by_fold"].items():\n        subset = table[(table.fold==fold)&(table.strategy==strategy)&(table.split=="outer")&\n            (table.decoder=="conservative")&(table["mode"]=="nested")].copy()\n        subset["source_strategy"] = subset.strategy\n        subset["strategy"] = "nested_inner_selected"\n        chosen.append(subset)\n    return pd.concat(chosen,ignore_index=True)\n\n\ndef confirmation_specs(specs,selections,seed):\n    """Nominate one non-baseline alternative using ONLY the same fold\'s inner."""\n    rows = []\n    for fold in dict.fromkeys(s["fold"] for s in specs):\n        candidates = [(s,d) for s,d in zip(specs,selections)\n                      if s["fold"]==fold and s["strategy"]!=PROFILES[0]]\n        if not candidates:\n            continue\n        spec,_ = max(candidates,key=lambda item:next(\n            r["f1_mean"] for r in item[1]["fixed_comparison"] if r["decoder"]=="conservative"))\n        for strategy,role in ((PROFILES[0],"baseline"),(spec["strategy"],"inner_nominated_alternative")):\n            rows.append(dict(spec,name=f"confirmation/{strategy}/{fold}/seed_{seed}",\n                strategy=strategy,seed=int(seed),confirmation_role=role))\n    return rows\n\n\ndef factorial_effects(table):\n    """Paired F1 differences in percentage points; no independence/significance claim."""\n    rows = []\n    screening = table[(table["mode"]=="nested") & (table.split=="outer")]\n    for (fold,seed,decoder),group in screening.groupby(["fold","seed","decoder"],sort=False):\n        score = group.set_index("strategy").event_f1_mean.to_dict()\n        if not set(PROFILES)<=set(score):\n            continue\n        a,b,c,d = [score[p] for p in PROFILES]\n        rows.append(dict(fold=fold,seed=int(seed),decoder=decoder,\n            normalization_with_residual_pp=100*(b-a),normalization_delta_only_pp=100*(d-c),\n            remove_residual_without_norm_pp=100*(c-a),remove_residual_with_norm_pp=100*(d-b),\n            normalization_average_pp=50*((b-a)+(d-c)),\n            remove_residual_average_pp=50*((c-a)+(d-b)),\n            interaction_pp=100*((d-c)-(b-a))))\n    return rows\n\n\ndef save_summary(run_root,lock):\n    table = collect_results(run_root)\n    table.to_csv(run_root/"experiment_comparison_v9.csv",index=False)\n    nested = table[table["mode"]=="nested"]\n    v7.aggregate_results(nested).to_csv(run_root/"experiment_aggregate_v9.csv",index=False)\n    chosen = selected_outer_rows(table,lock)\n    chosen.to_csv(run_root/"nested_selected_outer_v9.csv",index=False)\n    v7.aggregate_results(chosen).to_csv(run_root/"nested_selected_outer_aggregate_v9.csv",index=False)\n    effects = factorial_effects(table)\n    write_csv(run_root/"factorial_effects_v9.csv",effects,columns=[\n        "fold","seed","decoder","normalization_with_residual_pp","normalization_delta_only_pp",\n        "remove_residual_without_norm_pp","remove_residual_with_norm_pp",\n        "normalization_average_pp","remove_residual_average_pp","interaction_pp"])\n    paired = []\n    for (mode,fold,seed,split,decoder),group in table.groupby(["mode","fold","seed","split","decoder"],sort=False):\n        reference = group[group.strategy==PROFILES[0]]\n        if not len(reference):\n            continue\n        ref = reference.iloc[0]\n        for _,row in group.iterrows():\n            if row.strategy==PROFILES[0]:\n                continue\n            paired.append(dict(mode=mode,fold=fold,seed=int(seed),split=split,decoder=decoder,\n                strategy=row.strategy,baseline_f1=ref.event_f1_mean,variant_f1=row.event_f1_mean,\n                delta_f1_pp=100*(row.event_f1_mean-ref.event_f1_mean),\n                delta_f1_05_pp=100*(row.event_f1_05-ref.event_f1_05),\n                delta_fp=int(row.fp-ref.fp),delta_fn=int(row.fn-ref.fn),\n                delta_false_alarms_per_hour=row.false_alarms_per_hour-ref.false_alarms_per_hour))\n    write_csv(run_root/"paired_baseline_comparison_v9.csv",paired)\n    sampling_checks = []\n    runs = table[table["mode"].isin(["nested","confirmation"])].drop_duplicates("run")\n    for (mode,fold,seed),group in runs.groupby(["mode","fold","seed"],sort=False):\n        histories = [pd.read_csv(run_root/name/"training_history_v9.csv") for name in group.run]\n        epochs = sorted(set.intersection(*[set(frame.epoch) for frame in histories]))\n        for epoch in epochs:\n            hashes = {frame.loc[frame.epoch==epoch,"sampled_window_sha256"].iloc[0] for frame in histories}\n            if len(hashes)!=1:\n                raise AssertionError("Paired sample order differs between preprocessing profiles")\n        sampling_checks.append(dict(mode=mode,fold=fold,seed=int(seed),runs=len(group),\n            matched_common_epochs=epochs,passed=True))\n    write_json(run_root/"paired_sampling_verification_v9.json",sampling_checks)\n    confirmed = table[table["mode"]=="confirmation"]\n    if len(confirmed):\n        confirmation_policy = confirmed.copy()\n        confirmation_policy["source_strategy"] = confirmation_policy.strategy\n        confirmation_policy["strategy"] = confirmation_policy.confirmation_role\n        v7.aggregate_results(confirmation_policy).to_csv(run_root/"confirmation_aggregate_v9.csv",index=False)\n    # Event-level probabilities remain available even when no interval was proposed.\n    diagnostics = []\n    for path in sorted(run_root.rglob("cnn_tcn_v9_conservative_fixed_*_event_probabilities.csv")):\n        frame = pd.read_csv(path)\n        if frame.empty:\n            continue\n        report = read_json(path.with_name(path.name.replace("_event_probabilities.csv","_report.json")))\n        for name in ("strategy","mode","fold","seed","split"):\n            frame[name] = report[name]\n        diagnostics.extend(frame.to_dict("records"))\n    write_csv(run_root/"event_probability_comparison_v9.csv",diagnostics)\n    write_json(run_root/"study_limitations_v9.json",dict(\n        primary_comparison="Screening outer, conservative_fixed decoder, per-fold paired contrasts",\n        secondary_comparison="Inner-tuned conservative decoder; accuracy is frame-level",\n        confirmation="One non-baseline alternative per fold chosen only from that fold\'s screening inner",\n        uncertainty="One screening seed; confirmation has two profiles per fold, not full factorial replication",\n        historical="Outer and public have been inspected in earlier versions; exploratory development scores",\n        final_refit="Optional fixed-epoch all-train recipe selected by screening inner; public never selects it",\n        filter="EMA resets per split view, no trimming; normalization does not improve physical SNR"))\n    return table\n\n\ndef run_suite(root,run_root,cache,cfg,folds,seed=2027,confirmation_seed=42,\n              run_mode="continue",final_refit=True,public=True,implementation_hash=""):\n    root,run_root,cache = Path(root),Path(run_root),Path(cache)\n    cfg.validate()\n    if run_mode not in ("train","continue","export_only"):\n        raise ValueError("RUN_MODE must be train, continue or export_only")\n    if confirmation_seed is not None and int(confirmation_seed)==int(seed):\n        raise ValueError("Confirmation must use a different seed")\n    if public and not final_refit:\n        raise ValueError("Public requires final all-train refit")\n    specs = []\n    for profile in PROFILES:\n        for fold,inner,outer in folds:\n            if not inner or not outer or set(inner)&set(outer) or not re.fullmatch(r"[A-Za-z0-9_-]+",fold):\n                raise ValueError("Check fold name and disjoint inner/outer families")\n            specs.append(dict(name=f"{profile}/{fold}/seed_{int(seed)}",strategy=profile,\n                fold=fold,inner=sorted(inner),outer=sorted(outer),seed=int(seed)))\n    if not specs or len({s["name"] for s in specs})!=len(specs):\n        raise ValueError("Empty or duplicate run plan")\n    run_root.mkdir(parents=True,exist_ok=True)\n    cache.mkdir(parents=True,exist_ok=True)\n    plan = dict(version=VERSION,specs=specs,config=asdict(cfg),confirmation_seed=confirmation_seed,\n        final_refit=final_refit,public=public,implementation_hash=implementation_hash,\n        data_root=str(root.resolve()),\n        metadata_hashes={p:v6.file_hash(root/p) for p in ("streams_events.csv","streams_composition.csv")},\n        streams=[dict(path=p.relative_to(root).as_posix(),size=p.stat().st_size,mtime=p.stat().st_mtime_ns)\n                 for p in sorted((root/"streams").glob("*/*.h5"))],\n        paired_sampler="V8-B 14%, same seed/candidate pool and epoch order across preprocessing profiles")\n    if run_mode=="train" and (run_root/"suite_plan_v9.json").exists():\n        raise FileExistsError("Existing run: use continue or a new RUN_TAG")\n    v6.locked_json(run_root/"suite_plan_v9.json",plan)\n    _,parts,_,_ = read_metadata(root)\n    write_csv(run_root/"canonical_source_groups_v9.csv",parts.to_dict("records"))\n\n    def fit_inner(spec,mode):\n        out = run_root/spec["name"]\n        if (out/"decoder_selection_locked.json").exists():\n            selection = read_json(out/"decoder_selection_locked.json")\n            if selection["checkpoint_sha256"]!=v6.file_hash(out/"best_checkpoint_fixed_decoder.pt"):\n                raise ValueError("Selected checkpoint changed after inner lock")\n            return selection\n        ctx = prepare_context(root,out,cache,spec,replace(cfg,seed=spec["seed"]),mode=mode)\n        ctx = train_context(ctx,run_mode)\n        selection = select_decoders(ctx)\n        del ctx\n        v5.release_memory()\n        return selection\n\n    # Every checkpoint and recipe decision finishes before outer/public scoring.\n    selections = [fit_inner(spec,"nested") for spec in specs]\n    lock = choose_suite(specs,selections,run_root,cfg)\n    extra_specs = confirmation_specs(specs,selections,confirmation_seed) if confirmation_seed is not None else []\n    v6.locked_json(run_root/"confirmation_plan_locked_v9.json",dict(\n        specs=extra_specs,scope="Own-fold INNER only; no outer scores",\n        screening_selected=lock["strategy"],screening_selections=selections))\n    extra_selections = [fit_inner(spec,"confirmation") for spec in extra_specs]\n    lock["confirmation_policy"] = "Own-fold inner-nominated non-baseline vs A; does not change screening selection"\n    v6.locked_json(run_root/"selection_locked_before_outer_public.json",lock)\n    print("INNER choices locked. Final screening recipe:",lock["strategy"],flush=True)\n    all_specs = specs+extra_specs\n    all_selections = selections+extra_selections\n    for spec,selection in zip(all_specs,all_selections):\n        out = run_root/spec["name"]\n        if run_mode!="export_only" and (out/"run_complete_v9.json").exists():\n            continue\n        mode = "confirmation" if spec.get("confirmation_role") else "nested"\n        ctx = prepare_context(root,out,cache,spec,replace(cfg,seed=spec["seed"]),mode=mode)\n        ctx = load_selected(ctx)\n        export_context(ctx,selection,("train","inner","outer"))\n        del ctx\n        v5.release_memory()\n    if final_refit:\n        out = run_root/"final_all_train"\n        if run_mode=="export_only" or not (out/"run_complete_v9.json").exists():\n            spec = dict(name="final_all_train",strategy=lock["strategy"],fold="all",\n                inner=[],outer=[],seed=int(seed))\n            ctx = prepare_context(root,out,cache,spec,replace(cfg,seed=int(seed)),\n                mode="all_train",epochs=lock["refit_epochs"])\n            ctx = train_context(ctx,run_mode)\n            selection = dict(version=VERSION,parameters=lock["refit_parameters"],\n                best_epoch=ctx["best_epoch"],checkpoint_sha256=ctx["config"]["source_sha256"],\n                scope="Screening INNER recipe; no outer/public tuning")\n            v6.locked_json(out/"decoder_selection_locked.json",selection)\n            export_context(ctx,selection,("train","public") if public else ("train",),public=public)\n            del ctx\n            v5.release_memory()\n        for split in ("train","public"):\n            for suffix in ("submission.csv","report.json"):\n                path = out/f"cnn_tcn_v9_{split}_{suffix}"\n                if path.exists():\n                    shutil.copyfile(path,run_root/path.name)\n        shutil.copyfile(out/"cnn_tcn_v9_selected_model.pt",run_root/"cnn_tcn_v9_selected_model.pt")\n    table = save_summary(run_root,lock)\n    write_json(run_root/"suite_runs_v9.json",dict(version=VERSION,specs=all_specs,selected=lock,\n        confirmation_seed=confirmation_seed,final_refit=final_refit,public=public))\n    return table,lock\n\n\ndef plots_and_archive(run_root,show_all=True,include_probabilities=True):\n    """Read saved results only. Safe to rerun after training without radar access."""\n    import matplotlib.pyplot as plt\n    from IPython.display import display\n    run_root = Path(run_root)\n    index = read_json(run_root/"suite_runs_v9.json")\n    table = save_summary(run_root,index["selected"])\n    columns = ["event_f1_mean","event_f1_03","event_f1_05","model_frame_f1","model_frame_accuracy",\n        "model_balanced_accuracy","all_background_accuracy","decoded_frame_f1"]\n    shown_profiles = set()\n    for run,rows in table.groupby("run",sort=False):\n        out = run_root/run\n        r = read_json(out/"report_v9.json")\n        display_this = show_all or run=="final_all_train" or (r["mode"]=="nested" and r["strategy"] not in shown_profiles)\n        if display_this and r["mode"]=="nested":\n            shown_profiles.add(r["strategy"])\n        history = pd.read_csv(out/"training_history_v9.csv")\n        fig,axes = plt.subplots(1,2,figsize=(12,3.8))\n        axes[0].plot(history.epoch,history.train_loss,"o-",ms=3,label="Train loss (clean eval)")\n        axes[0].plot(history.epoch,history.augmented_train_loss,alpha=.35,label="Train loss (dropout on)")\n        if "val_loss" in history:\n            axes[0].plot(history.epoch,history.val_loss,"o-",ms=3,label="Inner validation loss")\n            axes[1].plot(history.epoch,history.val_event_f1_mean,"o-",label="Inner event F1 (fixed conservative)")\n            axes[1].plot(history.epoch,history.val_frame_f1,label="Inner frame F1")\n            axes[1].set_ylim(0,1.02)\n        else:\n            axes[1].plot(history.epoch,history.positive_owner_fraction,label="Positive-owner window fraction")\n            axes[1].plot(history.epoch,history.hard_positive_frame_fraction,label="Hard positive frame fraction")\n            axes[1].set_title("Refit: fixed epochs, no validation stopping")\n        axes[0].axvline(r["best_epoch"],color="gray",ls=":",label="Selected/final epoch")\n        axes[0].set_ylabel("Weighted window BCE")\n        for ax in axes:\n            ax.set_xlabel("Epoch");ax.grid(alpha=.2);ax.legend(fontsize=8)\n        fig.suptitle(run,fontsize=10)\n        fig.tight_layout();fig.savefig(out/"training_curves_v9.png",dpi=150)\n        if display_this:\n            plt.show()\n        plt.close(fig)\n        matrix = rows.set_index(["split","decoder"])[columns]\n        matrix.to_csv(out/"metrics_matrix_v9.csv")\n        fig,ax = plt.subplots(figsize=(12,max(2.8,len(matrix)*.38+2)))\n        vals = matrix.to_numpy(dtype=float)*100\n        heat = ax.imshow(np.ma.masked_invalid(vals),vmin=0,vmax=100,cmap="YlGnBu",aspect="auto")\n        ax.set_xticks(range(len(columns)),columns,rotation=22,ha="right")\n        ax.set_yticks(range(len(matrix)),[f"{a}/{b}" for a,b in matrix.index])\n        for a in range(len(matrix)):\n            for b in range(len(columns)):\n                ax.text(b,a,f"{vals[a,b]:.1f}",ha="center",va="center",fontsize=8,\n                    color="white" if vals[a,b]>65 else "black")\n        ax.set_title(run+" | event F1 and frame metrics (%)",fontsize=10)\n        fig.colorbar(heat,ax=ax);fig.tight_layout();fig.savefig(out/"f1_accuracy_matrix_v9.png",dpi=150)\n        if display_this:\n            print(run);display(matrix.style.format("{:.2%}",na_rep="N/A"));plt.show()\n        plt.close(fig)\n        selected_reports = [report for key,report in r["reports"].items() if key.startswith("conservative:")]\n        fig,axes = plt.subplots(1,len(selected_reports),figsize=(3.6*len(selected_reports),3.5),squeeze=False)\n        for ax,report in zip(axes[0],selected_reports):\n            m = report["frame_metrics"]\n            counts = np.array([[m["tn"],m["fp"]],[m["fn"],m["tp"]]])\n            ax.imshow(counts,cmap="Blues")\n            ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Non-fall","Fall"],yticklabels=["Non-fall","Fall"],\n                xlabel="Predicted",ylabel="Actual",title=report["split"])\n            for a in range(2):\n                for b in range(2):\n                    ax.text(b,a,f"{counts[a,b]:,}",ha="center",va="center",\n                        color="white" if counts[a,b]>counts.max()/2 else "black")\n        fig.tight_layout();fig.savefig(out/"frame_confusion_matrices_v9.png",dpi=150)\n        if run=="final_all_train":\n            plt.show()\n        plt.close(fig)\n        subtype = []\n        for report in selected_reports:\n            path = out/f"cnn_tcn_v9_conservative_{report[\'split\']}_subtype_recall.csv"\n            sub = pd.read_csv(path).groupby(["dataset","fall_type"],as_index=False)[["tp","fn","n_events"]].sum()\n            sub["recall"] = sub.tp/sub.n_events\n            sub["split"] = report["split"]\n            subtype.extend(sub.to_dict("records"))\n        write_csv(out/"selected_subtype_recall_v9.csv",subtype)\n        if run=="final_all_train":\n            print("Recall by dataset/type (check event counts):");display(pd.DataFrame(subtype))\n        # Deterministic example: first scored event, never a hand-picked public success.\n        split = next((s for s in ("public","outer","inner","train") if (out/f"probabilities_{s}_index.json").exists()),None)\n        if split is not None:\n            metadata = read_json(out/f"probabilities_{split}_index.json")\n            first = next(((i,key,view,event) for i,(key,view) in enumerate(sorted(metadata["views"].items()))\n                for event in view["events"] if not base.truth(event["ignore"]) and str(event["quality"]).lower()!="low"\n                and not base.truth(event["needs_review"])),None)\n            if first is not None:\n                i,key,view,event = first\n                with np.load(out/f"probabilities_{split}.npz") as saved:\n                    times,probs = saved[f"t{i}"],saved[f"p{i}"]\n                mask = (times>=event["t_start"]-4)&(times<=event["t_end"]+4)\n                fig,ax = plt.subplots(figsize=(10,3.6))\n                for channel,label in enumerate(("Action","Start","End")):\n                    ax.plot(times[mask],probs[channel,mask],label=label)\n                ax.axvspan(event["t_start"],event["t_end"],color="green",alpha=.15,label="Ground truth")\n                parameters = read_json(out/"decoder_selection_locked.json")["parameters"]["conservative"]\n                decoded,_ = v7.predictions({key:(times,probs)},{key:view},"conservative",parameters)\n                for number,e in enumerate(e for e in decoded if e["t_end"]>event["t_start"]-4 and e["t_start"]<event["t_end"]+4):\n                    ax.plot([e["t_start"],e["t_end"]],[-.07,-.07],lw=4,color="red",label="Predicted interval" if number==0 else None)\n                ax.set(xlabel="Seconds",ylabel="Probability",title=f"{split}: first scored event")\n                ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/"probability_timeline_v9.png",dpi=150)\n                if run=="final_all_train":\n                    plt.show()\n                plt.close(fig)\n    display(pd.read_csv(run_root/"experiment_aggregate_v9.csv").drop(columns="note").round(4))\n    print("Nested per-fold inner-selected procedure:")\n    display(pd.read_csv(run_root/"nested_selected_outer_aggregate_v9.csv").drop(columns="note").round(4))\n    print("PRIMARY: same fixed thresholds across profiles. Differences are percentage points:")\n    paired = pd.read_csv(run_root/"paired_baseline_comparison_v9.csv")\n    display(paired[(paired.split=="outer")&(paired.decoder=="conservative_fixed")].round(4))\n    effects = pd.read_csv(run_root/"factorial_effects_v9.csv")\n    display(effects[effects.decoder=="conservative_fixed"].round(3))\n    confirmation_path = run_root/"confirmation_aggregate_v9.csv"\n    if confirmation_path.exists():\n        print("Second seed: baseline vs own-inner-nominated alternative; check each fold too.")\n        display(pd.read_csv(confirmation_path).drop(columns="note").round(4))\n    primary = table[(table["mode"]=="nested")&(table.split=="outer")&\n                    (table.decoder=="conservative_fixed")]\n    if len(primary):\n        matrix = primary.pivot(index="fold",columns="strategy",values="event_f1_mean").reindex(columns=PROFILES)*100\n        fig,ax = plt.subplots(figsize=(10,3.8))\n        heat = ax.imshow(matrix.to_numpy(),vmin=0,vmax=100,cmap="YlGnBu",aspect="auto")\n        ax.set_xticks(range(len(matrix.columns)),matrix.columns,rotation=12,ha="right",fontsize=8)\n        ax.set_yticks(range(len(matrix.index)),matrix.index,fontsize=8)\n        for a in range(len(matrix)):\n            for b in range(len(matrix.columns)):\n                value = matrix.iloc[a,b]\n                ax.text(b,a,f"{value:.2f}",ha="center",va="center",color="white" if value>65 else "black")\n        ax.set_title("Screening outer event F1 (%) | fixed decoder thresholds")\n        fig.colorbar(heat,ax=ax);fig.tight_layout()\n        fig.savefig(run_root/"outer_f1_matrix_v9.png",dpi=150);plt.show();plt.close(fig)\n    print("Final recipe selected using INNER only:",index["selected"]["strategy"])\n    print("All-background accuracy is a reference, not the fall-detection objective.")\n    archive = run_root/"cnn_tcn_v9_suite_outputs.zip"\n    with base.zipfile.ZipFile(archive,"w",base.zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(run_root.rglob("*")):\n            if not path.is_file() or path.suffix not in (".csv",".json",".png",".npz",".pt"):\n                continue\n            if path.name=="last_checkpoint.pt" or not include_probabilities and path.name.startswith("probabilities_"):\n                continue\n            if path.name=="cnn_tcn_v9_selected_model.pt":\n                continue\n            z.write(path,path.relative_to(run_root))\n    print("Saved outputs:",archive)\n    return archive\n'
v9=types.ModuleType('cnn_tcn_v9')
sys.modules['cnn_tcn_v9']=v9
exec(compile(V9_SOURCE,'cnn_tcn_v9.py','exec'),v9.__dict__)
IMPLEMENTATION_SHA256='321cbf83c4e46cad8b659323ab6b48bc28852ad1df1975dcd09932b53055bc1f'
print('V9 definitions loaded; no training started.')


## 2. Cấu hình và ngân sách

DATA_ROOT phải chứa streams_events.csv, streams_composition.csv,
streams/train, streams/val và src/abc2027/evaluate.py. Notebook tự nhận diện
cả thư mục gói bị lồng thêm một cấp cùng tên. Có thể sửa đường dẫn theo Drive của bạn.

Mặc định: **12 lượt sàng lọc + 6 lượt seed thứ hai + 1 lượt all-train = 19 lượt train**.
Seed thứ hai so sánh A với một ứng viên B/C/D được chọn từ inner riêng của mỗi fold.
Vì ứng viên có thể khác giữa các fold, không diễn giải đây là lần lặp đủ cả 4 cấu hình.
Tắt CONFIRMATION_SEED=None để chỉ chạy 12+1 lượt. Cấu hình final vẫn chọn bằng inner
của vòng sàng lọc; kết quả seed thứ hai được dùng để kiểm tra độ ổn định.

Checkpoint chọn bằng event F1 inner với ngưỡng cố định. Quy tắc dừng sớm và
lịch LR giống nhau; số epoch thực chạy có thể khác. Cùng seed/fold/epoch có cùng cửa sổ,
thứ tự mẫu và ngân sách 110 bước x 32. Notebook kiểm tra hash danh sách mẫu.

RUN_MODE='continue' tiếp tục từ cuối epoch đã lưu. 'export_only' tạo lại outputs từ
checkpoint đã train xong. PLOT_ONLY=True chỉ đọc outputs để vẽ và nén lại.
Đổi code, dữ liệu hoặc kế hoạch cần RUN_TAG mới. Cache ở /content có thể tạo lại sau khi Colab ngắt;
checkpoint/output lưu Drive. Cache biến thể chuẩn hóa lớn hơn V8; cần đủ dung lượng ổ tạm.


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/abc2027_uwb_train_v1/abc2027_uwb_train_v1")
OUTPUT_BASE = Path("/content/drive/MyDrive/ABC2027_outputs")
CACHE_ROOT = Path("/content/abc2027_v9_feature_cache")
RUN_TAG = "cnn_tcn_v9_preprocessing_01"
RUN_ROOT = OUTPUT_BASE / RUN_TAG

SCREENING_SEED = 2027
CONFIRMATION_SEED = 42  # None: bỏ 6 lượt xác nhận bằng seed thứ hai
FOLDS = [
    ("outer_f7_a12_inner_f6_a10", {"fall:6","adl:10"}, {"fall:7","adl:12"}),
    ("outer_f6_a10_inner_f5_a9", {"fall:5","adl:9"}, {"fall:6","adl:10"}),
    ("outer_f5_a9_inner_f7_a12", {"fall:7","adl:12"}, {"fall:5","adl:9"}),
]
CONFIG = v9.ConfigV9(
    window_frames=80, model_size="standard", temporal_kernel=5,
    gain_augment=False, feature_noise_std=0., jitter_frames=0,
    lr=3e-4, weight_decay=2e-4, lr_floor_ratio=.1,
    batch_size=32, steps_per_epoch=110, max_epochs=20, min_epochs=10, patience=7,
    train_hop_frames=10, infer_hop_frames=2, train_eval_max_windows=2048,
    low_positive_fraction=.14, owner_draw_cap=4,
    frame_floor_quantile=.10, floor_sample_frames=100000, floor_absolute_min=1e-12,
)
RUN_MODE = "continue"  # train | continue | export_only
PLOT_ONLY = False
FINAL_REFIT_ALL_TRAIN = True
EVALUATE_PUBLIC = True  # yêu cầu FINAL_REFIT_ALL_TRAIN=True
SHOW_ALL_RUNS = False  # vẫn lưu biểu đồ cho tất cả run
INCLUDE_PROBABILITIES_IN_ZIP = True

CONFIG.validate()
if not PLOT_ONLY:
    candidates = [DATA_ROOT, DATA_ROOT/"abc2027_uwb_train_v1", DATA_ROOT.parent]
    DATA_ROOT = next((p for p in candidates if (p/"streams_events.csv").is_file()), DATA_ROOT)
    for name in ("streams_events.csv","streams_composition.csv","src/abc2027/evaluate.py"):
        assert (DATA_ROOT/name).is_file(), f"Thiếu {name}. Sửa DATA_ROOT: {DATA_ROOT}"
    for split in (["train","val"] if EVALUATE_PUBLIC else ["train"]):
        assert list((DATA_ROOT/"streams"/split).glob("stream_*.h5")), f"Thiếu streams/{split}"
    assert base.torch.cuda.is_available(), "Chọn Runtime > Change runtime type > GPU"
    assert not EVALUATE_PUBLIC or FINAL_REFIT_ALL_TRAIN
    print("GPU:",base.torch.cuda.get_device_name(0))
print("Screening:",4*len(FOLDS),"| Confirmation:",2*len(FOLDS) if CONFIRMATION_SEED is not None else 0,
      "| All-train:",int(FINAL_REFIT_ALL_TRAIN))
print("Data:", DATA_ROOT)
print("Outputs:", RUN_ROOT)


## 3. Bốn cấu hình và cách đọc kết quả

| Cấu hình | Input | Chuẩn hóa max từng frame |
|---|---|---|
| A_residual_delta | Residual + delta | Không |
| B_frame_norm | Residual + delta | Có, với mức sàn fit-only |
| C_delta_only | Chỉ delta, kênh residual bằng 0 | Không |
| D_delta_frame_norm | Chỉ delta, kênh residual bằng 0 | Có, với mức sàn fit-only |

Tất cả dùng sampler V8-B: 14% cửa sổ quanh event, có hoàn lại, cùng thứ tự mẫu;
tỷ lệ cửa sổ ngã khác tỷ lệ frame thật đang ngã. Không gain/noise augmentation hay jitter.
Split giữ fall, Autre, Empty cùng phiên nguồn về cùng phía; loại crossfade giữa các phía.
Fall và ADL có hệ mã nguồn riêng. Model vẫn học mọi loại ngã.

So sánh chính: **outer, decoder conservative_fixed**, với high=0.6/low=0.2 chung.
So sánh phụ: decoder conservative có high/low được dò trên inner.
B-A và D-C đo tác dụng chuẩn hóa; C-A và D-B đo tác dụng bỏ residual.
Nếu bỏ residual tốt hơn, chưa đủ kết luận nguyên nhân chắc chắn là quá độ đầu bộ lọc.
V9 này chưa thay cách khử nền hay kiểm chứng lại toàn bộ quy trình refit.

F1 chính thức là mean event F1 tại tIoU 0.1/0.3/0.5. Theo dõi thêm F1@0.5,
FP/giờ, FN, recall từng dataset/Chute và sai số start/end/duration.
Accuracy tính theo frame, luôn xem cùng all-background accuracy. Loss train/inner gần nhau
chưa đủ chứng minh model tốt; overfitting phải xem cùng event F1 ở nguồn giữ lại.
Thống kê đặc trưng physics_* là input trước z-score: B/D không còn đơn vị biên độ gốc,
C/D không còn residual. Không diễn giải chúng thành vận tốc/chiều cao vật lý.

Sau khi khóa lựa chọn bằng inner, model final học toàn bộ official train và xuất public một lần.
Refit có thể giảm điểm; không chọn lại cấu hình/ngưỡng bằng public.
Có thể tắt cả FINAL_REFIT_ALL_TRAIN và EVALUATE_PUBLIC để chỉ nghiên cứu outer.


In [ ]:
if PLOT_ONLY:
    assert (RUN_ROOT/"suite_runs_v9.json").is_file(), "Suite chưa hoàn tất. Kiểm tra RUN_ROOT."
else:
    V9_RESULTS, V9_SELECTION = v9.run_suite(
        DATA_ROOT, RUN_ROOT, CACHE_ROOT, CONFIG, FOLDS,
        seed=SCREENING_SEED, confirmation_seed=CONFIRMATION_SEED,
        run_mode=RUN_MODE, final_refit=FINAL_REFIT_ALL_TRAIN,
        public=EVALUATE_PUBLIC, implementation_hash=IMPLEMENTATION_SHA256,
    )
    print("Cấu hình final chọn từ screening INNER:",V9_SELECTION["strategy"])


## 4. Cell cuối: biểu đồ, ma trận điểm và ZIP

Cell dưới đọc outputs, lưu biểu đồ loss train vs inner validation cho từng run,
đường event F1 qua epoch, ma trận F1/accuracy và confusion matrix.
Run final chỉ có train loss vì số epoch đã khóa từ inner.
Mặc định hiện một fold mỗi cấu hình và final; SHOW_ALL_RUNS=True hiện toàn bộ.

Các tệp quan trọng:

- experiment_aggregate_v9.csv và outer_f1_matrix_v9.png: bảng đối chiếu A/B/C/D.
- paired_baseline_comparison_v9.csv: chênh lệch với A trong cùng fold/seed.
- factorial_effects_v9.csv: tác dụng từng yếu tố và tương tác, đơn vị điểm phần trăm.
- confirmation_aggregate_v9.csv: seed thứ hai, nếu bật.
- event_probability_comparison_v9.csv: xác suất ở mỗi cú ngã kể cả ca bị bỏ sót.
- final_all_train và cnn_tcn_v9_public_report.json: model final và public, nếu bật.

Mỗi run vẫn có checkpoint, training_history_v9.csv, checkpoint_event_f1.csv,
decoder grid, submission/report JSON, xác suất npz + index, lỗi biên/duration,
per-type recall, split/source/sampling manifests, normalizer và amplitude_floor_fit_only.json.
hard_negative_manifest.csv chỉ có header vì chưa bật hard-negative mining.
Chênh lệch nhỏ giữa fold/seed chưa phải bằng chứng thống kê; dữ liệu outer/public từng được xem
ở các phiên bản cũ. ZIP bỏ last_checkpoint.pt và bản sao selected model; file resume vẫn nằm Drive.


In [ ]:
from IPython.display import display, FileLink
V9_ZIP = v9.plots_and_archive(
    RUN_ROOT, show_all=SHOW_ALL_RUNS, include_probabilities=INCLUDE_PROBABILITIES_IN_ZIP,
)
display(FileLink(str(V9_ZIP)))
# Tải về máy nếu cần:
# from google.colab import files
# files.download(str(V9_ZIP))
